# Imports and Common Functions

In [ ]:
# !pip install --upgrade data_repo_client
# !pip install --upgrade xmltodict

In [ ]:
# Imports
import requests
import json
import google.auth
from google.cloud import storage
import xmltodict
import data_repo_client
import pandas as pd
import re
from time import sleep
import ast
import datetime
import os
ws_bucket = os.environ["WORKSPACE_BUCKET"]

# Function to refresh TDR API client
def refresh_tdr_api_client():
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    config = data_repo_client.Configuration()
    config.host = "https://data.terra.bio"
    config.access_token = creds.token
    api_client = data_repo_client.ApiClient(configuration=config)
    api_client.client_side_validation = False
    return api_client

# Function to pull existing AnVIL data from DUOS
def get_anvil_datasets_from_duos(duos_token, duos_env):
    
    # Determine the target URL from the env variable
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"
        
    # If no duos token provided, grab one for the pet SA
    if duos_token:
        token = duos_token
    else:
        creds, project = google.auth.default()
        auth_req = google.auth.transport.requests.Request()
        creds.refresh(auth_req)
        token = creds.token
    
    # Pull the list of DUOS datasets the user has access to (all public datasets if SA user used)
    print(f"Start time: {datetime.datetime.now()}")
    results = []
    datasets = requests.get(
        url=f"{url}/api/dataset/v3",
        headers={"Authorization": f"Bearer {token}"}
    ).json()
    
    # Limit datasets to just anvil datasets, then loop through and process
    anvil_datasets = []
    for dataset in datasets:
        if "ANVIL" in dataset.get("dataset_name").upper():
            anvil_datasets.append(dataset)
    datasets_to_process = len(anvil_datasets)
    datasets_processed = 0
    for dataset_entry in anvil_datasets:
        # Establish credentials
        if duos_token:
            token = duos_token
        else:
            creds, project = google.auth.default()
            auth_req = google.auth.transport.requests.Request()
            creds.refresh(auth_req)
            token = creds.token
        # Process dataset
        datasets_processed += 1
        dataset_id = dataset_entry["dataset_id"]
        print(f"Processing dataset {dataset_id} ({datasets_processed} of {datasets_to_process} datasets)...")
        dataset_details = requests.get(
            url=f"{url}/api/dataset/v2/{dataset_id}",
            headers={"Authorization": f"Bearer {token}"}
        ).json() 
        if dataset_details.get("study"):
            study_id = dataset_details["study"]["studyId"]
            if dataset_details["study"].get("description") and "Platform: AnVIL" in dataset_details["study"]["description"]: 
                study_name = dataset_details["study"]["name"]
                study_phs = ""
                for prop_entry in dataset_details["study"]["properties"]:
                    if prop_entry["key"] == "dbGaPPhsID":
                        study_phs = prop_entry["value"]
                        break
                dataset_name = dataset_details["name"]
                dataset_identifier = dataset_details["datasetIdentifier"]
                dac_id = dataset_details.get("dacId") if dataset_details.get("dacId") else ""
                data_use = dataset_details.get("dataUse")
                du_gru = data_use.get("generalUse") if data_use.get("generalUse") else False
                du_hmb = data_use.get("hmbResearch") if data_use.get("hmbResearch") else False
                du_disease = data_use.get("diseaseRestrictions") if data_use.get("diseaseRestrictions") else []
                du_poa = data_use.get("populationOriginsAncestry") if data_use.get("populationOriginsAncestry") else False
                du_ethics = data_use.get("ethicsApprovalRequired") if data_use.get("ethicsApprovalRequired") else False
                du_collab = data_use.get("collaboratorRequired") if data_use.get("collaboratorRequired") else False
                du_geog = data_use.get("geographicalRestrictions") if data_use.get("geographicalRestrictions") else ""
                du_genetic = data_use.get("geneticStudiesOnly") if data_use.get("geneticStudiesOnly") else False
                du_pub = data_use.get("publicationResults") if data_use.get("publicationResults") else False
                du_nmds = data_use.get("methodsResearch") if data_use.get("methodsResearch") else False
                du_npu = data_use.get("nonProfitUse") if data_use.get("nonProfitUse") else False
                du_other = data_use.get("other") if data_use.get("other") else ""
                data_location = ""
                data_location_url = ""
                access_management = ""
                for prop_entry in dataset_details["properties"]:
                    if prop_entry["propertyName"] == "Data Location":
                        data_location = prop_entry["propertyValue"]
                    elif prop_entry["propertyName"] == "URL":
                        data_location_url = prop_entry["propertyValue"]
                    elif prop_entry["propertyName"] == "Access Management":
                        access_management = prop_entry["propertyValue"]
                snapshot_id = ""
                workspace = ""
                if data_location == "TDR Location" and data_location_url and "https://data.terra.bio/snapshots/" in data_location_url:
                    snapshot_id = data_location_url.replace("https://data.terra.bio/snapshots/", "")
                elif data_location in ["AnVIL Workspace", "Terra Workspace"] and data_location_url and "#workspaces/" in data_location_url:
                    workspace = data_location_url.split("#workspaces/", 1)[1]
                public_visibility = dataset_details["study"].get("publicVisibility")
                results.append([study_id, study_name, study_phs, dataset_id, dataset_identifier, dataset_name, dac_id, access_management, du_gru, du_hmb, du_disease, du_poa, du_ethics, du_collab, du_geog, du_genetic, du_pub, du_nmds, du_npu, du_other, snapshot_id, workspace, public_visibility])
    
    # Return results
    print(f"End time: {datetime.datetime.now()}")
    return results

# Display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'center')
pd.set_option('display.precision', 3)

# Step 0: Review Existing AnVIL DUOS Entries

In [ ]:
#############################################
## Input Parameters
#############################################

# User token to use in pulling DUOS datasets (use "gcloud auth print-access-token | pbcopy" to get this)
# Leave empty to use SA token, which can pull all public datasets
duos_token = ""

# DUOS Environment
duos_env = "prod"

#############################################
## Execution
#############################################

# Fetch results
results = get_anvil_datasets_from_duos(duos_token, duos_env)

# Display results
df_results = pd.DataFrame(results, columns = ["Study ID", "Study Name", "Study PHS", "Dataset ID", "Dataset Identifier", "Dataset Name", "DAC ID", "Access", "GRU", "HMB", "DS", "POA", "IRB", "COL", "GS", "GSO", "PUB", "NMDS", "NPU", "OTHER", "Snapshot ID", "Workspace", "Public Visibility"])
df_results_sorted = df_results.sort_values(by=["Study ID", "Dataset ID"], ascending=[True, True], ignore_index=True)
print("\nResults:")
display(df_results)

# Step 1: Collect AnVIL Metadata for Review

In [ ]:
#############################################
## Functions
#############################################

def coalesce(*arg): 
    remove_list = ["", "NA", "N/A", "NONE", "TBD", "UNKNOWN", "UNSPECIFIED"]
    # update to remove N/A, None, Null, TBD
    for input_item in arg:
        if input_item is False or input_item == []:
            return input_item
        elif input_item:
            if isinstance(input_item, list):
                temp_list = [ele for ele in input_item if ele is not None and ele.upper() not in remove_list]
                if temp_list:
                    return temp_list
                else:
                    return []
            else:
                if str(input_item).upper() not in remove_list:
                    return input_item
    return None

def parse_consent(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.replace("_", "-")
    pattern = r"^(.+?)-(?:IRB|NPU|RD|MDS|PUB|COL|GSO)(?:-(?:IRB|NPU|RD|MDS|PUB|COL|GSO))*$"
    match = re.match(pattern, text)
    if match: 
        return match.group(1)
    else:
        return text

def format_description(input_string):
    output_string = str(input_string) if input_string else ""
    output_string = re.sub("\n\n\t", " ", output_string)
    output_string = re.sub("\t", " ", output_string)
    output_string = output_string.replace("\t", " ")
    output_string = re.sub("\n", "\\n", output_string)
    output_string = output_string.replace("\n", "\\n")
    output_string = re.sub("study.cgi\?study_id=|.\/study.cgi\?study_id=", "https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=", output_string)
    return output_string

def format_phs_id(input_str):
    try:
        num = re.search("phs0*([0-9]+)", input_str, re.IGNORECASE).group(1)
    except:
        num = ""
    if num:
        output_str = "phs" + str(num).zfill(6)
    else:
        output_str = ""
    return output_str

def try_join(l):
    try:
        if isinstance(l, list):
            return ', '.join(map(str, l))
        else:
            return l
    except TypeError:
        return l
    
def val_study_type_enum(l):
    if l and l not in ["Observational", "Interventional", "Descriptive", "Analytical", "Prospective", "Retrospective", "Case report", "Case series", "Cross-sectional", "Cohort study"]:
        return 1
    else:
        return 0

def val_nih_inst_center_sub_enum(l):
    if l and l not in ["NCI", "NEI", "NHLBI", "NHGRI", "NIA", "NIAAA", "NIAID", "NIAMS", "NIBIB", "NICHD", "NIDCD", "NIDCR", "NIDDK", "NIDA", "NIEHS", "NIGMS", "NIMH", "NIMHD", "NINDS", "NINR", "NLM", "CC", "CIT", "CSR", "FIC", "NCATS", "NCCIH"]:
        return 1
    else:
        return 0

def val_nih_ic_supp_study_enum(l):
    if l and isinstance(l, list):
        for item in l:
            if item not in ["NCI", "NEI", "NHLBI", "NHGRI", "NIA", "NIAAA", "NIAID", "NIAMS", "NIBIB", "NICHD", "NIDCD", "NIDCR", "NIDDK", "NIDA", "NIEHS", "NIGMS", "NIMH", "NIMHD", "NINDS", "NINR", "NLM", "CC", "CIT", "CSR", "FIC", "NCATS", "NCCIH"]:
                return 1
        return 0
    else:
        return 0

def val_file_type_enum(l):
    if l and isinstance(l, list):
        for item in l:
            if item not in ["Arrays", "Genome", "Exome", "Survey", "Phenotype"]:
                return 1
        return 0
    else:
        return 0

def fetch_dataset_details(snapshot_id_list, ds_consent_map, ds_consent_map_overrides, duos_token, duos_env, match_existing):
    
    # Initialize variables
    dataset_details_records = []
    current_datetime_string = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    in_progress_output_file = f"dataset_metadata_in_progress_{current_datetime_string}.tsv"

    # Determine the DUOS URL from the duos_env variable
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"

    # Build DUOS lookups
    print(f"Building DUOS dataset and study lookups...")
    print(f"In progress output file: {in_progress_output_file}")
    study_lookup = {}
    study_name_lookup = {}
    dataset_lookup = []
    datasets = requests.get(
        url=f"{url}/api/dataset/v3",
        headers={"Authorization": f"Bearer {duos_token}"}
    ).json()
    study_ids_processed = set()
    for dataset_entry in datasets:
        dataset_id = dataset_entry.get("dataset_id")
        dataset_name = dataset_entry.get("dataset_name")
        identifier = dataset_entry.get("identifier")
        study_id = dataset_entry.get("study_id")
        try:
            base_consent_group_name = re.search(r'(.*)_[0-9]{8}_ANV[0-9]+_[0-9]{12}$', dataset_name).group(1)
        except:
            base_consent_group_name = dataset_name
        if study_id:
            # Build dataset lookup
            dataset_lookup.append({
                "dataset_id": dataset_id,
                "consent_group_name": dataset_name, 
                "base_consent_group_name": base_consent_group_name,
                "identifier": identifier,
                "study_id": study_id
            })
            # Build study lookups
            if study_id not in study_ids_processed:
                study_ids_processed.add(study_id)
                study_details = requests.get(
                    url=f"{url}/api/dataset/registration/{identifier}",
                    headers={"Authorization": f"Bearer {duos_token}"}
                ).json()
                study_desc = study_details.get("studyDescription")
                if study_desc and "Platform: AnVIL" in study_desc:
                    study_phs = study_details.get("dbGaPPhsID")
                    if study_phs:
                        id_in_lookup = study_lookup.get(study_phs)
                        if id_in_lookup and id_in_lookup != study_id:
                            print(f"Warning: PHS ID {study_phs} tied to multiple studies in DUOS: {id_in_lookup}, {study_id}. Please review.")
                        else:
                            study_lookup[study_phs] = str(study_id)
                    study_name = study_details.get("studyName")
                    if study_name:
                        id_in_lookup = study_name_lookup.get(study_name)
                        if id_in_lookup and id_in_lookup != study_id:
                            print(f"Warning: Study Name '{study_name}' tied to multiple studies in DUOS: {id_in_lookup}, {study_id}. Please review.")
                        else:
                            study_name_lookup[study_name] = str(study_id)
                    
    # Loop through and process snapshots
    for snapshot_id in snapshot_id_list:
        
        # Initialize variables
        terra_dict = {}
        dbgap_xml_dict = {}
        dbgap_study_api_dict = {}
        dbgap_fhir_dict = {}
        final_results_dict = {}
        snapshot_phs_id = ""

        # Retrieve snapshot details
        print(f"Processing snapshot_id: {snapshot_id}...")
        final_results_dict = {}
        api_client = refresh_tdr_api_client()
        datasets_api = data_repo_client.DatasetsApi(api_client=api_client)
        snapshots_api = data_repo_client.SnapshotsApi(api_client=api_client)
        attempt_counter = 0
        snapshot_details = {}
        while attempt_counter <= 2:
            try:
                snapshot_details = snapshots_api.retrieve_snapshot(id=snapshot_id).to_dict()
                break
            except:
                sleep(5)
                attempt_counter += 1  
        snapshot_name = snapshot_details["name"]
        dataset_id = snapshot_details["source"][0]["dataset"]["id"]
        snapshot_phs_id = format_phs_id(snapshot_details["source"][0]["dataset"]["phs_id"])
        if snapshot_details["source"][0]["dataset"]["secure_monitoring_enabled"] == True:
            access_management = "controlled"
        else:
            access_management = "open"
        if snapshot_details["source"][0]["dataset_properties"].get("source_workspaces"):  
            source_workspace = snapshot_details["source"][0]["dataset_properties"]["source_workspaces"][0]
        else:
            source_workspace = None
        if snapshot_details["source"][0]["dataset_properties"].get("consent_name"):
            snapshot_consent_code = snapshot_details["source"][0]["dataset_properties"]["consent_name"]
        else:
            if access_management == "open":
                snapshot_consent_code = "NRES"
            else:
                snapshot_consent_code = None
        if snapshot_details["duos_firecloud_group"] != None:
            duos_id = snapshot_details["duos_firecloud_group"]["duos_id"]
        else:
            duos_id = None
        try:
            base_consent_group_name = re.search(r'(.*)_[0-9]{8}_ANV[0-9]+_[0-9]{12}$', snapshot_name).group(1)
        except:
            base_consent_group_name = snapshot_name
        if access_management == "open":
            consent_group_name = base_consent_group_name + " (NRES)"
        elif snapshot_consent_code:
            consent_group_name = base_consent_group_name + f" ({snapshot_consent_code})"
        else:
            consent_group_name = base_consent_group_name
        
        print("\tSnapshot PHS_ID: " + str(snapshot_phs_id))
        print("\tSnapshot Consent Code: " + str(snapshot_consent_code))
        print("\tSource Workspace: " + str(source_workspace))
        print("\tDUOS ID: " + str(duos_id))
        print("\tConsent Group Name: " + str(consent_group_name))
        
        # Derive study name for use in matching logic
        studyName = ""
        ws_study_name = ""
        xml_study_name = ""
        api_study_name = ""
        fhir_study_name = ""
        attempt_counter = 0
        while attempt_counter <= 2:
            creds, project = google.auth.default()
            auth_req = google.auth.transport.requests.Request()
            creds.refresh(auth_req)
            try:
                ws_attributes = requests.get(
                    url=f"https://api.firecloud.org/api/workspaces/anvil-datastorage/{source_workspace}?fields=workspace.attributes,workspace.authorizationDomain,workspace.googleProject,workspace.bucketName",
                    headers={"Authorization": f"Bearer {creds.token}"}
                ).json()
                if ws_attributes.get("workspace"):
                    ws_study_name = coalesce(ws_attributes["workspace"]["attributes"].get("library:projectName"), source_workspace)
                break
            except:
                sleep(5)
                attempt_counter += 1
        if snapshot_phs_id:
            phs_short = snapshot_phs_id.replace("phs", "")
            study_uid = ""
            dbgap_url = "https://dbgap.ncbi.nlm.nih.gov/ss/dbgapssws.cgi?request=Study&phs=" + phs_short
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    response = requests.get(url=dbgap_url)
                    xml_data = xmltodict.parse(response.text)
                    if xml_data["dbgapss"].get("Study"):
                        if isinstance(xml_data["dbgapss"]["Study"], list):
                            study_data = xml_data["dbgapss"]["Study"][0]
                        else:
                            study_data = xml_data["dbgapss"]["Study"] 
                        study_uid = study_data.get("@uid")
                        xml_study_name = study_data["StudyInfo"].get("StudyNameEntrez")
                    break
                except:
                    sleep(5)
                    attempt_counter += 1
            if study_uid:
                dbgap_study_url = "https://submit.ncbi.nlm.nih.gov/dbgap/api/v1/study_config/" + str(study_uid)
                attempt_counter = 0
                while attempt_counter <= 2:
                    try:
                        response = requests.get(url=dbgap_study_url)
                        study_api_data = json.loads(response.text)
                        if study_api_data.get("error") == None:
                            api_study_name = study_api_data["data"].get("report_name")
                        break
                    except:
                        sleep(5)
                        attempt_counter += 1 
            dbgap_fhir_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/ResearchStudy?_format=json&_id=" + snapshot_phs_id
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    response = requests.get(url=dbgap_fhir_url)
                    fhir_data = json.loads(response.text)
                    if fhir_data.get("entry"):
                        fhir_study_name = fhir_data["entry"][0]["resource"].get("title")
                    break
                except:
                    sleep(5)
                    attempt_counter += 1
        studyName = coalesce(fhir_study_name, xml_study_name, api_study_name, ws_study_name)
        if snapshot_phs_id and studyName and f" ({snapshot_phs_id})" not in studyName:
            studyName = studyName + f" ({snapshot_phs_id})"
        
        # Attempt to match the dataset to a DUOS Dataset ID based on consent group name
        match_duos_id = ""
        target_dataset_id = ""
        for dataset in dataset_lookup:
            if dataset["base_consent_group_name"] == base_consent_group_name or dataset["consent_group_name"] == consent_group_name:
                match_duos_id = dataset["identifier"]
                break
        match_study_id = ""
        
        # Attempt to match the dataset to a DUOS Study ID based on PHS ID or Study Name
        if snapshot_phs_id:
            match_study_id = study_lookup.get(snapshot_phs_id)
        else:
            match_study_id = study_name_lookup.get(studyName)

        # If a snapshot or match DUOS ID is present, use this to build the final result dictionary
        if match_existing and (duos_id or match_duos_id):

            # Pull existing DUOS study registration
            duos_id_to_use = coalesce(duos_id, match_duos_id)
            for dataset in dataset_lookup:
                if dataset["identifier"] == duos_id_to_use:
                    target_dataset_id = dataset["dataset_id"]
                    break
            duos_dict = {}
            duos_dict = requests.get(
                url=f"{url}/api/dataset/registration/{duos_id_to_use}",
                headers={"Authorization": f"Bearer {duos_token}"}
            ).json()
            #print(duos_dict)

            # Pull dataset details from DUOS (to get data use info) 
            if not duos_dict.get("consentGroups"):
                duos_dict["consentGroups"] = [{"datasetId": None}]
            duos_dataset_id = duos_dict["consentGroups"][0].get("datasetId")
            duos_data_use_dict = {}
            dac_id = ""
            if duos_dataset_id:
                dataset_details = requests.get(
                    url=f"{url}/api/dataset/v2/{duos_dataset_id}",
                    headers={"Authorization": f"Bearer {duos_token}"}
                ).json()
                duos_data_use_dict = dataset_details.get("dataUse")
                dac_id = dataset_details.get("dacId") if dataset_details.get("dacId") else ""

            # Build final results dictionary
            if snapshot_consent_code:
                consent_code = snapshot_consent_code.upper().replace("_", "-")
            else:
                consent_code = ""
            final_results_dict["snapshot_id"] = snapshot_id
            final_results_dict["snapshot_phs_id"] = snapshot_phs_id
            final_results_dict["snapshot_duos_id"] = duos_id
            final_results_dict["match_duos_id"] = match_duos_id
            final_results_dict["target_dataset_id"] = target_dataset_id
            final_results_dict["target_study_id"] = match_study_id
            studyName = duos_dict.get("studyName")
            dbGaP_study_name = duos_dict.get("dbGaPStudyRegistrationName")
            if snapshot_phs_id and studyName and f" ({snapshot_phs_id})" not in studyName:
                final_results_dict["studyName"] = studyName + f" ({snapshot_phs_id})"
            else:
                final_results_dict["studyName"] = studyName
            final_results_dict["studyType"] = duos_dict.get("studyType")
            final_results_dict["studyDescription"] = duos_dict.get("studyDescription")
            final_results_dict["dataTypes"] = duos_dict.get("dataTypes")
            final_results_dict["phenotypeIndication"] = duos_dict.get("phenotypeIndication")
            final_results_dict["species"] = duos_dict.get("species")
            final_results_dict["piName"] = duos_dict.get("piName")
            final_results_dict["dataCustodianEmail"] = duos_dict.get("dataCustodianEmail")
            final_results_dict["publicVisibility"] = duos_dict.get("publicVisibility")
            final_results_dict["nihAnvilUse"] = "I am NHGRI funded and I have a dbGaP PHS ID already" if duos_dict.get("nihAnvilUse") and 'already' in duos_dict.get("nihAnvilUse").lower() else "I am NHGRI funded and I do not have a dbGaP PHS ID"
            final_results_dict["submittingToAnvil"] = duos_dict.get("submittingToAnvil")
            if snapshot_phs_id:
                final_results_dict["dbGaPPhsID"] = snapshot_phs_id
            else:
                final_results_dict["dbGaPPhsID"] = duos_dict.get("dbGaPPhsID")
            if snapshot_phs_id and dbGaP_study_name and f" ({snapshot_phs_id})" in dbGaP_study_name:
                final_results_dict["dbGaPStudyRegistrationName"] = dbGaP_study_name.replace(f" ({snapshot_phs_id})", "")
            else:
                final_results_dict["dbGaPStudyRegistrationName"] = duos_dict.get("dbGaPStudyRegistrationName")
            final_results_dict["embargoReleaseDate"] = duos_dict.get("embargoReleaseDate")
            final_results_dict["sequencingCenter"] = duos_dict.get("sequencingCenter")
            final_results_dict["piEmail"] = duos_dict.get("piEmail")
            final_results_dict["piInstitution"] = duos_dict.get("piInstitution")
            final_results_dict["nihGrantContractNumber"] = duos_dict.get("nihGrantContractNumber")
            final_results_dict["nihICsSupportingStudy"] = duos_dict.get("nihICsSupportingStudy")
            final_results_dict["nihProgramOfficerName"] = duos_dict.get("nihProgramOfficerName")
            final_results_dict["nihInstitutionCenterSubmission"] = duos_dict.get("nihInstitutionCenterSubmission")
            final_results_dict["nihInstitutionalCertificationFileName"] = duos_dict.get("nihInstitutionalCertificationFileName")
            final_results_dict["nihGenomicProgramAdministratorName"] = duos_dict.get("nihGenomicProgramAdministratorName")
            final_results_dict["multiCenterStudy"] = duos_dict.get("multiCenterStudy")
            final_results_dict["collaboratingSites"] = duos_dict.get("collaboratingSites")
            final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSR"] = duos_dict.get("controlledAccessRequiredForGenomicSummaryResultsGSR")
            final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSRRequiredExplanation"] = duos_dict.get("controlledAccessRequiredForGenomicSummaryResultsGSRRequiredExplanation")
            final_results_dict["alternativeDataSharingPlan"] = duos_dict.get("alternativeDataSharingPlan")
            final_results_dict["alternativeDataSharingPlanReasons"] = duos_dict.get("alternativeDataSharingPlanReasons")
            final_results_dict["alternativeDataSharingPlanExplanation"] = duos_dict.get("alternativeDataSharingPlanExplanation")
            final_results_dict["alternativeDataSharingPlanFileName"] = duos_dict.get("alternativeDataSharingPlanFileName")
            final_results_dict["alternativeDataSharingPlanDataSubmitted"] = duos_dict.get("alternativeDataSharingPlanDataSubmitted")
            final_results_dict["alternativeDataSharingPlanDataReleased"] = duos_dict.get("alternativeDataSharingPlanDataReleased")
            final_results_dict["alternativeDataSharingPlanTargetDeliveryDate"] = duos_dict.get("alternativeDataSharingPlanTargetDeliveryDate")
            final_results_dict["alternativeDataSharingPlanTargetPublicReleaseDate"] = duos_dict.get("alternativeDataSharingPlanTargetPublicReleaseDate")
            final_results_dict["alternativeDataSharingPlanAccessManagement"] = duos_dict.get("alternativeDataSharingPlanAccessManagement")
            final_results_dict["consentGroups.consentGroupName"] = consent_group_name
            final_results_dict["consentGroups.accessManagement"] = access_management
            final_results_dict["dacId"] = dac_id
            final_results_dict["consentGroups.numberOfParticipants"] = duos_dict["consentGroups"][0].get("numberOfParticipants")
            final_results_dict["consentCode"] = consent_code
            final_results_dict["consentGroups.generalResearchUse"] = coalesce(duos_dict["consentGroups"][0].get("generalResearchUse"), duos_data_use_dict.get("generalUse"), False)
            final_results_dict["consentGroups.hmb"] = coalesce(duos_dict["consentGroups"][0].get("hmb"), duos_data_use_dict.get("hmbResearch"), False)
            final_results_dict["consentGroups.diseaseSpecificUse"] = coalesce(duos_dict["consentGroups"][0].get("diseaseSpecificUse"), duos_data_use_dict.get("diseaseRestrictions"), [])
            final_results_dict["consentGroups.gs"] = coalesce(duos_dict["consentGroups"][0].get("gs"), duos_data_use_dict.get("geographicalRestrictions"))
            final_results_dict["consentGroups.poa"] = coalesce(duos_dict["consentGroups"][0].get("poa"), duos_data_use_dict.get("populationOriginsAncestry"), False)
            final_results_dict["consentGroups.nmds"] = coalesce(duos_dict["consentGroups"][0].get("nmds"), False)
            final_results_dict["consentGroups.gso"] = coalesce(duos_dict["consentGroups"][0].get("gso"), duos_data_use_dict.get("geneticStudiesOnly"), False)
            final_results_dict["consentGroups.pub"] = coalesce(duos_dict["consentGroups"][0].get("pub"), duos_data_use_dict.get("publicationResults"), False)
            final_results_dict["consentGroups.col"] = coalesce(duos_dict["consentGroups"][0].get("col"), duos_data_use_dict.get("collaboratorRequired"), False)
            final_results_dict["consentGroups.irb"] = coalesce(duos_dict["consentGroups"][0].get("irb"), duos_data_use_dict.get("ethicsApprovalRequired"), False)
            final_results_dict["consentGroups.npu"] = coalesce(duos_dict["consentGroups"][0].get("npu"), False)
            final_results_dict["consentGroups.otherPrimary"] = coalesce(duos_dict["consentGroups"][0].get("otherPrimary"), duos_data_use_dict.get("other"))
            final_results_dict["consentGroups.otherSecondary"] = duos_dict["consentGroups"][0].get("otherSecondary")
            final_results_dict["consentGroups.mor"] = duos_dict["consentGroups"][0].get("mor")
            final_results_dict["consentGroups.morDate"] = duos_dict["consentGroups"][0].get("morDate")
            final_results_dict["consentGroups.dataLocation"] = "TDR Location"
            final_results_dict["consentGroups.url"] = "https://data.terra.bio/snapshots/" + snapshot_id
            if duos_dict["consentGroups"][0].get("fileTypes") and duos_dict["consentGroups"][0]["fileTypes"].get("fileType"):
                final_results_dict["consentGroups.fileTypes.fileType"] = duos_dict["consentGroups"][0]["fileTypes"][0].get("fileType")
            else:
                final_results_dict["consentGroups.fileTypes.fileType"] = None
            if duos_dict["consentGroups"][0].get("fileTypes") and duos_dict["consentGroups"][0]["fileTypes"].get("functionalEquivalence"):
                final_results_dict["consentGroups.fileTypes.functionalEquivalence"] = duos_dict["consentGroups"][0]["fileTypes"][0].get("functionalEquivalence")
            else:
                final_results_dict["consentGroups.fileTypes.functionalEquivalence"] = None
            collab_site = duos_dict.get("collaboratingSites")
            if collab_site:
                final_results_dict["consortium"] = collab_site[0]
            else:
                final_results_dict["consortium"] = None
            dataset_details_records.append(final_results_dict)
            continue

        # Pull information from original workspace (if listed)
        workspace_phs_id = ""
        if source_workspace:
            # Establish credentials
            creds, project = google.auth.default()
            auth_req = google.auth.transport.requests.Request()
            creds.refresh(auth_req)

            # Pull workspace attributes
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    ws_attributes = requests.get(
                        url=f"https://api.firecloud.org/api/workspaces/anvil-datastorage/{source_workspace}?fields=workspace.attributes,workspace.authorizationDomain,workspace.googleProject,workspace.bucketName",
                        headers={"Authorization": f"Bearer {creds.token}"}
                    ).json()
                    break
                except:
                    sleep(5)
                    attempt_counter += 1

            # Map to schema
            if ws_attributes.get("workspace"):
                terra_dict["studyType"] = ws_attributes["workspace"]["attributes"].get("library:studyDesign")
                terra_dict["studyDescription"] = ws_attributes["workspace"]["attributes"].get("description")
                if ws_attributes["workspace"]["attributes"].get("library:dataCategory"):
                    terra_dict["dataTypes"] = []
                    for item in ws_attributes["workspace"]["attributes"]["library:dataCategory"]["items"]:
                        inner_list = item.split(",")
                        for inner_item in inner_list:
                            inner_item = inner_item.replace("'", "").strip()
                            terra_dict["dataTypes"].append(inner_item)
                terra_dict["phenotypeIndication"] = ws_attributes["workspace"]["attributes"].get("library:indication")
                terra_dict["species"] = "Homo sapiens"
                terra_dict["piName"] = ws_attributes["workspace"]["attributes"].get("library:datasetOwner")
                terra_dict["dataCustodianEmail"] = [ws_attributes["workspace"]["attributes"].get("library:contactEmail")]
                if ws_attributes["workspace"]["attributes"].get("tag:tags"):
                    for tag in ws_attributes["workspace"]["attributes"].get("tag:tags")["items"]:
                        if "Consortium:" in tag:
                            terra_dict["consortium"] = tag.split(":")[1].strip()
                        elif "dbGaP:" in tag:
                            terra_dict["dbGaPPhsID"] = format_phs_id(tag.split(":")[1].strip())
                            if not snapshot_phs_id:
                                workspace_phs_id = format_phs_id(tag.split(":")[1].strip()) 
                                print(f"Warning: PHS ID ({workspace_phs_id}) found on workspace but not snapshot! Please resolve.")
                terra_dict["consentGroups.consentCode"] = ws_attributes["workspace"]["attributes"].get("library:dataUseRestriction")
                if ws_attributes["workspace"]["attributes"].get("library:datatype"):
                    terra_dict["consentGroups.fileTypes.fileType"] = ws_attributes["workspace"]["attributes"]["library:datatype"]["items"]
                if ws_attributes["workspace"]["attributes"].get("library:numSubjects"):
                    terra_dict["consentGroups.numberOfParticipants"] = ws_attributes["workspace"]["attributes"]["library:numSubjects"]
        #         print("------------------------------------------------------")
        #         print("terra_dict")
        #         print(terra_dict)

        # Pull study information from DUOS (if matched to DUOS Study based on PHS ID)
        if not match_study_id and workspace_phs_id:
            match_study_id = study_lookup.get(workspace_phs_id)
        duos_study_dict = {}
        if match_existing and match_study_id:
            duos_study_dict = requests.get(
                    url=f"{url}/api/dataset/study/registration/{match_study_id}",
                    headers={"Authorization": f"Bearer {duos_token}"}
                ).json()
            collab_site = duos_study_dict.get("collaboratingSites")
            if collab_site:
                duos_study_dict["consortium"] = collab_site[0]
        
        # Pull information from dbGaP (if phs_id listed)
        dac_names = ""
        dbgap_phs_id = coalesce(snapshot_phs_id, workspace_phs_id)
        if dbgap_phs_id:
            # Pull and parse XML
            phs_short = dbgap_phs_id.replace("phs", "")
            dbgap_url = "https://dbgap.ncbi.nlm.nih.gov/ss/dbgapssws.cgi?request=Study&phs=" + phs_short
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    response = requests.get(url=dbgap_url)
                    xml_data = xmltodict.parse(response.text)
                    break
                except:
                    sleep(5)
                    attempt_counter += 1
            study_uid = ""

            # Map to schema
            if xml_data["dbgapss"].get("Study"):
                if isinstance(xml_data["dbgapss"]["Study"], list):
                    study_data = xml_data["dbgapss"]["Study"][0]
                else:
                    study_data = xml_data["dbgapss"]["Study"] 
                study_uid = study_data.get("@uid")
                dbgap_xml_dict["studyDescription"] = study_data["StudyInfo"].get("Description")
                dbgap_xml_dict["dbGaPPhsID"] = dbgap_phs_id
                dbgap_xml_dict["dbGaPStudyRegistrationName"] = study_data["StudyInfo"].get("StudyNameEntrez")
                if study_data["Authority"]["Persons"].get("Person"):
                    for ap_entry in study_data["Authority"]["Persons"]["Person"]:
                        if ap_entry["Role"] == "PI":
                            dbgap_xml_dict["piName"] = ap_entry["@lname"] + ", " + ap_entry["@fname"]
                            dbgap_xml_dict["piEmail"] = ap_entry["@email"]
                            dbgap_xml_dict["piInstitution"] = ap_entry["Organization"]
                        elif ap_entry["Role"] == "PO" and ap_entry["Organization"] == "NIH":
                            dbgap_xml_dict["nihProgramOfficerName"] = ap_entry["@lname"] + ", " + ap_entry["@fname"]
                        elif ap_entry["Role"] == "GPA" and ap_entry["Organization"] == "NIH":
                            dbgap_xml_dict["nihGenomicProgramAdministratorName"] = ap_entry["@lname"] + ", " + ap_entry["@fname"]
                ic_list = []
                if isinstance(study_data["Authority"]["ICs"]["IC"], list):
                    for ic_entry in study_data["Authority"]["ICs"]["IC"]:
                        ic_list.append(ic_entry["@name"])
                else:
                    ic_list.append(study_data["Authority"]["ICs"]["IC"]["@name"])
                dbgap_xml_dict["nihICsSupportingStudy"] = ic_list
                dbgap_xml_dict["consentGroups.numberOfParticipants"] = study_data.get("@num_participants")
                dbgap_xml_dict["embargoReleaseDate"] = study_data["Policy"].get("@pub-embargo")
                if study_data["Policy"].get("ConsentGroup"):
                    if isinstance(study_data["Policy"]["ConsentGroup"], list):
                        dac_name_set = set()
                        for idx, consent in enumerate(study_data["Policy"]["ConsentGroup"]):
                            tmp_dac = consent["@dac_name"]
                            dac_name_set.add(tmp_dac)
                        dac_name_list = list(dac_name_set)
                        dac_names = ", ".join(dac_name_list)
                    else:
                        dac_names = study_data["Policy"]["ConsentGroup"]["@dac_name"]     
        #             print("------------------------------------------------------")
        #             print("dbgap_xml_dict")
        #             print(dbgap_xml_dict)

            # Pull and parse Study API JSON
            if study_uid:
                dbgap_study_url = "https://submit.ncbi.nlm.nih.gov/dbgap/api/v1/study_config/" + str(study_uid)
                attempt_counter = 0
                while attempt_counter <= 2:
                    try:
                        response = requests.get(url=dbgap_study_url)
                        study_api_data = json.loads(response.text)
                        break
                    except:
                        sleep(5)
                        attempt_counter += 1

                # Map to schema
                if study_api_data.get("error") == None:
                    dbgap_study_api_dict["studyDescription"] = study_api_data["data"].get("description")
                    dbgap_study_api_dict["phenotypeIndication"] = study_api_data["data"].get("primary_disease")
                    dbgap_study_api_dict["studyType"] = study_api_data["data"].get("study_design")
                    dbgap_study_api_dict["dbGaPPhsID"] = dbgap_phs_id
                    dbgap_study_api_dict["dbGaPStudyRegistrationName"] = study_api_data["data"].get("report_name")
                    for attr_entry in study_api_data["data"].get("attribution"):
                        if attr_entry.get("title") == "Principal Investigator":
                            dbgap_study_api_dict["piName"] = attr_entry.get("name")
                            dbgap_study_api_dict["piInstitution"] = attr_entry.get("institute")
                            break
        #             print("------------------------------------------------------")
        #             print("dbgap_study_api_dict")
        #             print(dbgap_study_api_dict)

            # Pull and parse FHIR API JSON
            dbgap_fhir_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/ResearchStudy?_format=json&_id=" + dbgap_phs_id
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    response = requests.get(url=dbgap_fhir_url)
                    fhir_data = json.loads(response.text)
                    break
                except:
                    sleep(5)
                    attempt_counter += 1

            # Map to schema
            if fhir_data.get("entry"):
                dbgap_fhir_dict["studyDescription"] = fhir_data["entry"][0]["resource"].get("description")
                dbgap_fhir_dict["dbGaPPhsID"] = dbgap_phs_id
                dbgap_fhir_dict["dbGaPStudyRegistrationName"] = fhir_data["entry"][0]["resource"].get("title")
                # NIH ICs
                if "Organization/" in fhir_data["entry"][0]["resource"]["sponsor"].get("reference"):
                    dbgap_fhir_dict["nihICsSupportingStudy"] = [fhir_data["entry"][0]["resource"]["sponsor"].get("reference")[13:]]
                else:
                    ic_display = fhir_data["entry"][0]["resource"]["sponsor"].get("display")
                    if ic_display == "National Human Genome Research Institute":
                        dbgap_fhir_dict["nihICsSupportingStudy"] = ["NHGRI"]
                    else:
                        dbgap_fhir_dict["nihICsSupportingStudy"] = [ic_display]
                # studyType
                if fhir_data["entry"][0]["resource"].get("category"):
                    for cat_entry in fhir_data["entry"][0]["resource"].get("category"):
                        if cat_entry.get("coding"):
                            for coding_entry in cat_entry.get("coding"):
                                if coding_entry.get("system") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/ResearchStudy-StudyDesign":
                                    value = coding_entry.get("display") if coding_entry.get("display") else coding_entry.get("code")
                                    if dbgap_fhir_dict.get("studyType") and value:
                                        dbgap_fhir_dict["studyType"] += f", {value}"
                                    elif value:
                                        dbgap_fhir_dict["studyType"] = value
                # dataTypes
                dt_list = []
                if fhir_data["entry"][0]["resource"].get("extension"): 
                    for ext_entry in fhir_data["entry"][0]["resource"].get("extension"):
                        if ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-MolecularDataTypes":
                            for inner_ext_entry in ext_entry.get("extension"):
                                if inner_ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-MolecularDataTypes-MolecularDataType":
                                    for coding_entry in inner_ext_entry["valueCodeableConcept"].get("coding"):
                                        dt_list.append(coding_entry.get("code"))
                dbgap_fhir_dict["dataTypes"] = dt_list
                # phenotypeIndication
                if fhir_data["entry"][0]["resource"].get("focus"):
                    for focus_entry in fhir_data["entry"][0]["resource"].get("focus"):
                        if focus_entry.get("coding"):
                            for coding_entry in focus_entry.get("coding"):
                                value = coding_entry.get("display") if coding_entry.get("display") else coding_entry.get("code")
                                if dbgap_fhir_dict.get("phenotypeIndication") and value:
                                    dbgap_fhir_dict["phenotypeIndication"] += f", {value}"
                                elif value:
                                    dbgap_fhir_dict["phenotypeIndication"] = value
                # numberOfParticipants
                if fhir_data["entry"][0]["resource"].get("extension"):
                    for ext_entry in fhir_data["entry"][0]["resource"].get("extension"):
                        if ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-Content":
                            for inner_ext_entry in ext_entry.get("extension"):
                                if inner_ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-Content-NumSubjects":
                                    dbgap_fhir_dict["consentGroups.numberOfParticipants"] = inner_ext_entry["valueCount"].get("code")
        #         print("------------------------------------------------------")
        #         print("dbgap_fhir_dict")
        #         print(dbgap_fhir_dict)

        # Reconcile information and create final results
        consent_code = coalesce(snapshot_consent_code, terra_dict.get("consentGroups.consentCode"), dbgap_fhir_dict.get("consentGroups.consentCode"), dbgap_xml_dict.get("consentGroups.consentCode"), dbgap_study_api_dict.get("consentGroups.consentCode"))
        if consent_code:
            consent_code = consent_code.upper().replace("_", "-")
            parsed_consent_code = parse_consent(consent_code)
        else:
            consent_code = ""
            parsed_consent_code = ""
        consortium = coalesce(duos_study_dict.get("consortium"), terra_dict.get("consortium"), dbgap_fhir_dict.get("consortium"), dbgap_xml_dict.get("consortium"), dbgap_study_api_dict.get("consortium"))
        dbGaPPhsID = coalesce(duos_study_dict.get("dbGaPPhsID"), dbgap_fhir_dict.get("dbGaPPhsID"), dbgap_xml_dict.get("dbGaPPhsID"), dbgap_study_api_dict.get("dbGaPPhsID"), terra_dict.get("dbGaPPhsID"))
        dbGaPStudyRegistrationName = coalesce(duos_study_dict.get("dbGaPStudyRegistrationName"), dbgap_fhir_dict.get("dbGaPStudyRegistrationName"), dbgap_xml_dict.get("dbGaPStudyRegistrationName"), dbgap_study_api_dict.get("dbGaPStudyRegistrationName"), terra_dict.get("dbGaPStudyRegistrationName"))
        if dbGaPPhsID and consent_code:
            study_consent = dbGaPPhsID + ":" + consent_code
            purl_doid = ds_consent_map_overrides.get(study_consent)
            if not purl_doid:
                purl_doid = ds_consent_map.get(parsed_consent_code)
            if purl_doid:
                if not isinstance(purl_doid, list):
                    purl_doid = [purl_doid]
            else:
                purl_doid = []
        else:
            purl_doid = []
        final_results_dict["snapshot_id"] = snapshot_id
        final_results_dict["snapshot_phs_id"] = snapshot_phs_id
        final_results_dict["snapshot_duos_id"] = duos_id
        final_results_dict["match_duos_id"] = match_duos_id
        final_results_dict["target_dataset_id"] = target_dataset_id
        final_results_dict["target_study_id"] = match_study_id
        studyName = coalesce(duos_study_dict.get("studyName"), studyName)
        if dbGaPPhsID and studyName and f" ({dbGaPPhsID})" not in studyName:
            final_results_dict["studyName"] = studyName + f" ({dbGaPPhsID})"
        else:
            final_results_dict["studyName"] = studyName
        final_results_dict["studyType"] = coalesce(duos_study_dict.get("studyType"), dbgap_fhir_dict.get("studyType"), dbgap_xml_dict.get("studyType"), dbgap_study_api_dict.get("studyType"), terra_dict.get("studyType"))
        final_results_dict["studyDescription"] = format_description(coalesce(duos_study_dict.get("studyDescription"), dbgap_fhir_dict.get("studyDescription"), dbgap_xml_dict.get("studyDescription"), dbgap_study_api_dict.get("studyDescription"), terra_dict.get("studyDescription")))
        if final_results_dict["studyDescription"]:
            if "Platform: AnVIL" not in final_results_dict["studyDescription"]:
                final_results_dict["studyDescription"] = final_results_dict["studyDescription"] + "\nPlatform: AnVIL"
        else:
            final_results_dict["studyDescription"] = "Platform: AnVIL"
        final_results_dict["dataTypes"] = coalesce(duos_study_dict.get("dataTypes"), terra_dict.get("dataTypes"), dbgap_fhir_dict.get("dataTypes"), dbgap_xml_dict.get("dataTypes"), dbgap_study_api_dict.get("dataTypes"))
        final_results_dict["phenotypeIndication"] = coalesce(duos_study_dict.get("phenotypeIndication"), terra_dict.get("phenotypeIndication"), dbgap_fhir_dict.get("phenotypeIndication"), dbgap_xml_dict.get("phenotypeIndication"), dbgap_study_api_dict.get("phenotypeIndication"))
        final_results_dict["species"] = "Human"
        final_results_dict["piName"] = coalesce(duos_study_dict.get("piName"), dbgap_fhir_dict.get("piName"), dbgap_xml_dict.get("piName"), dbgap_study_api_dict.get("piName"), terra_dict.get("piName"), "None")
        final_results_dict["dataCustodianEmail"] = ["help@lists.anvilproject.org"]
        final_results_dict["publicVisibility"] = True
        final_results_dict["nihAnvilUse"] = "I am NHGRI funded and I have a dbGaP PHS ID already" if dbGaPPhsID else "I am NHGRI funded and I do not have a dbGaP PHS ID"
        final_results_dict["submittingToAnvil"] = True
        final_results_dict["dbGaPPhsID"] = dbGaPPhsID
        if dbGaPPhsID and dbGaPStudyRegistrationName and f" ({dbGaPPhsID})" in dbGaPStudyRegistrationName:
            final_results_dict["dbGaPStudyRegistrationName"] = dbGaPStudyRegistrationName.replace(f" ({dbGaPPhsID})", "")
        else:
            final_results_dict["dbGaPStudyRegistrationName"] = dbGaPStudyRegistrationName
        final_results_dict["embargoReleaseDate"] = coalesce(duos_study_dict.get("embargoReleaseDate"), dbgap_fhir_dict.get("embargoReleaseDate"), dbgap_xml_dict.get("embargoReleaseDate"), dbgap_study_api_dict.get("embargoReleaseDate"), terra_dict.get("embargoReleaseDate"))
        final_results_dict["sequencingCenter"] = None
        final_results_dict["piEmail"] = coalesce(duos_study_dict.get("piEmail"), dbgap_fhir_dict.get("piEmail"), dbgap_xml_dict.get("piEmail"), dbgap_study_api_dict.get("piEmail"), terra_dict.get("piEmail"))
        final_results_dict["piInstitution"] = coalesce(duos_study_dict.get("piInstitution"), dbgap_fhir_dict.get("piInstitution"), dbgap_xml_dict.get("piInstitution"), dbgap_study_api_dict.get("piInstitution"), terra_dict.get("piInstitution"))
        final_results_dict["nihGrantContractNumber"] = None
        final_results_dict["nihICsSupportingStudy"] = coalesce(duos_study_dict.get("nihICsSupportingStudy"), dbgap_fhir_dict.get("nihICsSupportingStudy"), dbgap_xml_dict.get("nihICsSupportingStudy"), dbgap_study_api_dict.get("nihICsSupportingStudy"), terra_dict.get("nihICsSupportingStudy"))
        final_results_dict["nihProgramOfficerName"] = coalesce(duos_study_dict.get("nihProgramOfficerName"), dbgap_fhir_dict.get("nihProgramOfficerName"), dbgap_xml_dict.get("nihProgramOfficerName"), dbgap_study_api_dict.get("nihProgramOfficerName"), terra_dict.get("nihProgramOfficerName"))
        final_results_dict["nihInstitutionCenterSubmission"] = "NHGRI"
        final_results_dict["nihInstitutionalCertificationFileName"] = None
        final_results_dict["nihGenomicProgramAdministratorName"] = coalesce(duos_study_dict.get("nihGenomicProgramAdministratorName"), dbgap_fhir_dict.get("nihGenomicProgramAdministratorName"), dbgap_xml_dict.get("nihGenomicProgramAdministratorName"), dbgap_study_api_dict.get("nihGenomicProgramAdministratorName"), terra_dict.get("nihGenomicProgramAdministratorName"))
        final_results_dict["multiCenterStudy"] = None
        final_results_dict["collaboratingSites"] = [consortium] if consortium else []
        final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSR"] = None
        final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSRRequiredExplanation"] = None
        final_results_dict["alternativeDataSharingPlan"] = False
        final_results_dict["alternativeDataSharingPlanReasons"] = []
        final_results_dict["alternativeDataSharingPlanExplanation"] = None
        final_results_dict["alternativeDataSharingPlanFileName"] = None
        final_results_dict["alternativeDataSharingPlanDataSubmitted"] = None
        final_results_dict["alternativeDataSharingPlanDataReleased"] = None
        final_results_dict["alternativeDataSharingPlanTargetDeliveryDate"] = None
        final_results_dict["alternativeDataSharingPlanTargetPublicReleaseDate"] = None
        final_results_dict["alternativeDataSharingPlanAccessManagement"] = None
        final_results_dict["consentGroups.consentGroupName"] = consent_group_name
        if access_management == "controlled" and 'NHGRI' not in dac_names:
            final_results_dict["consentGroups.accessManagement"] = "external"
        else:
            final_results_dict["consentGroups.accessManagement"] = access_management
        final_results_dict["dacId"] = dac_names
        final_results_dict["consentGroups.numberOfParticipants"] = coalesce(terra_dict.get("consentGroups.numberOfParticipants"), dbgap_fhir_dict.get("consentGroups.numberOfParticipants"), dbgap_xml_dict.get("consentGroups.numberOfParticipants"), dbgap_study_api_dict.get("consentGroups.numberOfParticipants"), "0")
        final_results_dict["consentCode"] = consent_code
        final_results_dict["consentGroups.generalResearchUse"] = True if access_management == "controlled" and "GRU" in consent_code else False
        final_results_dict["consentGroups.hmb"] = True if access_management == "controlled" and "HMB" in consent_code else False
        if purl_doid:
            final_results_dict["consentGroups.diseaseSpecificUse"] = purl_doid
        else:
            final_results_dict["consentGroups.diseaseSpecificUse"] = []
        final_results_dict["consentGroups.gs"] = consent_code if access_management == "controlled" and "GS-" in consent_code else None
        final_results_dict["consentGroups.poa"] = True if access_management == "controlled" and "POA" in consent_code else False
        final_results_dict["consentGroups.nmds"] = True if access_management == "controlled" and "NMDS" in consent_code else False
        final_results_dict["consentGroups.gso"] = True if access_management == "controlled" and "GSO" in consent_code else False
        final_results_dict["consentGroups.pub"] = True if access_management == "controlled" and "PUB" in consent_code else False 
        final_results_dict["consentGroups.col"] = True if access_management == "controlled" and "COL-" in consent_code else False
        final_results_dict["consentGroups.irb"] = True if access_management == "controlled" and "IRB" in consent_code else False
        final_results_dict["consentGroups.npu"] = True if access_management == "controlled" and "NPU" in consent_code else False
        final_results_dict["consentGroups.otherPrimary"] = consent_code if (consent_code and access_management == "controlled" and not (final_results_dict["consentGroups.generalResearchUse"] or final_results_dict["consentGroups.hmb"] or final_results_dict["consentGroups.diseaseSpecificUse"] or final_results_dict["consentGroups.gs"] or final_results_dict["consentGroups.poa"] or final_results_dict["consentGroups.nmds"] or final_results_dict["consentGroups.gso"] or final_results_dict["consentGroups.pub"] or final_results_dict["consentGroups.col"] or final_results_dict["consentGroups.irb"] or final_results_dict["consentGroups.npu"])) else None
        final_results_dict["consentGroups.otherSecondary"] = None
        final_results_dict["consentGroups.mor"] = None
        final_results_dict["consentGroups.morDate"] = None
        final_results_dict["consentGroups.dataLocation"] = "TDR Location"
        final_results_dict["consentGroups.url"] = "https://data.terra.bio/snapshots/" + snapshot_id
        final_results_dict["consentGroups.fileTypes.fileType"] = coalesce(terra_dict.get("consentGroups.fileTypes.fileType"), dbgap_fhir_dict.get("consentGroups.fileTypes.fileType"), dbgap_xml_dict.get("consentGroups.fileTypes.fileType"), dbgap_study_api_dict.get("consentGroups.fileTypes.fileType"))
        final_results_dict["consentGroups.fileTypes.functionalEquivalence"] = None
        final_results_dict["consortium"] = consortium
        dataset_details_records.append(final_results_dict)
        
        # Write out in progress results
        ip_output = pd.DataFrame(dataset_details_records)
        ip_output_sorted = ip_output.sort_values(by=["studyName", "consentGroups.consentGroupName"], ascending=[True, True], ignore_index=True)
        ip_output_sorted.to_csv(in_progress_output_file, index=False, sep="\t")
        !gsutil cp $in_progress_output_file $ws_bucket/dataset_metadata/output/in_progress/ 2> stdout
        
    # Return results
    return dataset_details_records

#############################################
## Input Parameters
#############################################

# Specify the snapshots to pull data for:
snapshot_id_list = [
    '1cee5d6a-8899-4dc1-aadd-635fd696c155',
    'b75774c4-5443-47db-aa6b-e5cabacf70f3',
    '8336cc7d-4b4b-41d2-af31-e27f21b3a200',
    '43c7c8a9-6e41-483b-bed3-3321f17ce2c7',
    'a46f3bf9-1343-457d-877f-c994e0aa371d',
    'fcf14d92-e951-487a-a4e5-85423b3553cc',
    '482743a5-95fe-49f6-8b11-556484f3c0aa',
    'dd05e6e1-ee90-49d0-9e2c-4ba688df1923',
    'd3427a3b-3ece-42a6-a9b4-7b2cb8bcb8d4',
    '54996c85-36d3-43ad-8d1c-6d3a50834133',
    'eb32a4f2-74bc-464d-b731-9e2c945ae373',
    '2e9f20f1-521c-4087-9d00-3aa6a6cf9c38',
    '11a7f843-5304-49da-9f29-c7be8927c9d4',
    '10ac69f0-8feb-461d-96d3-35c2acbd8077',
    'f98f4e04-c4a2-4ce4-9720-d6ade48b9406',
    '58e35120-422d-4939-8226-e0f3e898bda6',
]

# Specify a file that maps DS consent codes with modifiers removed (IRB, NPU, RD, MDS, PUB, COL, GSO) to DOID
# Form should be JSON with structure {"DS-CODE": ["http://purl.obolibrary.org/obo/DOID_12345"], ...}
ds_consent_map_file = "dataset_metadata/support/ds_consent_mapping.json"

# Specify a file of overrides to the DS consent map (for cases where a DS code has different definitions for different studies)
# Form should be JSON with structure {"PHS:DS-CODE": ["http://purl.obolibrary.org/obo/DOID_12345"], ...}
ds_consent_map_override_file = "dataset_metadata/support/ds_consent_mapping_overrides.json"

# Token for use in DUOS (use "gcloud auth print-access-token | pbcopy" to get this)
# DUOS variables:
#  - duos_token - Token for use in DUOS (use: gcloud auth print-access-token | pbcopy)
#  - duos_env - DUOS environment to use
duos_token = ""
duos_env = "prod"

# Specify whether results should be written out to a file in the workspace bucket
write_to_bucket = True

# Set the below to "False" to turn off matching to existing dataset/study information in DUOS (to see what the original values would have been)
match_existing = True

#############################################
## Execution
#############################################

# Load in contents of consent map files
storage_client = storage.Client()
bucket = storage_client.get_bucket(ws_bucket.replace("gs://", ""))
consent_map_blob = bucket.blob(ds_consent_map_file)
ds_consent_map = json.loads(consent_map_blob.download_as_string(client=None))
consent_map_override_blob = bucket.blob(ds_consent_map_override_file) 
ds_consent_map_overrides = json.loads(consent_map_override_blob.download_as_string(client=None))

# Pull dataset details
dataset_details_records = fetch_dataset_details(snapshot_id_list, ds_consent_map, ds_consent_map_overrides, duos_token, duos_env, match_existing)
output = pd.DataFrame(dataset_details_records)
output_sorted = output.sort_values(by=["studyName", "consentGroups.consentGroupName"], ascending=[True, True], ignore_index=True)

#############################################
## Validation and Output
#############################################
# Create copy of dataframe for unique value validation
output_unique_val = output_sorted.copy()

# Convert study list fields to strings
list_fields = ["dataTypes", "dataCustodianEmail", "nihICsSupportingStudy", "collaboratingSites", "alternativeDataSharingPlanReasons"]
for field in list_fields:
    output_unique_val[field] = [try_join(l) for l in output_unique_val[field]]

# Get unique values per study-level field, by study
study_level_col_list = []
for col in output_unique_val.columns:
    if "consentGroups." not in col and col not in ["studyName", "snapshot_id", "consortium", "consentCode", "snapshot_duos_id", "match_duos_id"]:
        study_level_col_list.append(col)
df_unique = output_unique_val.groupby("studyName")[study_level_col_list].nunique()
df_unique["unique_value_validation"] = df_unique.max(axis=1)
df_unique["unique_value_validation"] = ["Pass" if l <= 1 else "Fail" for l in df_unique["unique_value_validation"]]

# Create copy of dataframe for enum validation
output_enum_val = output_sorted.copy()

# Validate enum fields
output_enum_val["studyType"] = [val_study_type_enum(l) for l in output_enum_val["studyType"]]
output_enum_val["nihInstitutionCenterSubmission"] = [val_nih_inst_center_sub_enum(l) for l in output_enum_val["nihInstitutionCenterSubmission"]]
output_enum_val["nihICsSupportingStudy"] = [val_nih_ic_supp_study_enum(l) for l in output_enum_val["nihICsSupportingStudy"]]
output_enum_val["consentGroups.fileTypes.fileType"] = [val_file_type_enum(l) for l in output_enum_val["consentGroups.fileTypes.fileType"]]
study_enum_cols = ["studyType", "nihInstitutionCenterSubmission", "nihICsSupportingStudy"]
df_study_enum = output_enum_val.groupby("studyName")[study_enum_cols].sum()
df_study_enum["study_enum_value_validation"] = df_study_enum.max(axis=1)
df_study_enum["study_enum_value_validation"] = ["Pass" if l < 1 else "Fail" for l in df_study_enum["study_enum_value_validation"]]
consent_group_enum_cols = ["consentGroups.fileTypes.fileType"]
df_consent_group_enum = output_enum_val.groupby("consentGroups.consentGroupName")[consent_group_enum_cols].sum()
df_consent_group_enum["consent_group_enum_value_validation"] = df_consent_group_enum.max(axis=1)
df_consent_group_enum["consent_group_enum_value_validation"] = ["Pass" if l < 1 else "Fail" for l in df_consent_group_enum["consent_group_enum_value_validation"]]

# Join validation dataframes to original dataframe
output_sorted_validated = output_sorted.join(df_unique["unique_value_validation"], on="studyName", how="left")
output_sorted_validated = output_sorted_validated.join(df_study_enum["study_enum_value_validation"], on="studyName", how="left")
output_sorted_validated = output_sorted_validated.join(df_consent_group_enum["consent_group_enum_value_validation"], on="consentGroups.consentGroupName", how="left")

# Write out output
current_datetime_string = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
output_file = f"dataset_metadata_{current_datetime_string}.tsv"
output_sorted_validated.to_csv(output_file, index=False, sep="\t")
!gsutil cp $output_file $ws_bucket/dataset_metadata/output/ 2> stdout

# Display outputs
print("----------------------------------------------------------------------------------------------------")
print("----------------------------------------------------------------------------------------------------")
print("Validated Metadata Output:")
display(output_sorted_validated.style.hide(axis="index"))


# Step 2: Load Reviewed Metadata into DUOS

In [ ]:
#############################################
## Functions
#############################################

def format_list(input_list, min_items):
    if input_list:
        if isinstance(input_list, list):
            return input_list
        elif isinstance(input_list, str):
            return format_list(ast.literal_eval(input_list), min_items)
        else:
            return []
    else:
        if min_items > 0:
            i = 0
            temp_list = []
            while i < min_items:
                temp_list.append("Unknown")
                i += 1
            return temp_list
        else:
            return []
    
def format_file_types(ft_list, fe):
    if ft_list:
        output_list = []
        formatted_ft_list = format_list(ft_list, 0)
        for ft in formatted_ft_list:
            ft_dict = {"fileType": ft}
            if fe:
                ft_dict["functionalEquivalence"] = fe
            else:
                ft_dict["functionalEquivalence"] = "Unknown"
            output_list.append(ft_dict)
        return output_list
    else:
        return []
    
def upload_to_duos(input_file, token, env, study_upload_list, preview_only, include_anvil_in_preview):
    
    # Determine the target URL from the env variable
    if env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"
    
    # Pull down specified file from the cloud
    results_log = []
    print(f"Downloading input file {input_file}...")
    try:
        input_df = pd.read_csv(input_file, delimiter = "\t", encoding='unicode_escape')
        input_df = input_df.astype(object).where(pd.notnull(input_df),None)
        input_df.fillna("",inplace=True)
        input_dict = input_df.to_dict(orient="records")
        results_log.append(["Input File Download", "Succeeded", ""])
    except Exception as e:
        msg = f"Error downloading input file ({input_file}): {str(e)}"
        results_log.append(["Input File Download", "Failed", msg])
        print(msg)
        return results_log

    # Parse and build DUOS schema for inputted file
    print("Parsing input file and formatting into DUOS schema...")
    upload_dict = {}
    study_lookup = {}
    try:
        # Determine data submitter id
        response = requests.get(
            url=f"{url}/api/user/me",
            headers={"Authorization": f"Bearer {token}"}
        ).json()
        data_submitter_id = response["userId"]
        # Build dictionary for upload
        existing_dataset_cnt = 0
        new_dataset_cnt = 0
        for input_entry in input_dict:
            snapshot_id = input_entry["snapshot_id"]
            dataset_id = str(int(input_entry["target_dataset_id"])) if input_entry["target_dataset_id"] else input_entry["target_dataset_id"]
            dataset_name = input_entry["consentGroups.consentGroupName"]
            study_id = str(int(input_entry["target_study_id"])) if input_entry["target_study_id"] else input_entry["target_study_id"]
            study_name = input_entry["studyName"]
            if study_id:
                study_lookup[study_name] = study_id
            tar_ds_id = dataset_id if dataset_id else "ID_TBD"
            tar_st_id = study_id if study_id else "ID_TBD"
            access_type = input_entry["consentGroups.accessManagement"]
            file_types_dict = []
            if input_entry.get("consentGroups.fileTypes"):
                file_types_dict = json.loads(input_dict[0]["consentGroups.fileTypes"])
            print(f"Parsing and formatting metadata for snapshot {snapshot_id} from the input file. Target study is: {study_name} ({tar_st_id}). Target consent group is: {dataset_name} ({tar_ds_id})")

            # If this is an existing dataset in the specified existing study, provide limited consent group information (for updates only)
            if dataset_id:
                existing_dataset_cnt += 1
                consent_group_dict = {
                            "consentGroupName": dataset_name,
                            "datasetId": dataset_id,
                            "numberOfParticipants": input_entry["consentGroups.numberOfParticipants"],
                            "dataLocation": input_entry["consentGroups.dataLocation"],
                            "url": input_entry["consentGroups.url"],
                            "fileTypes": []
                            #"fileTypes": format_file_types(input_entry["consentGroups.fileTypes.fileType"], input_entry["consentGroups.fileTypes.functionalEquivalence"]) --> Enumeration, exclude for now
                    }
            # If this is a new dataset that is open access, provide limited consent group information
            elif access_type == "open":
                new_dataset_cnt += 1
                consent_group_dict = {
                            "consentGroupName": dataset_name,
                            "accessManagement": access_type,
                            "numberOfParticipants": input_entry["consentGroups.numberOfParticipants"],
                            "dataLocation": input_entry["consentGroups.dataLocation"],
                            "url": input_entry["consentGroups.url"],
                            "fileTypes": []
                            #"fileTypes": format_file_types(input_entry["consentGroups.fileTypes.fileType"], input_entry["consentGroups.fileTypes.functionalEquivalence"]) --> Enumeration, exclude for now
                    }
            # If this is a new dataset that is external access, provide consent group information (minus the dac_id)
            elif access_type == "external":
                new_dataset_cnt += 1
                consent_group_dict = {
                            "consentGroupName": dataset_name,
                            "accessManagement": access_type,
                            "numberOfParticipants": input_entry["consentGroups.numberOfParticipants"],
                            "generalResearchUse": input_entry["consentGroups.generalResearchUse"],
                            "hmb": input_entry["consentGroups.hmb"],
                            "diseaseSpecificUse": format_list(input_entry["consentGroups.diseaseSpecificUse"], 0),
                            "gs": input_entry["consentGroups.gs"],
                            "poa": input_entry["consentGroups.poa"],
                            "nmds": input_entry["consentGroups.nmds"],
                            "gso": input_entry["consentGroups.gso"],
                            "pub": input_entry["consentGroups.pub"],
                            "col": input_entry["consentGroups.col"],
                            "irb": input_entry["consentGroups.irb"],
                            "npu": input_entry["consentGroups.npu"],
                            "otherPrimary": input_entry["consentGroups.otherPrimary"],
                            #"otherSecondary": input_entry["consentGroups.otherSecondary"], --> Excluding for now, per JL's request
                            #"mor": input_entry["consentGroups.mor"], --> Date formatting validation for morDate, exclude for now
                            #"morDate": input_entry["consentGroups.morDate"], --> Date formatting validation, exclude for now
                            "dataLocation": input_entry["consentGroups.dataLocation"],
                            "url": input_entry["consentGroups.url"],
                            "fileTypes": []
                            #"fileTypes": format_file_types(input_entry["consentGroups.fileTypes.fileType"], input_entry["consentGroups.fileTypes.functionalEquivalence"]) --> Enumeration, exclude for now
                    }
            # If this is a new dataset that is NOT open or external access, provide the full consent group information
            else:
                new_dataset_cnt += 1
                consent_group_dict = {
                            "consentGroupName": dataset_name,
                            "dataAccessCommitteeId": input_entry["dacId"],
                            "accessManagement": access_type,
                            "numberOfParticipants": input_entry["consentGroups.numberOfParticipants"],
                            "generalResearchUse": input_entry["consentGroups.generalResearchUse"],
                            "hmb": input_entry["consentGroups.hmb"],
                            "diseaseSpecificUse": format_list(input_entry["consentGroups.diseaseSpecificUse"], 0),
                            "gs": input_entry["consentGroups.gs"],
                            "poa": input_entry["consentGroups.poa"],
                            "nmds": input_entry["consentGroups.nmds"],
                            "gso": input_entry["consentGroups.gso"],
                            "pub": input_entry["consentGroups.pub"],
                            "col": input_entry["consentGroups.col"],
                            "irb": input_entry["consentGroups.irb"],
                            "npu": input_entry["consentGroups.npu"],
                            "otherPrimary": input_entry["consentGroups.otherPrimary"],
                            #"otherSecondary": input_entry["consentGroups.otherSecondary"], --> Excluding for now, per JL's request
                            #"mor": input_entry["consentGroups.mor"], --> Date formatting validation for morDate, exclude for now
                            #"morDate": input_entry["consentGroups.morDate"], --> Date formatting validation, exclude for now
                            "dataLocation": input_entry["consentGroups.dataLocation"],
                            "url": input_entry["consentGroups.url"],
                            "fileTypes": []
                            #"fileTypes": format_file_types(input_entry["consentGroups.fileTypes.fileType"], input_entry["consentGroups.fileTypes.functionalEquivalence"]) --> Enumeration, exclude for now
                    }

            # If the study associated with the record is not already in the upload dictionary, create a new study dict and append the consent group dict
            study_dict = {}
            consent_group_list = []
            if study_name not in upload_dict.keys():
                consent_group_list.append(consent_group_dict)
                study_dict = {
                    "studyName": study_name,
                    #"studyType": input_entry["studyType"], --> Enumeration, exclude for now
                    "studyDescription": input_entry["studyDescription"].replace("\\n", "\n"),
                    "dataTypes": format_list(input_entry["dataTypes"], 1),
                    "phenotypeIndication": input_entry["phenotypeIndication"],
                    "species": input_entry["species"],
                    "piName": input_entry["piName"] if input_entry["piName"] else "NA",
                    "dataSubmitterUserId": data_submitter_id,
                    "dataCustodianEmail": format_list(input_entry["dataCustodianEmail"], 0),
                    "publicVisibility": input_entry["publicVisibility"],
                    "nihAnvilUse": input_entry["nihAnvilUse"],
                    "submittingToAnvil": input_entry["submittingToAnvil"],
                    "dbGaPPhsID": input_entry["dbGaPPhsID"],
                    "dbGaPStudyRegistrationName": input_entry["studyName"],
                    #"embargoReleaseDate": input_entry["embargoReleaseDate"], --> Date formatting validation, exclude for now
                    "sequencingCenter": input_entry["sequencingCenter"],
                    "piEmail": input_entry["piEmail"],
                    #"piInstitution": input_entry["piInstitution"], --> Integer ID for registered institutions, exclude for now
                    "piInstitution": 0,
                    "nihGrantContractNumber": "Unknown", # Required currently
                    "nihICsSupportingStudy": format_list(input_entry["nihICsSupportingStudy"], 0),
                    "nihProgramOfficerName": input_entry["nihProgramOfficerName"],
                    "nihInstitutionCenterSubmission": input_entry["nihInstitutionCenterSubmission"],
                    "nihInstitutionalCertificationFileName": input_entry["nihInstitutionalCertificationFileName"],
                    "nihGenomicProgramAdministratorName": input_entry["nihGenomicProgramAdministratorName"],
                    "collaboratingSites": format_list(input_entry["collaboratingSites"], 0),
                    "alternativeDataSharingPlan": input_entry["alternativeDataSharingPlan"],
                    "alternativeDataSharingPlanExplanation": input_entry["alternativeDataSharingPlanExplanation"],
                    "alternativeDataSharingPlanReasons": ["Other"] if input_entry["alternativeDataSharingPlan"] == True and input_entry["alternativeDataSharingPlanReasons"] == "[]" else format_list(input_entry["alternativeDataSharingPlanReasons"], 0), 
                    "consentGroups": consent_group_list
                }
                upload_dict[study_name] = study_dict
            # If the study is already in the upload dictionary, create an updated study dict and extend its list of consent groups
            else:
                study_dict = upload_dict[study_name].copy()
                for consent_group in study_dict["consentGroups"]:
                    if consent_group["consentGroupName"] != consent_group_dict["consentGroupName"]:
                        consent_group_list.append(consent_group)
                consent_group_list.append(consent_group_dict)
                study_dict["consentGroups"] = consent_group_list
                upload_dict[study_name] = study_dict
        msg = f"Input file formatting complete. Existing Datasets: {existing_dataset_cnt} New Datasets: {new_dataset_cnt}"
        print(msg)
        results_log.append(["Input File Parsing and Formatting", "Succeeded", msg])
    except Exception as e:
        msg = f"Error parsing and formatting input file: {str(e)}"
        results_log.append(["Input File Parsing and Formatting", "Failed", msg])
        print(msg)
        #return results_log

    # Loop through studies to upload and augment with an missing existing datasets
    print("Augmenting upload set with missing existing datasets...")
    dataset_upload_aug_list = []
    for study in upload_dict.keys():
        if study in study_upload_list or len(study_upload_list) == 0:
            study_id = study_lookup.get(study)
            if study_id:
                try:
                    study_datasets_in_duos = set()
                    study_datasets_in_input = set()
                    study_datasets_diff = set()
                    study_details = requests.get(
                            url=f"{url}/api/dataset/study/registration/{study_id}",
                            headers={"Authorization": f"Bearer {token}"}
                        ).json()
                    for dataset in study_details["consentGroups"]:
                        if dataset.get("datasetId"):
                            study_datasets_in_duos.add(dataset.get("datasetId"))
                    for dataset in upload_dict[study]["consentGroups"]:
                        if dataset.get("datasetId"):
                            study_datasets_in_input.add(dataset.get("datasetId"))
                    for dataset_in_duos in study_datasets_in_duos:
                        if str(dataset_in_duos) not in study_datasets_in_input:
                            study_datasets_diff.add(dataset_in_duos)
                    # Add missing datasets to the upload dict
                    temp_cg = upload_dict[study]["consentGroups"].copy()
                    for missing_dataset_id in study_datasets_diff:
                        dataset_upload_aug_list.append(missing_dataset_id)
                        dataset_details = requests.get(
                            url=f"{url}/api/dataset/v2/{missing_dataset_id}",
                            headers={"Authorization": f"Bearer {token}"}
                        ).json()
                        name = dataset_details["name"]
                        data_loc = ""
                        data_loc_url = ""
                        num_participants = 0
                        for prop_entry in dataset_details["properties"]:
                            if prop_entry["propertyName"] == "Data Location":
                                data_loc = prop_entry["propertyValue"]
                            elif prop_entry["propertyName"] == "# of participants":
                                num_participants = prop_entry["propertyValue"]
                            elif prop_entry["propertyName"] == "URL":
                                data_loc_url = prop_entry["propertyValue"] 
                        consent_group_dict = {
                            "consentGroupName": dataset_details["name"],
                            "datasetId": missing_dataset_id,
                            "numberOfParticipants": num_participants,
                            "dataLocation": data_loc,
                            "url": data_loc_url,
                            "fileTypes": []
                        }
                        temp_cg.append(consent_group_dict)
                    upload_dict[study]["consentGroups"] = temp_cg
                except:
                    print(f"WARNING: Issue retrieving study details for study_id {study_id}. May cause issues with upload downstream.")

    # Preview of upload input dictionary
    if preview_only:
        # Build a preview of the upload dictionary data
        print("Building upload set preview...")
        output_preview = []
        study_id_set = set()
        for study_name, study_dict in upload_dict.items():
            if study_name in study_upload_list or len(study_upload_list) == 0:
                study_id = study_lookup.get(study_name) if study_lookup.get(study_name) else f"ID_TBD ({study_name})"
                study_id_set.add(study_id)
                study_phs = study_dict["dbGaPPhsID"]
                public_visibility = study_dict["publicVisibility"]
                for consent_group in study_dict["consentGroups"]:
                    dataset_id = consent_group.get("datasetId")
                    dataset_name = consent_group.get("consentGroupName")
                    snapshot_url = consent_group.get("url")
                    if snapshot_url and "https://data.terra.bio/snapshots/" in snapshot_url and dataset_id in dataset_upload_aug_list:
                        snapshot_id = snapshot_url.replace("https://data.terra.bio/snapshots/", "")
                        record_src = "upload_aug"
                    elif snapshot_url and "https://data.terra.bio/snapshots/" in snapshot_url:
                        snapshot_id = snapshot_url.replace("https://data.terra.bio/snapshots/", "")
                        record_src = "upload"
                    else:
                        snapshot_id = ""
                        record_src = "upload_aug"
                    if dataset_id:
                        dataset_details = requests.get(
                            url=f"{url}/api/dataset/v2/{dataset_id}",
                            headers={"Authorization": f"Bearer {token}"}
                        ).json()
                        dataset_identifier = dataset_details.get("datasetIdentifier")
                        dac_id = dataset_details.get("dacId") if dataset_details.get("dacId") else ""
                        duos_data_use_dict = dataset_details.get("dataUse")
                        du_gru = duos_data_use_dict.get("generalUse") if duos_data_use_dict.get("generalUse") else False
                        du_hmb = duos_data_use_dict.get("hmbResearch") if duos_data_use_dict.get("hmbResearch") else False
                        du_disease = duos_data_use_dict.get("diseaseRestrictions") if duos_data_use_dict.get("diseaseRestrictions") else []
                        du_poa = duos_data_use_dict.get("populationOriginsAncestry") if duos_data_use_dict.get("populationOriginsAncestry") else False
                        du_ethics = duos_data_use_dict.get("ethicsApprovalRequired") if duos_data_use_dict.get("ethicsApprovalRequired") else False
                        du_collab = duos_data_use_dict.get("collaboratorRequired") if duos_data_use_dict.get("collaboratorRequired") else False
                        du_geog = duos_data_use_dict.get("geographicalRestrictions") if duos_data_use_dict.get("geographicalRestrictions") else ""
                        du_genetic = duos_data_use_dict.get("geneticStudiesOnly") if duos_data_use_dict.get("geneticStudiesOnly") else False
                        du_pub = duos_data_use_dict.get("publicationResults") if duos_data_use_dict.get("publicationResults") else False
                        du_nmds = duos_data_use_dict.get("methodsResearch") if duos_data_use_dict.get("methodsResearch") else False
                        du_npu = duos_data_use_dict.get("nonProfitUse") if duos_data_use_dict.get("nonProfitUse") else False
                        du_other = duos_data_use_dict.get("other") if duos_data_use_dict.get("other") else ""
                        access_management = ""
                        for prop_entry in dataset_details["properties"]:
                            if prop_entry["propertyName"] == "Access Management":
                                access_management = prop_entry["propertyValue"]
                                break
                    else:
                        dataset_id = f"ID_TBD ({dataset_name})"
                        dataset_identifier = "ID_TBD"
                        dac_id = consent_group.get("dataAccessCommitteeId") if consent_group.get("dataAccessCommitteeId") else ""
                        du_gru = consent_group.get("generalResearchUse") if consent_group.get("generalResearchUse") else False
                        du_hmb = consent_group.get("hmb") if consent_group.get("hmb") else False
                        du_disease = consent_group.get("diseaseSpecificUse") if consent_group.get("diseaseSpecificUse") else []
                        du_poa = consent_group.get("poa") if consent_group.get("poa") else False
                        du_ethics = consent_group.get("irb") if consent_group.get("irb") else False
                        du_collab = consent_group.get("col") if consent_group.get("col") else False
                        du_geog = consent_group.get("gs") if consent_group.get("gs") else ""
                        du_genetic = consent_group.get("gso") if consent_group.get("gso") else False
                        du_pub = consent_group.get("pub") if consent_group.get("pub") else False
                        du_nmds = consent_group.get("nmds") if consent_group.get("nmds") else False
                        du_npu = consent_group.get("npu") if consent_group.get("npu") else False
                        du_other = consent_group.get("otherPrimary") if consent_group.get("otherPrimary") else ""
                        access_management = consent_group.get("accessManagement") if consent_group.get("accessManagement") else "" 
                    output_preview.append([study_id, study_name, study_phs, dataset_id, dataset_identifier, dataset_name, dac_id, access_management, du_gru, du_hmb, du_disease, du_poa, du_ethics, du_collab, du_geog, du_genetic, du_pub, du_nmds, du_npu, du_other, snapshot_id, "", public_visibility, record_src])

        # Add in AnVIL datasets not in the upload dictionary 
        if len(study_upload_list) == 0 and include_anvil_in_preview:
            anvil_datasets_in_duos = get_anvil_datasets_from_duos(token, env)
            for dataset in anvil_datasets_in_duos:
                dataset_exists = False
                for output_dataset in output_preview:
                    if str(dataset[3]) == str(output_dataset[3]):
                        dataset_exists = True
                        break
                if not dataset_exists:
                    rec_to_add = dataset.copy()
                    rec_to_add.append("prod_add")
                    output_preview.append(rec_to_add)

        # Display output preview
        df_results = pd.DataFrame(output_preview, columns = ["Study ID", "Study Name", "Study PHS", "Dataset ID", "Dataset Identifier", "Dataset Name", "DAC ID", "Access", "GRU", "HMB", "DS", "POA", "IRB", "COL", "GS", "GSO", "PUB", "NMDS", "NPU", "OTHER", "Snapshot ID", "Workspace", "Public Visibility", "Record Source"])
        print("\nOutput Preview:")
        display(df_results)
    
    else:
        print("Uploading studies to DUOS...")
        for study in upload_dict.keys():
            if study in study_upload_list or len(study_upload_list) == 0:
                study_id = study_lookup.get(study)
                # For studies that don't exist in DUOS, create a new study
                if not study_id:
                    print("Study does NOT currently exist in DUOS. Registering new study...")
                    #print(json.dumps(upload_dict[study]))
                    try:
                        new_study_response = requests.post(
                            url=f"{url}/api/dataset/v3",
                            headers={"Authorization": f"Bearer {token}"},
                            files = {
                                "dataset": json.dumps(upload_dict[study]),
                                "alternativeDataSharingPlan": "",
                                "consentGroups[0].nihInstitutionalCertificationFile": ""  
                            }
                        ).json()
                        if new_study_response.get("studyId"):
                            study_id = new_study_response["studyId"]
                            msg = f"Study registration succeeded! Study Id: {study_id}"
                            results_log.append([f"New Study Registration - {study}", "Succeeded", msg])
                            print(msg)
                        else:
                            err_msg = new_study_response["message"]
                            msg = f"Study registration failed: {err_msg}"
                            results_log.append([f"New Study Registration - {study}", "Failed", msg])
                            print(msg)
                    except Exception as e:
                        msg = f"Study registration failed: {str(e)}"
                        results_log.append([f"New Study Registration - {study}", "Failed", msg])
                        print(msg)
                # For studies that already exist in DUOS, update the existing study
                else:
                    print("Study DOES currently exist in DUOS. Updating study...")
                    try:
                        # Update study in DUOS
#                         print(study_id)
#                         print(json.dumps(upload_dict[study]))
                        update_study_response = requests.put(
                            url=f"{url}/api/dataset/study/{study_id}",
                            headers={"Authorization": f"Bearer {token}"},
                            files = {
                                "dataset": json.dumps(upload_dict[study]),
                                "alternativeDataSharingPlan": "",
                                "consentGroups[0].nihInstitutionalCertificationFile": ""  
                            }
                        ).json()   
                        if update_study_response.get("studyId"):
                            study_id = update_study_response["studyId"]
                            msg = f"Study registration succeeded! Study Id: {study_id}"
                            results_log.append([f"Study Registration Update - {study}", "Succeeded", msg])
                            print(msg)
                        else:
                            err_msg = update_study_response["message"]
                            msg = f"Study registration failed: {err_msg}"
                            results_log.append([f"Study Registration Update - {study}", "Failed", msg])
                            print(msg)
                    except Exception as e:
                        msg = f"Study registration failed: {str(e)}"
                        results_log.append([f"Study Registration Update - {study}", "Failed", msg])
                        print(msg)
    
    # Return results
    return results_log


#############################################
## Input Parameters
#############################################

# Cloud path to file to process
input_file_gcs_path = "gs://fc-2a9eefc3-0302-427f-9ac3-82f078741c03/dataset_metadata/input/metadata_review_anvil12_20251016.tsv"

# User token (use gcloud auth print-access-token | pbcopy to get this)
duos_token = ""

# Environment
duos_env = "prod"

# Study Upload List (to limit the studies upload, leave empty for all)
study_upload_list = [
]

# Specifies whether the upload should run (False) or if only a preview of the upload should be displayed
preview_only = False

# For preview_only = True cases, specifies whether AnVIL datasets not included in the upload should also be displayed
include_anvil_in_preview = False

#############################################
## Execution
#############################################

upload_results = upload_to_duos(input_file_gcs_path, duos_token, duos_env, study_upload_list, preview_only, include_anvil_in_preview)
df_results = pd.DataFrame(upload_results, columns = ["Item", "Status", "Message"])
print("\nUpload Results:")
display(df_results)


# Step 3: Sync DUOS Datasets with TDR Snapshots
Based on the provided pairs of Snapshots and DUOS IDs:
- If no DUOS ID is specified, the existing DUOS ID will be removed from the snapshot. The DUOS group will NOT be automatically removed from the snapshot's auth domain group, given that auth domain group may be shared with multiple snapshots. 
- If a DUOS ID is specified and the snapshot doesn't currently have one, the DUOS ID will be linked to the snapshot, the DUOS group will be added to the snapshot auth domain group, and the DUOS dataset registration will be updated with to point to the snapshot. 
- If a DUOS ID is specified and the snapshot currently has a different one, the new DUOS ID will be linked to the snapshot, the new DUOS group will be added to the snapshot auth domain group, the old DUOS group will be removed from the snapshot auth domain group, and the DUOS dataset registration will be updated to point to the snapshot.

In [ ]:
#############################################
## Functions
#############################################

def sync_duos_ids_and_snapshots(duos_token, duos_env, non_snapshot_override, snapshot_duos_list):
    results_log = []

    # Determine the target URL from the env variable
    if duos_env == "prod":
        duos_url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        duos_url = "https://consent.dsde-dev.broadinstitute.org"
    
    # Loop through input snapshots and link DUOS IDs to them
    print("Syncing DUOS IDs to Snapshots...")
    for ss_duos_entry in snapshot_duos_list:
        
        # Set variables
        snapshot_id = ss_duos_entry[0]
        new_duos_id = ss_duos_entry[1]
        print(f"Processing snapshot ID = {snapshot_id}")
        if new_duos_id:
            duos_action = "LINK"
        else:
            duos_action = "UNLINK"
        new_duos_group = ""
        current_duos_group = ""
        
        # Establish TDR client
        creds, project = google.auth.default()
        auth_req = google.auth.transport.requests.Request()
        creds.refresh(auth_req)
        api_client = refresh_tdr_api_client()
        snapshots_api = data_repo_client.SnapshotsApi(api_client=api_client)
        duos_api = data_repo_client.DuosApi(api_client=api_client)
        
        # Establish DUOS token to use (if no token provided, grab one for the pet SA)
        if duos_token:
            token = duos_token
        else:
            token = creds.token
        
        # Get current DUOS ID on the snapshot
        print(f"\tRetrieving original DUOS ID from snapshot {snapshot_id} (if any)...")
        current_duos_id = "Unknown"
        attempt_counter = 0
        while attempt_counter <= 2:
            try:
                response = snapshots_api.retrieve_snapshot(id=snapshot_id, include=["DUOS"]).to_dict()
                duos_firecloud_group = response["duos_firecloud_group"]
                if duos_firecloud_group:
                    current_duos_id = response["duos_firecloud_group"].get("duos_id")
                break
            except Exception as e:
                msg = f"Error retrieving original DUOS ID from Snapshot: {str(e)}"
                if attempt_counter >= 2:
                    results_log.append([f"DUOS ID to Snapshot Linkage ({snapshot_id} - {duos_id})", "Failed", msg])
                    break
                sleep(5)
                attempt_counter += 1
            
        # Processing DUOS_ID-Snapshot sync
        if duos_action == "LINK":
            # Link the DUOS ID to the snapshot
            print(f"\tLinking DUOS ID {new_duos_id} to snapshot...")
            if new_duos_id != current_duos_id: 
                attempt_counter = 0
                while attempt_counter <= 2:
                    try:
                        response = snapshots_api.link_duos_dataset_to_snapshot(id=snapshot_id, duos_id=new_duos_id).to_dict()
                        if response.get("linked"):
                            results_log.append([f"DUOS ID to Snapshot Linkage ({snapshot_id} - {new_duos_id})", "Success", ""])
                            break
                        elif response.get("message"):
                            response_message = response.get("message")
                            msg = f"Error linking DUOS ID to Snapshot: {response_message}"
                            if attempt_counter >= 2:
                                results_log.append([f"DUOS ID to Snapshot Linkage ({snapshot_id} - {new_duos_id})", "Failed", msg])
                                break
                    except Exception as e:
                        msg = f"Error linking DUOS ID to Snapshot: {str(e)}"
                        if attempt_counter >= 2:
                            results_log.append([f"DUOS ID to Snapshot Linkage ({snapshot_id} - {new_duos_id})", "Failed", msg])
                        sleep(5)
                        attempt_counter += 1   
                sleep(15)
            else:
                print(f"\t\tLinkage skipped (DUOS ID {new_duos_id} is already linked to the snapshot)")
            
            # Get DUOS user group associated with new and current duos id
            if new_duos_id:
                print(f"\tFetching DUOS user group for new DUOS ID {new_duos_id}...")
                attempt_counter = 0
                while attempt_counter <= 2:
                    try:  
                        response = duos_api.retrieve_duos_firecloud_group(duos_id=new_duos_id).to_dict()
                        new_duos_group = response["firecloud_group_email"]
                        results_log.append([f"DUOS User Group Fetching ({new_duos_id})", "Success", new_duos_group])
                        break
                    except Exception as e:
                        msg = f"Error fetching DUOS user group for DUOS ID {new_duos_id}: {str(e)}"
                        if attempt_counter >= 2:
                            results_log.append([f"DUOS User Group Fetching ({new_duos_id})", "Failed", msg])
                        sleep(5)
                        attempt_counter += 1 
            if current_duos_id != "Unknown":
                if new_duos_id == current_duos_id:
                    print(f"\tNew DUOS ID matches original DUOS ID on snapshot, so no additional DUOS user group to fetch.")
                    current_duos_group = new_duos_group
                else:
                    print(f"\tFetching DUOS user group for original DUOS ID {current_duos_id}...")
                    attempt_counter = 0
                    while attempt_counter <= 2:
                        try:  
                            response = duos_api.retrieve_duos_firecloud_group(duos_id=current_duos_id).to_dict()
                            current_duos_group = response["firecloud_group_email"]
                            results_log.append([f"DUOS User Group Fetching ({current_duos_id})", "Success", current_duos_group])
                            break
                        except Exception as e:
                            msg = f"Error fetching DUOS user group for DUOS ID {current_duos_id}: {str(e)}"
                            if attempt_counter >= 2:
                                results_log.append([f"DUOS User Group Fetching ({current_duos_id})", "Failed", msg])
                            sleep(5)
                            attempt_counter += 1 

            # Add the DUOS user group to any DAC groups on the snapshot
            print(f"\tAdding DUOS user group {new_duos_group} to snapshot DAC user group(s)...")
            dac_groups = []
            try:
                response = snapshots_api.retrieve_snapshot_policies(id=snapshot_id).to_dict()
                if response.get("auth_domain"):
                    dac_groups = response["auth_domain"]
                if dac_groups:
                    print(f"\t\tDAC user group(s) found on snapshot: {dac_groups}.")
                    for dac_group in dac_groups:
                        response = requests.put(
                            url=f"https://api.firecloud.org/api/groups/{dac_group}/member/{new_duos_group}",
                            headers={"Authorization": f"Bearer {creds.token}"}
                        )
                        if response.status_code != 204:
                            results_log.append([f"DUOS Group to DAC Group Addition ({new_duos_group} - {dac_group})", "Failed", "Error adding DUOS group to DAC group."])
                        else:
                            results_log.append([f"DUOS Group to DAC Group Addition ({new_duos_group} - {dac_group})", "Success", ""])
                else:
                    msg = f"No DAC user group(s) found on snapshot."
                    print(f"\t\t{msg}")
                    results_log.append([f"DUOS Group to DAC Group Addition ({snapshot_id} - {new_duos_id})", "Warning", msg])   
            except Exception as e:
                msg = f"Error adding DUOS Group to DAC Group: {str(e)}"
                results_log.append([f"DUOS Group to DAC Group Addition ({snapshot_id} - {new_duos_id})", "Failed", msg])
                
            # Retrieve DUOS registration
            print(f"\tRetrieving DUOS dataset registration for DUOS ID {new_duos_id}...")
            duos_dataset_id = ""
            try:
                dataset_details = requests.get(
                    url=f"{duos_url}/api/tdr/{new_duos_id}",
                    headers={"Authorization": f"Bearer {token}"}
                ).json()
                duos_dataset_id = dataset_details["datasetId"]
                duos_dataset_name = dataset_details["name"]
                data_location = "non_snapshot"
                for dataset_property in dataset_details["properties"]:
                    if dataset_property["propertyName"] == "Data Location" and dataset_property["propertyValue"] == "TDR Location":
                        data_location = "snapshot"
                        break
                results_log.append([f"DUOS Dataset Retrieval ({snapshot_id} - {new_duos_id})", "Success", f"Dataset_id = {duos_dataset_id}"])
            except Exception as e:
                msg = f"Error retrieving DUOS dataset registration: {str(e)}"
                results_log.append([f"DUOS Dataset Retrieval ({snapshot_id} - {new_duos_id})", "Failed", msg])
            
            # Update snapshot on the DUOS registration
            print(f"\tUpdating DUOS dataset registration for DUOS ID {new_duos_id}...")
            if data_location != "snapshot" and non_snapshot_override == False:
                msg = "Non-snapshot data location with non_snapshot_override parameter set to False. Skipping update."
                print(f"\t\t{msg}")
                results_log.append([f"DUOS Dataset Patch ({snapshot_id} - {new_duos_id})", "Skipped", msg])
            else:
                if duos_dataset_id and duos_dataset_name:
                    payload = {
                        "name": duos_dataset_name,
                        "properties": [{
                            "propertyName": "URL",
                            "propertyValue": f"https://data.terra.bio/snapshots/{snapshot_id}",
                            "schemaProperty": "url",
                            "propertyType": "String"
                        }, {
                            "propertyName": "Data Location",
                            "propertyValue": "TDR Location",
                            "schemaProperty": "dataLocation",
                            "propertyType": "String"
                        }]
                    }
                    try:
                        dataset_patch_response = requests.patch(
                            url=f"{duos_url}/api/dataset/{duos_dataset_id}",
                            headers={"Authorization": f"Bearer {token}"},
                            json=payload
                        )
                        if dataset_patch_response.status_code == 200:
                            results_log.append([f"DUOS Dataset Patch ({snapshot_id} - {new_duos_id})", "Success", ""])
                        elif dataset_patch_response.status_code == 304:
                            results_log.append([f"DUOS Dataset Patch ({snapshot_id} - {new_duos_id})", "Success", "Dataset not modified. No patch required."])
                        else:
                            err = dataset_patch_response.text
                            msg = f"Error patching DUOS Dataset: {err}"
                            results_log.append([f"DUOS Dataset Patch ({snapshot_id} - {new_duos_id})", "Failed", msg])
                    except Exception as e:
                        msg = f"Error patching DUOS Dataset: {str(e)}"
                        results_log.append([f"DUOS Dataset Patch ({snapshot_id} - {new_duos_id})", "Failed", msg])
                else:
                    results_log.append([f"DUOS Dataset Patch ({snapshot_id} - {new_duos_id})", "Failed", "Missing DUOS dataset ID or name"])     
        
        elif duos_action == "UNLINK" and current_duos_id != "Unknown":
            # Unlink the current DUOS ID from the snapshot (if any)
            print(f"\tNo DUOS ID specified. Removing current DUOS ID {current_duos_id} from the snapshot...")
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    response = snapshots_api.unlink_duos_dataset_from_snapshot(id=snapshot_id).to_dict()
                    if response.get("message"):
                        response_message = response.get("message")
                        msg = f"Error removing DUOS ID from Snapshot: {response_message}"
                        if attempt_counter >= 2:
                            results_log.append([f"Remove existing DUOS ID from Snapshot ({snapshot_id})", "Failed", msg])
                            break
                    else:
                        results_log.append([f"Remove existing DUOS ID from Snapshot ({snapshot_id})", "Success", ""])
                        break
                except Exception as e:
                    msg = f"Error removing DUOS ID from Snapshot: {str(e)}"
                    if attempt_counter >= 2:
                        results_log.append([f"Remove existing DUOS ID from Snapshot ({snapshot_id})", "Failed", msg])
                    sleep(5)
                    attempt_counter += 1
        else:
            print(f"\tNo DUOS ID specified and no current DUOS ID to remove from the snapshot.")
            results_log.append([f"Remove existing DUOS ID from Snapshot ({snapshot_id})", "Success", ""])
            
        # If a new duos ID is replacing an existing duos ID, remove the current DUOS user group from any DAC groups on the snapshot
        if new_duos_group and current_duos_group and new_duos_group != current_duos_group:
            print(f"\tRemoving DUOS user group {current_duos_group} from snapshot DAC user group(s)...")
            dac_groups = []
            try:
                response = snapshots_api.retrieve_snapshot_policies(id=snapshot_id).to_dict()
                if response.get("auth_domain"):
                    dac_groups = response["auth_domain"]
                if dac_groups:
                    print(f"\t\t\t- DAC user group(s) found on snapshot: {dac_groups}.")
                    for dac_group in dac_groups:
                        response = requests.delete(
                            url=f"https://api.firecloud.org/api/groups/{dac_group}/member/{current_duos_group}",
                            headers={"Authorization": f"Bearer {creds.token}"}
                        )
                        if response.status_code != 204:
                            results_log.append([f"DUOS Group Removal from DAC Group ({current_duos_group} - {dac_group})", "Failed", "Error removing DUOS group from DAC group."])
                        else:
                            results_log.append([f"DUOS Group Removal from DAC Group ({current_duos_group} - {dac_group})", "Success", ""])
                else:
                    msg = f"No DAC user group(s) found on snapshot."
                    print(f"\t\t{msg}")
                    results_log.append([f"DUOS Group Removal from DAC Group ({current_duos_group} - N/A)", "Warning", msg])   
            except Exception as e:
                msg = f"Error removing DUOS Group from DAC Group: {str(e)}"
                results_log.append([f"DUOS Group Removal from DAC Group ({current_duos_group})", "Failed", msg])
                
    return results_log

#############################################
## Input Parameters
#############################################

# Token for use in DUOS (use "gcloud auth print-access-token | pbcopy" to get this)
duos_token = ""

# DUOS Environment
duos_env = "prod"

# If a snapshot is not the current data location in DUOS, whether the data location should be overwritten with a snapshot
non_snapshot_override = False

# Snapshot list
snapshot_duos_list = [
    #['snapshot_id', 'duos_id (or empty string to remove existing duos_id from snapshot)']
    ['0569d409-9716-4c02-bc00-5295c411bf88', 'DUOS-000195'],
    ['5781aa42-5281-4301-a10a-27a915937cbf', 'DUOS-000450'],
    ['26520fce-5a1b-407f-9d28-35b4603a5443', 'DUOS-000756'],
    ['a0f09578-13c4-43e9-9471-aa38628250da', 'DUOS-000758'],
    ['47a2db44-72f4-478c-b289-1c5de5f388b5', 'DUOS-000760'],
    ['b8f39bec-0137-485b-a6f2-e8c5cf8e80e8', 'DUOS-000761'],
    ['13d64983-7c5d-4175-bd9e-41605e104258', 'DUOS-000762'],
    ['b1c14449-3764-466d-8add-ee3850e590e8', 'DUOS-000474'],
    ['88bfb8e8-49d2-4eee-b27a-6ffa8d3a8958', 'DUOS-000471'],
    ['03160265-4962-4fe5-811c-f1cdfa45ac74', 'DUOS-000470'],
    ['1302d64a-4960-4c5d-bc76-c57f1896b73f', 'DUOS-000235'],
    ['4b79114d-8d84-426e-b48f-482d3606d5bd', 'DUOS-000485'],
    ['d6e055f7-6da2-44b7-9c14-da88bc992356', 'DUOS-000769'],
    ['010b4ea5-9dd4-4e2b-add8-8e69f7c509f4', 'DUOS-000766'],
    ['27bc0823-df40-4e30-a2e7-f3114592783f', 'DUOS-000767'],
    ['44ac2a8d-f16a-4d0c-ab89-6956331d290e', 'DUOS-000755'],
    ['e48ebc69-4481-49ab-a107-3836012755d8', 'DUOS-000757'],
    ['ab9e976f-6760-426c-bf85-15987dcd7a8a', 'DUOS-000759'],
]

#############################################
## Execution
#############################################

results = sync_duos_ids_and_snapshots(duos_token, duos_env, non_snapshot_override, snapshot_duos_list)
df_results = pd.DataFrame(results, columns = ["Item", "Status", "Message"])
print("\nLinking Results:")
display(df_results)

# DUOS Utilities

## Create SA User in DUOS

In [ ]:
#############################################
## Functions
#############################################

def establish_user(duos_env):
    # Establish credentials
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)

    # Create user
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"
    response = requests.post(
        url=f"{url}/api/user",
        headers={"Authorization": f"Bearer {creds.token}"}
    )
    
    # Return status code
    if response.status_code in [200, 201]:
        print("Success!")
    else:
        response_txt = response.text
        print(f"Error creating user: {response_txt}")
    
#############################################
## Input Parameters
#############################################

# Environment
duos_env = "prod"

#############################################
## Execution
#############################################

response = establish_user(duos_env)


## Create Study Lookup

In [ ]:
#############################################
## Input Parameters
#############################################

# User token (use gcloud auth print-access-token to get this)
duos_token = ""

# Environment
duos_env = "prod"


#############################################
## Execution
#############################################

if duos_env == "prod":
    url = "https://consent.dsde-prod.broadinstitute.org"
else:
    url = "https://consent.dsde-dev.broadinstitute.org"
    
# Iterate through studies sequentially
results = []
study_id = 1
missing_count = 0
print("Pulling study registrations...")
while missing_count < 20:
    try:
        print(f"Attempting to pull registration for study_id = {study_id}...")
        study_registration = requests.get(
            url=f"{url}/api/dataset/study/registration/{study_id}",
            headers={"Authorization": f"Bearer {duos_token}"}
        ).json()
        study_name = study_registration["studyName"]
        consent_groups = study_registration["consentGroups"]
        cg_length = 0
        if consent_groups:
            cg_length = len(consent_groups)
        results.append([study_id, study_name, cg_length])
        missing_count = 0
    except:
        print("\t- Unable to pull!")
        missing_count += 1
    study_id += 1

# Display study lookup
print("\nDUOS Studies: ")
results_df = pd.DataFrame(results, columns=["study_id", "study_name", "consent_group_count"])  
display(results_df)
        

In [ ]:
study_registration

## Delete Studies

In [ ]:
#############################################
## Input Parameters
#############################################

# User token (use gcloud auth print-access-token to get this)
duos_token = ""

# Environment
duos_env = "prod"

# Studies to delete
study_id_list = [
    '192'
]

#############################################
## Execution
#############################################

if duos_env == "prod":
    url = "https://consent.dsde-prod.broadinstitute.org"
else:
    url = "https://consent.dsde-dev.broadinstitute.org"

# Delete studies
for study_id in study_id_list:
    print(f"Deleting study ID {study_id}")
    response = requests.delete(
        url=f"{url}/api/dataset/study/{study_id}",
        headers={"Authorization": f"Bearer {duos_token}"} 
    )
    if response.status_code == 200:
        print("Study deleted successfully.")
    else:
        msg = response.json()["message"]
        print(f"Error deleting study: {msg}")
    

## Delete Datasets

In [ ]:
#############################################
## Input Parameters
#############################################

# User token (use "gcloud auth print-access-token | pbcopy" to get this)
duos_token = ""

# Environment
duos_env = "prod"

# Datasets to delete
dataset_id_list = [ 
    '833'
]

#############################################
## Execution
#############################################

if duos_env == "prod":
    url = "https://consent.dsde-prod.broadinstitute.org"
else:
    url = "https://consent.dsde-dev.broadinstitute.org"

# Delete datasets
for dataset_id in dataset_id_list:
    print(f"Deleting dataset ID {dataset_id}")
    # Delete Dataset
    response = requests.delete(
        url=f"{url}/api/dataset/{dataset_id}",
        headers={"Authorization": f"Bearer {duos_token}"} 
    )
    if response.status_code == 200:
        print("Dataset deleted successfully.")
    else:
        msg = response.json()["message"]
        print(f"Error deleting dataset: {msg}")
        

## Export All Datasets

In [ ]:
#############################################
## Functions
#############################################

def coalesce(*arg): 
    remove_list = ["", "NA", "N/A", "NONE", "TBD", "UNKNOWN", "UNSPECIFIED"]
    # update to remove N/A, None, Null, TBD
    for input_item in arg:
        if input_item is False or input_item == []:
            return input_item
        elif input_item:
            if isinstance(input_item, list):
                temp_list = [ele for ele in input_item if ele is not None and ele.upper() not in remove_list]
                if temp_list:
                    return temp_list
                else:
                    return []
            else:
                if str(input_item).upper() not in remove_list:
                    return input_item
    return None

def format_description(input_string):
    output_string = str(input_string) if input_string else ""
    output_string = re.sub("\n\n\t", " ", output_string)
    output_string = re.sub("\t", " ", output_string)
    output_string = output_string.replace("\t", " ")
    output_string = re.sub("\n", "\\n", output_string)
    output_string = output_string.replace("\n", "\\n")
    output_string = re.sub("study.cgi\?study_id=|.\/study.cgi\?study_id=", "https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=", output_string)
    return output_string

def export_duos_datasets(duos_token, duos_env):
    # Determine the target URL from the env variable
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"
        
    # If no duos token provided, grab one for the pet SA
    if duos_token:
        token = duos_token
    else:
        creds, project = google.auth.default()
        auth_req = google.auth.transport.requests.Request()
        creds.refresh(auth_req)
        token = creds.token
        
    # Pull a list of existing AnVIL studies and datasets from DUOS
    print(f"Start time: {datetime.datetime.now()}")
    results = []
    datasets = requests.get(
        url=f"{url}/api/dataset/v3",
        headers={"Authorization": f"Bearer {token}"}
    ).json()

    # Loop through datasets and capture information from dataset schema
    datasets_to_process = len(datasets)
    datasets_processed = 0
    dataset_details_records = []
    for dataset in datasets:
        # Establish credentials
        if duos_token:
            token = duos_token
        else:
            creds, project = google.auth.default()
            auth_req = google.auth.transport.requests.Request()
            creds.refresh(auth_req)
            token = creds.token
        
        # Process dataset
        datasets_processed += 1
        duos_identifier = dataset["identifier"]
        print(f"Processing dataset {duos_identifier} ({datasets_processed} of {datasets_to_process} datasets)...")
        duos_dict = {}
        duos_dict = requests.get(
            url=f"{url}/api/dataset/registration/{duos_identifier}",
            headers={"Authorization": f"Bearer {token}"}
        ).json()

        # Pull additional dataset details from DUOS if needed (to get data use info) 
        if not duos_dict.get("consentGroups"):
            duos_dict["consentGroups"] = [{"datasetId": None}]
        duos_dataset_id = duos_dict["consentGroups"][0].get("datasetId")
        duos_data_use_dict = {}
        dac_id = ""
        study_id = ""
        dataset_id = ""
        if duos_dataset_id:
            dataset_details = requests.get(
                url=f"{url}/api/dataset/v2/{duos_dataset_id}",
                headers={"Authorization": f"Bearer {token}"}
            ).json()
            duos_data_use_dict = dataset_details.get("dataUse")
            dac_id = dataset_details.get("dacId") if dataset_details.get("dacId") else ""
            study_id = dataset_details.get("studyId") if dataset_details.get("studyId") else ""
            dataset_id = dataset_details.get("datasetId") if dataset_details.get("datasetId") else ""

        # Format output
        final_results_dict = {}
        final_results_dict["duosId"] = duos_identifier
        final_results_dict["studyId"] = study_id
        final_results_dict["datasetId"] = dataset_id
        final_results_dict["dacId"] = dac_id
        final_results_dict["studyName"] = duos_dict.get("studyName")
        final_results_dict["studyType"] = duos_dict.get("studyType")
        final_results_dict["studyDescription"] = format_description(duos_dict.get("studyDescription"))
        final_results_dict["dataTypes"] = duos_dict.get("dataTypes")
        final_results_dict["phenotypeIndication"] = duos_dict.get("phenotypeIndication")
        final_results_dict["species"] = duos_dict.get("species")
        final_results_dict["piName"] = duos_dict.get("piName")
        final_results_dict["dataCustodianEmail"] = duos_dict.get("dataCustodianEmail")
        final_results_dict["publicVisibility"] = duos_dict.get("publicVisibility")
        final_results_dict["nihAnvilUse"] = "I am NHGRI funded and I have a dbGaP PHS ID already" if duos_dict.get("nihAnvilUse") and 'already' in duos_dict.get("nihAnvilUse").lower() else "I am NHGRI funded and I do not have a dbGaP PHS ID"
        final_results_dict["submittingToAnvil"] = duos_dict.get("submittingToAnvil")
        final_results_dict["dbGaPPhsID"] = duos_dict.get("dbGaPPhsID")
        final_results_dict["dbGaPStudyRegistrationName"] = duos_dict.get("dbGaPStudyRegistrationName")
        final_results_dict["embargoReleaseDate"] = duos_dict.get("embargoReleaseDate")
        final_results_dict["sequencingCenter"] = duos_dict.get("sequencingCenter")
        final_results_dict["piEmail"] = duos_dict.get("piEmail")
        final_results_dict["piInstitution"] = duos_dict.get("piInstitution")
        final_results_dict["nihGrantContractNumber"] = duos_dict.get("nihGrantContractNumber")
        final_results_dict["nihICsSupportingStudy"] = duos_dict.get("nihICsSupportingStudy")
        final_results_dict["nihProgramOfficerName"] = duos_dict.get("nihProgramOfficerName")
        final_results_dict["nihInstitutionCenterSubmission"] = duos_dict.get("nihInstitutionCenterSubmission")
        final_results_dict["nihInstitutionalCertificationFileName"] = duos_dict.get("nihInstitutionalCertificationFileName")
        final_results_dict["nihGenomicProgramAdministratorName"] = duos_dict.get("nihGenomicProgramAdministratorName")
        final_results_dict["multiCenterStudy"] = duos_dict.get("multiCenterStudy")
        final_results_dict["collaboratingSites"] = duos_dict.get("collaboratingSites")
        final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSR"] = duos_dict.get("controlledAccessRequiredForGenomicSummaryResultsGSR")
        final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSRRequiredExplanation"] = duos_dict.get("controlledAccessRequiredForGenomicSummaryResultsGSRRequiredExplanation")
        final_results_dict["alternativeDataSharingPlan"] = duos_dict.get("alternativeDataSharingPlan")
        final_results_dict["alternativeDataSharingPlanReasons"] = duos_dict.get("alternativeDataSharingPlanReasons")
        final_results_dict["alternativeDataSharingPlanExplanation"] = duos_dict.get("alternativeDataSharingPlanExplanation")
        final_results_dict["alternativeDataSharingPlanFileName"] = duos_dict.get("alternativeDataSharingPlanFileName")
        final_results_dict["alternativeDataSharingPlanDataSubmitted"] = duos_dict.get("alternativeDataSharingPlanDataSubmitted")
        final_results_dict["alternativeDataSharingPlanDataReleased"] = duos_dict.get("alternativeDataSharingPlanDataReleased")
        final_results_dict["alternativeDataSharingPlanTargetDeliveryDate"] = duos_dict.get("alternativeDataSharingPlanTargetDeliveryDate")
        final_results_dict["alternativeDataSharingPlanTargetPublicReleaseDate"] = duos_dict.get("alternativeDataSharingPlanTargetPublicReleaseDate")
        final_results_dict["alternativeDataSharingPlanAccessManagement"] = duos_dict.get("alternativeDataSharingPlanAccessManagement")
        final_results_dict["consentGroups.consentGroupName"] = duos_dict["consentGroups"][0].get("consentGroupName")
        final_results_dict["consentGroups.accessManagement"] = duos_dict["consentGroups"][0].get("accessManagement")
        final_results_dict["consentGroups.numberOfParticipants"] = duos_dict["consentGroups"][0].get("numberOfParticipants")
        final_results_dict["consentGroups.generalResearchUse"] = coalesce(duos_dict["consentGroups"][0].get("generalResearchUse"), duos_data_use_dict.get("generalUse"), False)
        final_results_dict["consentGroups.hmb"] = coalesce(duos_dict["consentGroups"][0].get("hmb"), duos_data_use_dict.get("hmbResearch"), False)
        final_results_dict["consentGroups.diseaseSpecificUse"] = coalesce(duos_dict["consentGroups"][0].get("diseaseSpecificUse"), duos_data_use_dict.get("diseaseRestrictions"), [])
        final_results_dict["consentGroups.gs"] = coalesce(duos_dict["consentGroups"][0].get("gs"), duos_data_use_dict.get("geographicalRestrictions"))
        final_results_dict["consentGroups.poa"] = coalesce(duos_dict["consentGroups"][0].get("poa"), duos_data_use_dict.get("populationOriginsAncestry"), False)
        final_results_dict["consentGroups.nmds"] = coalesce(duos_dict["consentGroups"][0].get("nmds"), False)
        final_results_dict["consentGroups.gso"] = coalesce(duos_dict["consentGroups"][0].get("gso"), duos_data_use_dict.get("geneticStudiesOnly"), False)
        final_results_dict["consentGroups.pub"] = coalesce(duos_dict["consentGroups"][0].get("pub"), duos_data_use_dict.get("publicationResults"), False)
        final_results_dict["consentGroups.col"] = coalesce(duos_dict["consentGroups"][0].get("col"), duos_data_use_dict.get("collaboratorRequired"), False)
        final_results_dict["consentGroups.irb"] = coalesce(duos_dict["consentGroups"][0].get("irb"), duos_data_use_dict.get("ethicsApprovalRequired"), False)
        final_results_dict["consentGroups.npu"] = coalesce(duos_dict["consentGroups"][0].get("npu"), False)
        final_results_dict["consentGroups.otherPrimary"] = coalesce(duos_dict["consentGroups"][0].get("otherPrimary"), duos_data_use_dict.get("other"))
        final_results_dict["consentGroups.otherSecondary"] = duos_dict["consentGroups"][0].get("otherSecondary")
        final_results_dict["consentGroups.mor"] = duos_dict["consentGroups"][0].get("mor")
        final_results_dict["consentGroups.morDate"] = duos_dict["consentGroups"][0].get("morDate")
        final_results_dict["consentGroups.dataLocation"] = duos_dict["consentGroups"][0].get("dataLocation")
        final_results_dict["consentGroups.url"] = duos_dict["consentGroups"][0].get("url")
        final_results_dict["consentGroups.fileTypes"] = str(duos_dict["consentGroups"][0].get("fileTypes"))
        dataset_details_records.append(final_results_dict)

    # Write out output
    results_df = pd.DataFrame(dataset_details_records) 
    results_df_sorted = results_df.sort_values(by=["studyId", "datasetId"], ascending=[True, True], ignore_index=True)
    current_datetime_string = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    output_file = f"duos_metadata_{current_datetime_string}.tsv"
    results_df_sorted.to_csv(output_file, index=False, sep="\t")
    !gsutil cp $output_file $ws_bucket/dataset_metadata/output/duos_export/ 2> stdout

    # Display results
    print(f"End time: {datetime.datetime.now()}")
    print("\nDUOS Datasets: ")
    display(results_df_sorted)

#############################################
## Input Parameters
#############################################

# User token to use in pulling DUOS datasets (use "gcloud auth print-access-token | pbcopy" to get this)
# Leave empty to use SA token, which can pull all public datasets
duos_token = ""

# Environment
duos_env = "prod"

#############################################
## Execution
#############################################

export_duos_datasets(duos_token, duos_env)
    

## Update Study and/or Dataset Metadata

### Update Study and/or Dataset Metadata via Study Registration

This can be used for the bulk of study and/or dataset metadata updates required in DUOS, with the following exceptions: 
1) For larger studies (i.e. those containing a large number of datasets, the study registration update API may time out. For dataset-level updates to those studies, it may make more sense to use one of the alternate update paths below. 
2) The following fields are NOT modifiable using the study registration update API: Data Submitter Name/Email, Access Management, Consent Group Name, Data Access Committee ID, all Data Use fields. Updates to these fields, if possible in DUOS, will need to be handled using one of the alternate update paths below. 

In [ ]:
#############################################
## Functions
#############################################

def update_duos_registrations(duos_token, duos_env, duos_updates):
    # Set environment URL
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"

    # Loop through duos datasets and update
    results = []
    for duos_dataset_id, duos_update in duos_updates.items():
        
        # Pull study registration from DUOS
        dataset_registration = requests.get(
            url=f"{url}/api/dataset/registration/{duos_dataset_id}",
            headers={"Authorization": f"Bearer {duos_token}"}
        ).json()
        study_id = dataset_registration["studyId"]
        study_registration = requests.get(
            url=f"{url}/api/dataset/study/registration/{study_id}",
            headers={"Authorization": f"Bearer {duos_token}"}
        ).json()
#         print("Original:")
#         print(study_registration)

        # Make updates to study registration
        print(f"Updating study registration for dataset id {duos_dataset_id} (study id = {study_id})...")
        study_property_updates = duos_update.get("study_updates")
        dataset_property_updates = duos_update.get("dataset_updates") 
        
        # Update study properties
        if study_property_updates:
            for key, val in study_property_updates.items():
                study_registration[key] = val
                
        # Update dataset properties
        if dataset_property_updates:
            consent_group_number = None
            consent_groups = study_registration.get("consentGroups")
            if consent_groups:
                for idx, cg in enumerate(consent_groups):
                    if cg["datasetIdentifier"] == duos_dataset_id:
                        consent_group_number = idx
                        break
                if consent_group_number != None:
                    for key, val in dataset_property_updates.items():
                        study_registration["consentGroups"][consent_group_number][key] = val
            
        # Push updated study registration to DUOS
#         print("Updated:")
#         print(study_registration)
        update_study_response = requests.put(
            url=f"{url}/api/dataset/study/{study_id}",
            headers={"Authorization": f"Bearer {duos_token}"},
            files = {
                "dataset": json.dumps(study_registration),
                "alternativeDataSharingPlan": "",
                "consentGroups[0].nihInstitutionalCertificationFile": ""  
            }
        ).json()   
        if update_study_response.get("studyId"):
            study_id = update_study_response["studyId"]
            msg = f"Study registration succeeded! Study Id: {study_id}"
            print(msg)
            results.append([duos_dataset_id, study_id, "Success", msg])
        else:
            err_msg = update_study_response["message"]
            msg = f"Study registration failed: {err_msg}"
            print(msg)
            results.append([duos_dataset_id, study_id, "Failure", msg])
        
    
    # Display final results
    results_df = pd.DataFrame(results, columns=["duos_dataset_id", "study_id", "status", "message"])
    print("\nFinal Results: ")
    display(results_df)
    

#############################################
## Input Parameters
#############################################

# User token (use "gcloud auth print-access-token | pbcopy" to get this)
duos_token = ""

# Environment
duos_env = "prod"

# Updates to make, of form {"<duos_dataset_id>": {"study_updates": {"<studyProp>": "<new_value>"}, "dataset_updates": {"<datasetProp>": "<new_value>"}}}
duos_updates = {
#     "duos_dataset_id": {"study_updates": {}, "dataset_updates": {}}
    "DUOS-000299": {"study_updates": {"publicVisibility": False}}
}


#############################################
## Execution
#############################################

update_duos_registrations(duos_token, duos_env, duos_updates)


### Update Study and/or Dataset Metadata via Dataset Object
This should be used for cases that cannot be handled via study registration update. This includes: 
1. Dataset-level updates to datasets that are part of large studies (and time out using the study registration update path). 
2. Updates to fields NOT modifiable using the study registration update API, including Data Submitter Name/Email, Access Management, Consent Group Name, and Data Access Committee ID. 

*Note: The only attributes supported today are Consent Group Name ("consentGroupName"), Access Management ("accessManagement"), and Participant Count ("numberOfParticipants"). To support additional properties, extend the property_map dict in the update_duos_dataset_object function.*

*Note 2: This does not currently work for datasets not assigned to DACs, as a DAC is required in the update request, but DAC changes are not supported (so can't just pass a null/default value).*

In [ ]:
#############################################
## Functions
#############################################

def update_duos_dataset_object(duos_token, duos_env, duos_updates):
    # Map of properties to update with this API
    property_map = {
        "accessManagement": "Access Management",
        "numberOfParticipants": "# of participants"
    }
    
    # Set environment URL
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"

    # Loop through duos datasets and update
    results = []
    for duos_dataset_id, duos_update in duos_updates.items():
        
        # Pull dataset object from DUOS
        print(f"Processing DUOS ID {duos_dataset_id}...")
        dataset_registration = requests.get(
            url=f"{url}/api/dataset/registration/{duos_dataset_id}",
            headers={"Authorization": f"Bearer {duos_token}"}
        ).json()
        dataset_id = dataset_registration["consentGroups"][0]["datasetId"]
        dataset_object = requests.get(
            url=f"{url}/api/dataset/v2/{dataset_id}",
            headers={"Authorization": f"Bearer {duos_token}"}
        ).json()

        # Build new dataset object
        dataset_name = dataset_object["name"]
        dataset_dac = dataset_object.get("dacId")
        dataset_properties = dataset_object["properties"]
        original_dataset_object = {
            "name": dataset_name,
            "dacId": dataset_dac,
            "properties": dataset_properties
        }
        print("Original:")
        print(original_dataset_object)
        for key, val in duos_update.items():
            mapped_key = property_map.get(key)
            if mapped_key:
                prop_found = False
                for idx, dataset_prop in enumerate(dataset_properties):
                    if mapped_key == dataset_prop["propertyName"]:
                        prop_found = True
                        new_prop_object = dataset_properties.pop(idx)
                        new_prop_object["propertyValue"] = val
                        break
                if not prop_found:
                    new_prop_object = {
                        "propertyName": mapped_key,
                        "propertyValue": val,
                    }
                dataset_properties.append(new_prop_object)
            elif key == "consentGroupName":
                dataset_name = val
#             # NOT YET SUPPORTED:
#             elif key == "dacId":
#                 new_dataset_dac = val
        new_dataset_object = {
            "name": dataset_name,
            "dacId": dataset_dac,
            "properties": dataset_properties
        }
            
        # Push updated dataset object to DUOS
        print("Updated:")
        print(new_dataset_object)
        print(f"Updating dataset object for DUOS ID {duos_dataset_id} (dataset id = {dataset_id})...")
        update_dataset_object = requests.put(
            url=f"{url}/api/dataset/v3/{dataset_id}",
            headers={"Authorization": f"Bearer {duos_token}"},
            files = {
                "dataset": json.dumps(new_dataset_object),
                "alternativeDataSharingPlan": "",
                "consentGroups[0].nihInstitutionalCertificationFile": ""  
            }
        ).json()   
        if update_dataset_object.get("datasetId"):
            dataset_id = update_dataset_object["datasetId"]
            msg = f"Dataset object update succeeded! Dataset Id: {dataset_id}"
            print(msg)
            results.append([duos_dataset_id, dataset_id, "Success", msg])
        else:
            err_msg = update_dataset_object["message"]
            msg = f"Dataset object update failed: {err_msg}"
            print(msg)
            results.append([duos_dataset_id, dataset_id, "Failure", msg])
            
        # Brief delay before starting next dataset
        sleep(10)
        
    
    # Display final results
    results_df = pd.DataFrame(results, columns=["duos_dataset_id", "dataset_id", "status", "message"])
    print("\nFinal Results: ")
    display(results_df)
    

#############################################
## Input Parameters
#############################################

# User token (use "gcloud auth print-access-token | pbcopy" to get this)
duos_token = ""

# Environment
duos_env = "dev"

# Updates to make, of form {"<duos_dataset_id>": {"<datasetProp>": "<new_value>"}}
duos_updates = {
#     "duos_dataset_id": {"study_updates": {}, "dataset_updates": {}}
    "DUOS-000626": {"accessManagement": "controlled"}
}


#############################################
## Execution
#############################################

update_duos_dataset_object(duos_token, duos_env, duos_updates)


## Copy Consent Group to New Study

In [ ]:
#############################################
## Functions
#############################################

def copy_consent_groups(duos_token, duos_env, copy_list):
    # Set environment URL
    if duos_env == "prod":
        url = "https://consent.dsde-prod.broadinstitute.org"
    else:
        url = "https://consent.dsde-dev.broadinstitute.org"

    # Loop through duos datasets and update
    results = []
    for copy_item in copy_list:
        error = False
        duos_dataset_id = copy_item.get("duos_dataset_id")
        target_study_id = copy_item.get("target_study_id")
        mode = copy_item.get("mode")
        new_consent_group_name = copy_item.get("new_consent_group_name")
        print(f"Copying {duos_dataset_id} consent group into study id {target_study_id}...")
        if duos_dataset_id and target_study_id and mode in ["MOVE", "COPY"]:
        
            # Pull consent group to copy
            print("Pulling consent group to copy...")
            try:
                dataset_registration = requests.get(
                    url=f"{url}/api/dataset/registration/{duos_dataset_id}",
                    headers={"Authorization": f"Bearer {duos_token}"}
                ).json()
                consent_groups = dataset_registration.get("consentGroups")
                if consent_groups:
                    new_consent_group = consent_groups[0]
                    orig_dataset_id = new_consent_group.pop("datasetId", None)
                    orig_duos_dataset_id = new_consent_group.pop("datasetIdentifier", None)
                    orig_dataset_name = new_consent_group.get("consentGroupName")
                else:
                    raise Exception("Error retrieving consent group for duos_dataset_id.") 
            except Exception as e:
                results.append([duos_dataset_id, target_study_id, "Failure", str(e)])
                continue
                
            # If MOVE mode, pull update the name of the original dataset
            if mode == "MOVE":
                print("Mode of 'MOVE', so renaming original consent group...")
                new_dataset_name = orig_dataset_name
                updated_dataset_name = orig_dataset_name + " TEMP"
                payload = {
                    "name": updated_dataset_name,
                    "properties": []
                }
                try:
                    dataset_patch_response = requests.patch(
                        url=f"{url}/api/dataset/{orig_dataset_id}",
                        headers={"Authorization": f"Bearer {duos_token}"},
                        json=payload
                    )
                    if dataset_patch_response.status_code not in (200, 304):
                        raise Exception("Error renaming original consent group.")
                except Exception as e:
                    results.append([duos_dataset_id, target_study_id, "Failure", str(e)])
                    continue
            else:
                if new_consent_group_name:
                    new_dataset_name = new_consent_group_name
                else:
                    results.append([duos_dataset_id, target_study_id, "Failure", "Missing or invalid input"])
                    continue
            new_consent_group["consentGroupName"] = new_dataset_name

            # Pull full study registration for target study and add new consent group
            print("Adding consent group to target study registration...")
            try:
                study_registration = requests.get(
                    url=f"{url}/api/dataset/study/registration/{target_study_id}",
                    headers={"Authorization": f"Bearer {duos_token}"}
                ).json()
                consent_groups = study_registration.get("consentGroups")
                if consent_groups:
                    updated_consent_groups = consent_groups.copy()
                    updated_consent_groups.append(new_consent_group)
                    study_registration["consentGroups"] = updated_consent_groups.copy()
                else:
                    raise Exception("Error retrieving the study registration for target_study_id.")
            except Exception as e:
                results.append([duos_dataset_id, target_study_id, "Failure", str(e)])
                continue

            # Push updated study registration to DUOS
            print("Pushing updated study registration to DUOS...")
            update_study_response = requests.put(
                url=f"{url}/api/dataset/study/{target_study_id}",
                headers={"Authorization": f"Bearer {duos_token}"},
                files = {
                    "dataset": json.dumps(study_registration),
                    "alternativeDataSharingPlan": "",
                    "consentGroups[0].nihInstitutionalCertificationFile": ""  
                }
            ).json()   
            if update_study_response.get("studyId"):
                study_id = update_study_response["studyId"]
                msg = f"Study registration succeeded! Study Id: {target_study_id}"
                print(msg)
                results.append([duos_dataset_id, target_study_id, "Success", msg])
            else:
                err_msg = update_study_response["message"]
                msg = f"Study registration failed: {err_msg}"
                print(msg)
                results.append([duos_dataset_id, target_study_id, "Failure", msg])
        
        else:
            results.append([duos_dataset_id, target_study_id, "Failure", "Missing or invalid input"])
            continue
        
    # Display final results
    results_df = pd.DataFrame(results, columns=["duos_dataset_id", "target_study_id", "status", "message"])
    print("\nFinal Results: ")
    display(results_df)
    

#############################################
## Input Parameters
#############################################

# User token (use "gcloud auth print-access-token | pbcopy" to get this)
duos_token = ""

# Environment
duos_env = "prod"

# List of copies to make
copy_list = [
#     {"<duos_dataset_id": "<DUOS-123456>", "target_study_id": "<123>", "mode": "<COPY/MOVE>", "new_consent_group_name": "<Name>"},
    {"duos_dataset_id": "DUOS-000666", "target_study_id": "73", "mode": "MOVE", "new_consent_group_name": ""}
]


#############################################
## Execution
#############################################

copy_consent_groups(duos_token, duos_env, copy_list)



# dbGaP Utilities

## Convert dbGaP Metadata to DUOS Format

In [ ]:
#############################################
## Functions
#############################################

def parse_consent(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.replace("_", "-")
    pattern = r"^(.+?)-(?:IRB|NPU|RD|MDS|PUB|COL|GSO)(?:-(?:IRB|NPU|RD|MDS|PUB|COL|GSO))*$"
    match = re.match(pattern, text)
    if match: 
        return match.group(1)
    else:
        return text

def format_description(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'</?p>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'\t', ' ', text)
    text = '\n'.join([line.strip() for line in text.split('\n')])
    text = text.strip()
    text = re.sub(r'\n', '\\n', text)
    text = re.sub("study.cgi\?study_id=|.\/study.cgi\?study_id=", "https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=", text)
    return text

def format_phs_id(input_str):
    try:
        num = re.search("phs0*([0-9]+)", input_str, re.IGNORECASE).group(1)
    except:
        num = ""
    if num:
        output_str = "phs" + str(num).zfill(6)
    else:
        output_str = ""
    return output_str   

def fetch_and_format_dbgap_metadata(phs_id_list, ds_consent_map, ds_consent_map_overrides, desc_tag, duos_token, duos_env, write_to_bucket, display_output):
    
    # Initialize variables
    results_records = []
    
    # Build DUOS study lookup
    study_lookup = {}
    study_results = []
    if duos_token and duos_env in ["prod", "dev"]:
        print("Attempting to build DUOS study lookups...")
        try:
            if duos_env == "prod":
                url = "https://consent.dsde-prod.broadinstitute.org"
            else:
                url = "https://consent.dsde-dev.broadinstitute.org"
            study_id = 1
            missing_count = 0
            while missing_count < 20:
                try:
                    study_registration = requests.get(
                        url=f"{url}/api/dataset/study/registration/{study_id}",
                        headers={"Authorization": f"Bearer {duos_token}"}
                    ).json()
                    study_name = study_registration["studyName"]
                    dbgap_phs = study_registration["dbGaPPhsID"]
                    study_results.append([study_id, dbgap_phs])
                    missing_count = 0
                except Exception as e:
                    missing_count += 1
                study_id += 1
            for study_result in study_results:
                study_id = study_result[0]
                study_phs_id = study_result[1]
                if study_phs_id and study_id:
                    existing_list = study_lookup.get(study_phs_id)
                    if existing_list:
                        existing_list.append(study_id)
                        study_lookup[study_phs_id] = existing_list
                    else:
                        study_lookup[study_phs_id] = [study_id]
        except:
            print("Error building DUOS study lookup. Skipping.")     
    else:
        print("Skipping DUOS study lookups...")
        
    # Loop through and process PHS IDs
    for phs_id in phs_id_list:
        
        # Pull study information from dbGaP
        dbgap_dict = {}
        dbgap_phs_id = format_phs_id(phs_id)
        matching_duos_study_ids = study_lookup.get(dbgap_phs_id)
        print(f"Processing input PHS: {phs_id} ({dbgap_phs_id})")
        # Start with dbGaP XML
        if dbgap_phs_id:
            phs_short = dbgap_phs_id.replace("phs", "")
            dbgap_url = "https://dbgap.ncbi.nlm.nih.gov/ss/dbgapssws.cgi?request=Study&phs=" + phs_short
            attempt_counter = 0
            while attempt_counter <= 2:
                try:
                    response = requests.get(url=dbgap_url)
                    xml_data = xmltodict.parse(response.text)
                    break
                except:
                    sleep(5)
                    attempt_counter += 1
            if xml_data["dbgapss"].get("Study"):
                study_data = {}
                if isinstance(xml_data["dbgapss"]["Study"], list):
                    for study in xml_data["dbgapss"]["Study"]:
                        study_status = study["Status"]["@name"]
                        if study_status == "released":
                            study_data = study
                            break
                else:
                    study_data = xml_data["dbgapss"]["Study"]
                if study_data:
                    study_uid = study_data.get("@uid")
                    dbgap_dict["anvilPartner"] = False
                    dbgap_dict["dbGaPPhsID"] = dbgap_phs_id
                    dbgap_dict["numberOfParticipants"] = study_data.get("@num_participants")
                    study_info = study_data.get("StudyInfo")
                    if study_info:
                        dbgap_dict["studyDescription"] = study_data["StudyInfo"].get("Description")
                        dbgap_dict["dbGaPStudyRegistrationName"] = study_data["StudyInfo"].get("StudyNameEntrez")
                    if study_data.get("Authority"):
                        if study_data["Authority"].get("Persons"):
                            if study_data["Authority"]["Persons"].get("Person"):
                                for ap_entry in study_data["Authority"]["Persons"]["Person"]:
                                    role_object = ap_entry.get("Role")
                                    if isinstance(role_object, dict):
                                        role = role_object.get("#text")
                                    else:
                                        role = role_object
                                    org = ap_entry.get("Organization")
                                    if role and role == "PI":
                                        first_name = ap_entry.get("@fname") if ap_entry.get("@fname") else ""
                                        last_name = ap_entry.get("@lname") if ap_entry.get("@lname") else ""
                                        email = ap_entry.get("@email") if ap_entry.get("@email") else ""
                                        institute = ap_entry.get("Organization") if ap_entry.get("Organization") else ""
                                        dbgap_dict["piName"] = last_name.upper() + ", " + first_name.upper()
                                        dbgap_dict["piEmail"] = email.upper()
                                        dbgap_dict["piInstitution"] = institute.upper()
                                    elif role and role == "PO" and org and org == "NIH":
                                        first_name = ap_entry.get("@fname") if ap_entry.get("@fname") else ""
                                        last_name = ap_entry.get("@lname") if ap_entry.get("@lname") else ""
                                        dbgap_dict["nihProgramOfficerName"] = last_name.upper() + ", " + first_name.upper()
                                    elif role and role == "GPA" and org and org == "NIH":
                                        first_name = ap_entry.get("@fname") if ap_entry.get("@fname") else ""
                                        last_name = ap_entry.get("@lname") if ap_entry.get("@lname") else ""
                                        dbgap_dict["nihGenomicProgramAdministratorName"] = last_name.upper() + ", " + first_name.upper()
                        if study_data["Authority"].get("ICs"):
                            if study_data["Authority"]["ICs"].get("IC"):
                                ic_list = []
                                if isinstance(study_data["Authority"]["ICs"]["IC"], list):
                                    for ic_entry in study_data["Authority"]["ICs"]["IC"]:
                                        name = ic_entry.get("@name")
                                        if name:
                                            ic_list.append(name)
                                else:
                                    name = study_data["Authority"]["ICs"]["IC"].get("@name")
                                    if name:
                                        ic_list.append(name)
                                dbgap_dict["nihICsSupportingStudy"] = ic_list

                    # Supplement with information from dbGaP FHIR
                    dbgap_fhir_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/ResearchStudy?_format=json&_id=" + dbgap_phs_id
                    attempt_counter = 0
                    while attempt_counter <= 2:
                        try:
                            response = requests.get(url=dbgap_fhir_url)
                            fhir_data = json.loads(response.text)
                            break
                        except:
                            sleep(5)
                            attempt_counter += 1
                    if fhir_data.get("entry"):
                        # studyType
                        if fhir_data["entry"][0]["resource"].get("category"):
                            for cat_entry in fhir_data["entry"][0]["resource"].get("category"):
                                if cat_entry.get("coding"):
                                    for coding_entry in cat_entry.get("coding"):
                                        if coding_entry.get("system") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/ResearchStudy-StudyDesign":
                                            value = coding_entry.get("display") if coding_entry.get("display") else coding_entry.get("code")
                                            if dbgap_dict.get("studyType") and value:
                                                dbgap_dict["studyType"] += f", {value}"
                                            elif value:
                                                dbgap_dict["studyType"] = value
                        # dataTypes
                        dt_list = []
                        if fhir_data["entry"][0]["resource"].get("extension"): 
                            for ext_entry in fhir_data["entry"][0]["resource"].get("extension"):
                                if ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-MolecularDataTypes":
                                    for inner_ext_entry in ext_entry.get("extension"):
                                        if inner_ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-MolecularDataTypes-MolecularDataType":
                                            for coding_entry in inner_ext_entry["valueCodeableConcept"].get("coding"):
                                                dt_list.append(coding_entry.get("code"))
                        dbgap_dict["dataTypes"] = dt_list
                        # phenotypeIndication
                        if fhir_data["entry"][0]["resource"].get("focus"):
                            for focus_entry in fhir_data["entry"][0]["resource"].get("focus"):
                                if focus_entry.get("coding"):
                                    for coding_entry in focus_entry.get("coding"):
                                        value = coding_entry.get("display") if coding_entry.get("display") else coding_entry.get("code")
                                        if dbgap_dict.get("phenotypeIndication") and value:
                                            dbgap_dict["phenotypeIndication"] += f", {value}"
                                        elif value:
                                            dbgap_dict["phenotypeIndication"] = value
                        # study description
                        if not dbgap_dict.get("studyDescription"):
                            dbgap_dict["studyDescription"] = fhir_data["entry"][0]["resource"].get("description")
                        # number of participants
                        if not dbgap_dict.get("numberOfParticipants"):
                            if fhir_data["entry"][0]["resource"].get("extension"):
                                for ext_entry in fhir_data["entry"][0]["resource"].get("extension"):
                                    if ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-Content":
                                        for inner_ext_entry in ext_entry.get("extension"):
                                            if inner_ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-Content-NumSubjects":
                                                dbgap_dict["numberOfParticipants"] = inner_ext_entry["valueCount"].get("code")

                    # Supplement with information from dbGaP study config page
                    if study_uid:
                        dbgap_study_url = "https://submit.ncbi.nlm.nih.gov/dbgap/api/v1/study_config/" + str(study_uid)
                        attempt_counter = 0
                        while attempt_counter <= 2:
                            try:
                                response = requests.get(url=dbgap_study_url)
                                study_api_data = json.loads(response.text)
                                break
                            except:
                                sleep(5)
                                attempt_counter += 1
                        if study_api_data.get("error") == None:
                            if not dbgap_dict.get("phenotypeIndication"):
                                dbgap_dict["phenotypeIndication"] = study_api_data["data"].get("primary_disease")
                            if not dbgap_dict.get("studyType"):
                                dbgap_dict["studyType"] = study_api_data["data"].get("study_design")
                            if not dbgap_dict.get("studyDescription"):
                                dbgap_dict["studyDescription"] = study_api_data["data"].get("description")

                    # Loop through consent codes and build final records
                    if study_data.get("Policy"):
                        dbgap_dict["embargoReleaseDate"] = study_data["Policy"].get("@pub-embargo")
                        if study_data["Policy"].get("TrustedPartners"):
                            if study_data["Policy"]["TrustedPartners"].get("TrustedPartner"):
                                trusted_partner_list = study_data["Policy"]["TrustedPartners"]["TrustedPartner"] if isinstance(study_data["Policy"]["TrustedPartners"]["TrustedPartner"], list) else [study_data["Policy"]["TrustedPartners"]["TrustedPartner"]]
                                for trusted_partner in trusted_partner_list:
                                    if trusted_partner["@trp_db_name"] == "AnVIL":
                                        dbgap_dict["anvilPartner"] = True
                                        break
                        if study_data["Policy"].get("ConsentGroup"):
                            consent_list = study_data["Policy"]["ConsentGroup"] if isinstance(study_data["Policy"]["ConsentGroup"], list) else [study_data["Policy"]["ConsentGroup"]]
                            for consent in consent_list:
                                final_results_dict = {}
                                anvil_partner = dbgap_dict.get("anvilPartner")
                                consent_code = consent["@name"]
                                parsed_consent_code = parse_consent(consent_code)
                                consent_desc = consent.get("@title")
                                dac = consent.get("@dac_name")
                                access_management = "controlled"
                                if dbgap_phs_id and consent_code:
                                    study_consent = dbgap_phs_id + ":" + consent_code
                                    purl_doid = ds_consent_map_overrides.get(study_consent)
                                    if not purl_doid:
                                        purl_doid = ds_consent_map.get(parsed_consent_code)
                                    if purl_doid:
                                        if not isinstance(purl_doid, list):
                                            purl_doid = [purl_doid]
                                    else:
                                        purl_doid = []
                                else:
                                    purl_doid = []
                                final_results_dict["snapshot_id"] = "NA"
                                final_results_dict["snapshot_phs_id"] = "NA"
                                final_results_dict["snapshot_duos_id"] = "NA"
                                final_results_dict["match_duos_id"] = "NA"
                                final_results_dict["target_dataset_id"] = "NA"
                                final_results_dict["target_study_id"] = matching_duos_study_ids if matching_duos_study_ids else "NA"
                                study_name = dbgap_dict.get("dbGaPStudyRegistrationName")
                                if dbgap_phs_id and study_name and f" ({dbgap_phs_id})" not in study_name:
                                    final_results_dict["studyName"] = study_name + f" ({dbgap_phs_id})"
                                else:
                                    final_results_dict["studyName"] = study_name
                                final_results_dict["studyType"] = dbgap_dict.get("studyType")
                                final_results_dict["studyDescription"] = format_description(dbgap_dict.get("studyDescription"))
                                if desc_tag:
                                    final_results_dict["studyDescription"] = final_results_dict["studyDescription"] + "\\n" + desc_tag
                                final_results_dict["dataTypes"] = dbgap_dict.get("dataTypes") if dbgap_dict.get("dataTypes") else []
                                final_results_dict["phenotypeIndication"] = dbgap_dict.get("phenotypeIndication")
                                final_results_dict["species"] = "Human"
                                final_results_dict["piName"] = dbgap_dict.get("piName")
                                final_results_dict["dataCustodianEmail"] = ["help@lists.anvilproject.org"] if anvil_partner else []
                                final_results_dict["publicVisibility"] = True
                                if dac and dac == "NHGRI":
                                    nih_use = "I am NHGRI funded and I have a dbGaP PHS ID already"
                                elif anvil_partner:
                                    nih_use = "I am not NHGRI funded but I am seeking to submit data to AnVIL"
                                else:
                                    nih_use = "I am not NHGRI funded and do not plan to store data in AnVIL"
                                final_results_dict["nihAnvilUse"] = nih_use
                                final_results_dict["submittingToAnvil"] = anvil_partner 
                                final_results_dict["dbGaPPhsID"] = dbgap_phs_id
                                if dbgap_phs_id and study_name and f" ({dbgap_phs_id})" in study_name:
                                    final_results_dict["dbGaPStudyRegistrationName"] = study_name.replace(f" ({dbgap_phs_id})", "")
                                else:
                                    final_results_dict["dbGaPStudyRegistrationName"] = study_name
                                final_results_dict["embargoReleaseDate"] = dbgap_dict.get("embargoReleaseDate")
                                final_results_dict["sequencingCenter"] = None
                                final_results_dict["piEmail"] = dbgap_dict.get("piEmail")
                                final_results_dict["piInstitution"] = dbgap_dict.get("piInstitution")
                                final_results_dict["nihGrantContractNumber"] = None
                                final_results_dict["nihICsSupportingStudy"] = dbgap_dict.get("nihICsSupportingStudy")
                                final_results_dict["nihProgramOfficerName"] = dbgap_dict.get("nihProgramOfficerName")
                                final_results_dict["nihInstitutionCenterSubmission"] = dac
                                final_results_dict["nihInstitutionalCertificationFileName"] = None
                                final_results_dict["nihGenomicProgramAdministratorName"] = dbgap_dict.get("nihGenomicProgramAdministratorName")
                                final_results_dict["multiCenterStudy"] = None
                                final_results_dict["collaboratingSites"] = []
                                final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSR"] = None
                                final_results_dict["controlledAccessRequiredForGenomicSummaryResultsGSRRequiredExplanation"] = None
                                final_results_dict["alternativeDataSharingPlan"] = False
                                final_results_dict["alternativeDataSharingPlanReasons"] = []
                                final_results_dict["alternativeDataSharingPlanExplanation"] = None
                                final_results_dict["alternativeDataSharingPlanFileName"] = None
                                final_results_dict["alternativeDataSharingPlanDataSubmitted"] = None
                                final_results_dict["alternativeDataSharingPlanDataReleased"] = None
                                final_results_dict["alternativeDataSharingPlanTargetDeliveryDate"] = None
                                final_results_dict["alternativeDataSharingPlanTargetPublicReleaseDate"] = None
                                final_results_dict["alternativeDataSharingPlanAccessManagement"] = None
                                final_results_dict["consentGroups.consentGroupName"] = f"{dbgap_phs_id} ({consent_code})" 
                                if access_management == "controlled" and dac and dac != 'NHGRI':
                                    final_results_dict["consentGroups.accessManagement"] = "external"
                                else:
                                    final_results_dict["consentGroups.accessManagement"] = access_management
                                final_results_dict["dacId"] = dac
                                final_results_dict["consentGroups.numberOfParticipants"] = dbgap_dict.get("numberOfParticipants") if dbgap_dict.get("numberOfParticipants") else "0"
                                final_results_dict["consentCode"] = consent_code
                                final_results_dict["consentDesc"] = consent_desc
                                final_results_dict["consentGroups.generalResearchUse"] = True if access_management == "controlled" and "GRU" in consent_code else False
                                final_results_dict["consentGroups.hmb"] = True if access_management == "controlled" and "HMB" in consent_code else False
                                if purl_doid:
                                    final_results_dict["consentGroups.diseaseSpecificUse"] = purl_doid
                                else:
                                    final_results_dict["consentGroups.diseaseSpecificUse"] = []
                                final_results_dict["consentGroups.gs"] = consent_code if access_management == "controlled" and "GS-" in consent_code else None
                                final_results_dict["consentGroups.poa"] = True if access_management == "controlled" and "POA" in consent_code else False
                                final_results_dict["consentGroups.nmds"] = True if access_management == "controlled" and "NMDS" in consent_code else False
                                final_results_dict["consentGroups.gso"] = True if access_management == "controlled" and "GSO" in consent_code else False
                                final_results_dict["consentGroups.pub"] = True if access_management == "controlled" and "PUB" in consent_code else False 
                                final_results_dict["consentGroups.col"] = True if access_management == "controlled" and "COL-" in consent_code else False
                                final_results_dict["consentGroups.irb"] = True if access_management == "controlled" and "IRB" in consent_code else False
                                final_results_dict["consentGroups.npu"] = True if access_management == "controlled" and "NPU" in consent_code else False
                                final_results_dict["consentGroups.otherPrimary"] = consent_code if (consent_code and access_management == "controlled" and not (final_results_dict["consentGroups.generalResearchUse"] or final_results_dict["consentGroups.hmb"] or final_results_dict["consentGroups.diseaseSpecificUse"] or final_results_dict["consentGroups.gs"] or final_results_dict["consentGroups.poa"] or final_results_dict["consentGroups.nmds"] or final_results_dict["consentGroups.gso"] or final_results_dict["consentGroups.pub"] or final_results_dict["consentGroups.col"] or final_results_dict["consentGroups.irb"] or final_results_dict["consentGroups.npu"])) else None
                                final_results_dict["consentGroups.otherSecondary"] = None
                                final_results_dict["consentGroups.mor"] = None
                                final_results_dict["consentGroups.morDate"] = None
                                final_results_dict["consentGroups.dataLocation"] = ""
                                final_results_dict["consentGroups.url"] = ""
                                final_results_dict["consentGroups.fileTypes.fileType"] = None
                                final_results_dict["consentGroups.fileTypes.functionalEquivalence"] = None
                                final_results_dict["consortium"] = None
                                results_records.append(final_results_dict)
        
    # Write out and display results
    if results_records:
        output = pd.DataFrame(results_records)
        output_sorted = output.sort_values(by=["studyName", "consentGroups.consentGroupName"], ascending=[True, True], ignore_index=True)
        if write_to_bucket:
            current_datetime_string = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
            output_file = f"dbgap_metadata_{current_datetime_string}.tsv"
            output_sorted.to_csv(output_file, index=False, sep="\t")
            !gsutil cp $output_file $ws_bucket/dataset_metadata/output/ 2> stdout
            print(f"File written to {ws_bucket}/dataset_metadata/output/{output_file}!")
        if display_output:
            print("----------------------------------------------------------------------------------------------------")
            print("----------------------------------------------------------------------------------------------------")
            print("Metadata Output:")
            display(output_sorted.style.hide(axis="index"))
    

#############################################
## Input Parameters
#############################################

# PHS IDs to process:
phs_id_list = [
    'phs000001',
    'phs000004',
    'phs000005',
    'phs000007',
    'phs000016',
    'phs000017',
    'phs000018',
    'phs000019',
    'phs000020',
    'phs000021',
    'phs000048',
    'phs000086',
    'phs000088',
    'phs000089',
    'phs000090',
    'phs000091',
    'phs000092',
    'phs000093',
    'phs000094',
    'phs000095',
    'phs000096',
    'phs000100',
    'phs000101',
    'phs000102',
    'phs000103',
    'phs000122',
    'phs000123',
    'phs000124',
    'phs000125',
    'phs000126',
    'phs000127',
    'phs000128',
    'phs000130',
    'phs000138',
    'phs000139',
    'phs000140',
    'phs000142',
    'phs000143',
    'phs000144',
    'phs000145',
    'phs000147',
    'phs000153',
    'phs000159',
    'phs000160',
    'phs000166',
    'phs000167',
    'phs000168',
    'phs000169',
    'phs000170',
    'phs000171',
    'phs000178',
    'phs000179',
    'phs000180',
    'phs000181',
    'phs000182',
    'phs000183',
    'phs000184',
    'phs000185',
    'phs000187',
    'phs000188',
    'phs000196',
    'phs000197',
    'phs000199',
    'phs000200',
    'phs000201',
    'phs000202',
    'phs000203',
    'phs000204',
    'phs000205',
    'phs000206',
    'phs000207',
    'phs000209',
    'phs000210',
    'phs000211',
    'phs000215',
    'phs000216',
    'phs000217',
    'phs000218',
    'phs000219',
    'phs000220',
    'phs000221',
    'phs000222',
    'phs000223',
    'phs000224',
    'phs000225',
    'phs000226',
    'phs000227',
    'phs000228',
    'phs000233',
    'phs000234',
    'phs000235',
    'phs000236',
    'phs000237',
    'phs000238',
    'phs000239',
    'phs000240',
    'phs000244',
    'phs000245',
    'phs000246',
    'phs000247',
    'phs000248',
    'phs000249',
    'phs000250',
    'phs000251',
    'phs000252',
    'phs000253',
    'phs000254',
    'phs000255',
    'phs000256',
    'phs000257',
    'phs000258',
    'phs000259',
    'phs000260',
    'phs000261',
    'phs000262',
    'phs000263',
    'phs000264',
    'phs000265',
    'phs000266',
    'phs000267',
    'phs000268',
    'phs000269',
    'phs000272',
    'phs000273',
    'phs000274',
    'phs000275',
    'phs000276',
    'phs000277',
    'phs000279',
    'phs000280',
    'phs000281',
    'phs000282',
    'phs000283',
    'phs000284',
    'phs000285',
    'phs000286',
    'phs000287',
    'phs000288',
    'phs000289',
    'phs000290',
    'phs000291',
    'phs000292',
    'phs000293',
    'phs000294',
    'phs000295',
    'phs000296',
    'phs000297',
    'phs000298',
    'phs000299',
    'phs000300',
    'phs000301',
    'phs000302',
    'phs000303',
    'phs000304',
    'phs000305',
    'phs000306',
    'phs000307',
    'phs000308',
    'phs000309',
    'phs000310',
    'phs000311',
    'phs000313',
    'phs000314',
    'phs000315',
    'phs000327',
    'phs000328',
    'phs000329',
    'phs000330',
    'phs000331',
    'phs000332',
    'phs000333',
    'phs000334',
    'phs000335',
    'phs000336',
    'phs000337',
    'phs000338',
    'phs000339',
    'phs000340',
    'phs000341',
    'phs000342',
    'phs000343',
    'phs000344',
    'phs000345',
    'phs000346',
    'phs000347',
    'phs000348',
    'phs000349',
    'phs000350',
    'phs000351',
    'phs000352',
    'phs000353',
    'phs000354',
    'phs000355',
    'phs000356',
    'phs000357',
    'phs000358',
    'phs000359',
    'phs000360',
    'phs000361',
    'phs000362',
    'phs000363',
    'phs000364',
    'phs000365',
    'phs000366',
    'phs000367',
    'phs000368',
    'phs000369',
    'phs000370',
    'phs000371',
    'phs000372',
    'phs000373',
    'phs000374',
    'phs000375',
    'phs000376',
    'phs000377',
    'phs000378',
    'phs000379',
    'phs000380',
    'phs000381',
    'phs000382',
    'phs000383',
    'phs000384',
    'phs000385',
    'phs000386',
    'phs000387',
    'phs000388',
    'phs000389',
    'phs000390',
    'phs000391',
    'phs000392',
    'phs000393',
    'phs000394',
    'phs000395',
    'phs000396',
    'phs000397',
    'phs000398',
    'phs000399',
    'phs000400',
    'phs000401',
    'phs000402',
    'phs000403',
    'phs000404',
    'phs000405',
    'phs000406',
    'phs000407',
    'phs000408',
    'phs000409',
    'phs000410',
    'phs000413',
    'phs000414',
    'phs000415',
    'phs000416',
    'phs000417',
    'phs000418',
    'phs000419',
    'phs000420',
    'phs000421',
    'phs000422',
    'phs000423',
    'phs000424',
    'phs000425',
    'phs000426',
    'phs000427',
    'phs000428',
    'phs000429',
    'phs000430',
    'phs000431',
    'phs000433',
    'phs000434',
    'phs000435',
    'phs000436',
    'phs000437',
    'phs000438',
    'phs000439',
    'phs000440',
    'phs000441',
    'phs000442',
    'phs000443',
    'phs000444',
    'phs000445',
    'phs000447',
    'phs000448',
    'phs000449',
    'phs000450',
    'phs000451',
    'phs000452',
    'phs000454',
    'phs000455',
    'phs000456',
    'phs000457',
    'phs000458',
    'phs000459',
    'phs000460',
    'phs000461',
    'phs000462',
    'phs000463',
    'phs000464',
    'phs000465',
    'phs000466',
    'phs000467',
    'phs000468',
    'phs000469',
    'phs000470',
    'phs000471',
    'phs000472',
    'phs000473',
    'phs000474',
    'phs000475',
    'phs000476',
    'phs000477',
    'phs000478',
    'phs000479',
    'phs000480',
    'phs000481',
    'phs000482',
    'phs000483',
    'phs000484',
    'phs000485',
    'phs000486',
    'phs000487',
    'phs000488',
    'phs000489',
    'phs000490',
    'phs000491',
    'phs000492',
    'phs000493',
    'phs000494',
    'phs000495',
    'phs000496',
    'phs000497',
    'phs000498',
    'phs000499',
    'phs000500',
    'phs000501',
    'phs000502',
    'phs000503',
    'phs000504',
    'phs000505',
    'phs000507',
    'phs000508',
    'phs000509',
    'phs000510',
    'phs000511',
    'phs000512',
    'phs000513',
    'phs000514',
    'phs000515',
    'phs000516',
    'phs000517',
    'phs000518',
    'phs000519',
    'phs000520',
    'phs000521',
    'phs000522',
    'phs000523',
    'phs000524',
    'phs000525',
    'phs000527',
    'phs000528',
    'phs000529',
    'phs000530',
    'phs000531',
    'phs000532',
    'phs000533',
    'phs000534',
    'phs000535',
    'phs000536',
    'phs000537',
    'phs000538',
    'phs000539',
    'phs000540',
    'phs000541',
    'phs000542',
    'phs000543',
    'phs000544',
    'phs000545',
    'phs000546',
    'phs000547',
    'phs000548',
    'phs000549',
    'phs000550',
    'phs000551',
    'phs000552',
    'phs000553',
    'phs000554',
    'phs000555',
    'phs000556',
    'phs000557',
    'phs000558',
    'phs000559',
    'phs000560',
    'phs000561',
    'phs000562',
    'phs000563',
    'phs000564',
    'phs000565',
    'phs000567',
    'phs000568',
    'phs000569',
    'phs000570',
    'phs000571',
    'phs000572',
    'phs000573',
    'phs000574',
    'phs000575',
    'phs000576',
    'phs000577',
    'phs000578',
    'phs000579',
    'phs000580',
    'phs000581',
    'phs000582',
    'phs000583',
    'phs000584',
    'phs000585',
    'phs000586',
    'phs000587',
    'phs000588',
    'phs000589',
    'phs000590',
    'phs000591',
    'phs000592',
    'phs000593',
    'phs000594',
    'phs000595',
    'phs000596',
    'phs000597',
    'phs000598',
    'phs000599',
    'phs000600',
    'phs000601',
    'phs000602',
    'phs000603',
    'phs000604',
    'phs000605',
    'phs000606',
    'phs000607',
    'phs000608',
    'phs000609',
    'phs000610',
    'phs000611',
    'phs000612',
    'phs000613',
    'phs000614',
    'phs000615',
    'phs000616',
    'phs000617',
    'phs000618',
    'phs000619',
    'phs000620',
    'phs000621',
    'phs000622',
    'phs000623',
    'phs000624',
    'phs000625',
    'phs000626',
    'phs000627',
    'phs000628',
    'phs000629',
    'phs000630',
    'phs000631',
    'phs000632',
    'phs000633',
    'phs000634',
    'phs000635',
    'phs000636',
    'phs000637',
    'phs000638',
    'phs000639',
    'phs000640',
    'phs000641',
    'phs000642',
    'phs000643',
    'phs000644',
    'phs000645',
    'phs000646',
    'phs000647',
    'phs000648',
    'phs000649',
    'phs000650',
    'phs000651',
    'phs000652',
    'phs000653',
    'phs000654',
    'phs000655',
    'phs000656',
    'phs000657',
    'phs000658',
    'phs000659',
    'phs000660',
    'phs000661',
    'phs000662',
    'phs000663',
    'phs000664',
    'phs000665',
    'phs000666',
    'phs000667',
    'phs000668',
    'phs000669',
    'phs000670',
    'phs000671',
    'phs000672',
    'phs000673',
    'phs000674',
    'phs000675',
    'phs000676',
    'phs000677',
    'phs000678',
    'phs000679',
    'phs000680',
    'phs000681',
    'phs000682',
    'phs000684',
    'phs000685',
    'phs000686',
    'phs000687',
    'phs000688',
    'phs000689',
    'phs000690',
    'phs000691',
    'phs000692',
    'phs000693',
    'phs000694',
    'phs000695',
    'phs000696',
    'phs000697',
    'phs000698',
    'phs000699',
    'phs000700',
    'phs000701',
    'phs000702',
    'phs000703',
    'phs000704',
    'phs000705',
    'phs000706',
    'phs000707',
    'phs000708',
    'phs000709',
    'phs000710',
    'phs000711',
    'phs000712',
    'phs000713',
    'phs000714',
    'phs000715',
    'phs000716',
    'phs000717',
    'phs000718',
    'phs000719',
    'phs000720',
    'phs000721',
    'phs000722',
    'phs000723',
    'phs000724',
    'phs000725',
    'phs000726',
    'phs000727',
    'phs000728',
    'phs000729',
    'phs000730',
    'phs000731',
    'phs000732',
    'phs000733',
    'phs000734',
    'phs000735',
    'phs000737',
    'phs000738',
    'phs000739',
    'phs000740',
    'phs000741',
    'phs000742',
    'phs000743',
    'phs000744',
    'phs000745',
    'phs000746',
    'phs000747',
    'phs000748',
    'phs000749',
    'phs000750',
    'phs000751',
    'phs000752',
    'phs000753',
    'phs000754',
    'phs000755',
    'phs000756',
    'phs000757',
    'phs000758',
    'phs000759',
    'phs000760',
    'phs000761',
    'phs000762',
    'phs000763',
    'phs000764',
    'phs000765',
    'phs000766',
    'phs000767',
    'phs000768',
    'phs000769',
    'phs000770',
    'phs000771',
    'phs000772',
    'phs000773',
    'phs000774',
    'phs000775',
    'phs000776',
    'phs000777',
    'phs000778',
    'phs000779',
    'phs000780',
    'phs000781',
    'phs000782',
    'phs000783',
    'phs000784',
    'phs000785',
    'phs000786',
    'phs000787',
    'phs000788',
    'phs000789',
    'phs000790',
    'phs000791',
    'phs000792',
    'phs000793',
    'phs000794',
    'phs000795',
    'phs000796',
    'phs000797',
    'phs000798',
    'phs000799',
    'phs000801',
    'phs000802',
    'phs000803',
    'phs000804',
    'phs000806',
    'phs000807',
    'phs000808',
    'phs000809',
    'phs000810',
    'phs000811',
    'phs000812',
    'phs000813',
    'phs000814',
    'phs000815',
    'phs000816',
    'phs000817',
    'phs000818',
    'phs000819',
    'phs000820',
    'phs000821',
    'phs000822',
    'phs000823',
    'phs000824',
    'phs000825',
    'phs000826',
    'phs000827',
    'phs000828',
    'phs000829',
    'phs000830',
    'phs000831',
    'phs000832',
    'phs000833',
    'phs000834',
    'phs000835',
    'phs000836',
    'phs000837',
    'phs000838',
    'phs000839',
    'phs000840',
    'phs000841',
    'phs000842',
    'phs000843',
    'phs000844',
    'phs000845',
    'phs000846',
    'phs000847',
    'phs000848',
    'phs000849',
    'phs000850',
    'phs000852',
    'phs000853',
    'phs000854',
    'phs000855',
    'phs000856',
    'phs000857',
    'phs000858',
    'phs000859',
    'phs000860',
    'phs000861',
    'phs000862',
    'phs000863',
    'phs000864',
    'phs000866',
    'phs000867',
    'phs000868',
    'phs000869',
    'phs000870',
    'phs000871',
    'phs000872',
    'phs000873',
    'phs000874',
    'phs000875',
    'phs000876',
    'phs000877',
    'phs000878',
    'phs000879',
    'phs000880',
    'phs000881',
    'phs000882',
    'phs000883',
    'phs000884',
    'phs000885',
    'phs000886',
    'phs000887',
    'phs000888',
    'phs000889',
    'phs000890',
    'phs000891',
    'phs000892',
    'phs000893',
    'phs000894',
    'phs000895',
    'phs000896',
    'phs000897',
    'phs000898',
    'phs000899',
    'phs000900',
    'phs000901',
    'phs000902',
    'phs000903',
    'phs000904',
    'phs000905',
    'phs000906',
    'phs000907',
    'phs000908',
    'phs000909',
    'phs000910',
    'phs000911',
    'phs000912',
    'phs000913',
    'phs000914',
    'phs000915',
    'phs000916',
    'phs000917',
    'phs000918',
    'phs000919',
    'phs000920',
    'phs000921',
    'phs000922',
    'phs000923',
    'phs000924',
    'phs000925',
    'phs000926',
    'phs000927',
    'phs000928',
    'phs000929',
    'phs000930',
    'phs000931',
    'phs000932',
    'phs000933',
    'phs000934',
    'phs000935',
    'phs000936',
    'phs000937',
    'phs000938',
    'phs000939',
    'phs000940',
    'phs000941',
    'phs000942',
    'phs000943',
    'phs000944',
    'phs000945',
    'phs000946',
    'phs000947',
    'phs000948',
    'phs000949',
    'phs000950',
    'phs000951',
    'phs000952',
    'phs000953',
    'phs000954',
    'phs000955',
    'phs000956',
    'phs000957',
    'phs000958',
    'phs000959',
    'phs000960',
    'phs000961',
    'phs000962',
    'phs000963',
    'phs000964',
    'phs000965',
    'phs000966',
    'phs000967',
    'phs000968',
    'phs000969',
    'phs000970',
    'phs000971',
    'phs000972',
    'phs000973',
    'phs000974',
    'phs000975',
    'phs000976',
    'phs000977',
    'phs000978',
    'phs000979',
    'phs000980',
    'phs000981',
    'phs000982',
    'phs000983',
    'phs000984',
    'phs000985',
    'phs000986',
    'phs000987',
    'phs000988',
    'phs000989',
    'phs000990',
    'phs000991',
    'phs000992',
    'phs000993',
    'phs000994',
    'phs000996',
    'phs000997',
    'phs000998',
    'phs000999',
    'phs001000',
    'phs001001',
    'phs001002',
    'phs001003',
    'phs001004',
    'phs001005',
    'phs001006',
    'phs001007',
    'phs001008',
    'phs001009',
    'phs001010',
    'phs001011',
    'phs001012',
    'phs001013',
    'phs001014',
    'phs001015',
    'phs001016',
    'phs001017',
    'phs001018',
    'phs001019',
    'phs001020',
    'phs001021',
    'phs001022',
    'phs001023',
    'phs001024',
    'phs001025',
    'phs001026',
    'phs001027',
    'phs001028',
    'phs001029',
    'phs001030',
    'phs001031',
    'phs001032',
    'phs001033',
    'phs001034',
    'phs001035',
    'phs001036',
    'phs001037',
    'phs001038',
    'phs001039',
    'phs001040',
    'phs001041',
    'phs001042',
    'phs001043',
    'phs001044',
    'phs001045',
    'phs001046',
    'phs001048',
    'phs001049',
    'phs001050',
    'phs001052',
    'phs001053',
    'phs001054',
    'phs001055',
    'phs001056',
    'phs001057',
    'phs001058',
    'phs001059',
    'phs001060',
    'phs001061',
    'phs001062',
    'phs001063',
    'phs001064',
    'phs001065',
    'phs001066',
    'phs001067',
    'phs001068',
    'phs001069',
    'phs001070',
    'phs001071',
    'phs001072',
    'phs001073',
    'phs001074',
    'phs001075',
    'phs001076',
    'phs001077',
    'phs001078',
    'phs001079',
    'phs001080',
    'phs001081',
    'phs001082',
    'phs001083',
    'phs001084',
    'phs001085',
    'phs001086',
    'phs001087',
    'phs001088',
    'phs001089',
    'phs001090',
    'phs001091',
    'phs001092',
    'phs001093',
    'phs001094',
    'phs001095',
    'phs001096',
    'phs001097',
    'phs001098',
    'phs001099',
    'phs001100',
    'phs001101',
    'phs001102',
    'phs001103',
    'phs001104',
    'phs001105',
    'phs001106',
    'phs001107',
    'phs001108',
    'phs001109',
    'phs001110',
    'phs001111',
    'phs001112',
    'phs001113',
    'phs001114',
    'phs001115',
    'phs001116',
    'phs001117',
    'phs001118',
    'phs001119',
    'phs001120',
    'phs001121',
    'phs001122',
    'phs001123',
    'phs001124',
    'phs001125',
    'phs001126',
    'phs001127',
    'phs001128',
    'phs001129',
    'phs001130',
    'phs001131',
    'phs001132',
    'phs001133',
    'phs001134',
    'phs001135',
    'phs001136',
    'phs001137',
    'phs001138',
    'phs001139',
    'phs001140',
    'phs001141',
    'phs001142',
    'phs001143',
    'phs001145',
    'phs001146',
    'phs001148',
    'phs001150',
    'phs001151',
    'phs001152',
    'phs001153',
    'phs001154',
    'phs001156',
    'phs001157',
    'phs001158',
    'phs001159',
    'phs001161',
    'phs001163',
    'phs001164',
    'phs001165',
    'phs001166',
    'phs001167',
    'phs001168',
    'phs001169',
    'phs001170',
    'phs001171',
    'phs001172',
    'phs001173',
    'phs001174',
    'phs001175',
    'phs001176',
    'phs001177',
    'phs001178',
    'phs001179',
    'phs001180',
    'phs001181',
    'phs001182',
    'phs001183',
    'phs001184',
    'phs001185',
    'phs001187',
    'phs001188',
    'phs001189',
    'phs001190',
    'phs001191',
    'phs001192',
    'phs001193',
    'phs001194',
    'phs001195',
    'phs001196',
    'phs001197',
    'phs001198',
    'phs001199',
    'phs001200',
    'phs001201',
    'phs001202',
    'phs001203',
    'phs001204',
    'phs001205',
    'phs001206',
    'phs001207',
    'phs001208',
    'phs001209',
    'phs001210',
    'phs001211',
    'phs001212',
    'phs001213',
    'phs001214',
    'phs001215',
    'phs001216',
    'phs001217',
    'phs001218',
    'phs001219',
    'phs001220',
    'phs001221',
    'phs001222',
    'phs001223',
    'phs001224',
    'phs001225',
    'phs001226',
    'phs001227',
    'phs001228',
    'phs001229',
    'phs001230',
    'phs001231',
    'phs001232',
    'phs001233',
    'phs001234',
    'phs001235',
    'phs001236',
    'phs001237',
    'phs001238',
    'phs001239',
    'phs001241',
    'phs001242',
    'phs001243',
    'phs001244',
    'phs001245',
    'phs001246',
    'phs001247',
    'phs001248',
    'phs001249',
    'phs001250',
    'phs001251',
    'phs001252',
    'phs001253',
    'phs001254',
    'phs001255',
    'phs001256',
    'phs001257',
    'phs001258',
    'phs001259',
    'phs001260',
    'phs001261',
    'phs001262',
    'phs001263',
    'phs001264',
    'phs001265',
    'phs001266',
    'phs001267',
    'phs001268',
    'phs001269',
    'phs001270',
    'phs001271',
    'phs001272',
    'phs001273',
    'phs001274',
    'phs001275',
    'phs001276',
    'phs001277',
    'phs001278',
    'phs001279',
    'phs001280',
    'phs001281',
    'phs001282',
    'phs001283',
    'phs001284',
    'phs001285',
    'phs001286',
    'phs001287',
    'phs001288',
    'phs001289',
    'phs001290',
    'phs001291',
    'phs001292',
    'phs001293',
    'phs001294',
    'phs001295',
    'phs001296',
    'phs001297',
    'phs001298',
    'phs001299',
    'phs001300',
    'phs001301',
    'phs001302',
    'phs001303',
    'phs001304',
    'phs001305',
    'phs001306',
    'phs001307',
    'phs001308',
    'phs001309',
    'phs001310',
    'phs001311',
    'phs001312',
    'phs001313',
    'phs001314',
    'phs001315',
    'phs001316',
    'phs001317',
    'phs001318',
    'phs001319',
    'phs001320',
    'phs001321',
    'phs001322',
    'phs001323',
    'phs001324',
    'phs001325',
    'phs001326',
    'phs001327',
    'phs001328',
    'phs001329',
    'phs001330',
    'phs001331',
    'phs001332',
    'phs001333',
    'phs001334',
    'phs001335',
    'phs001336',
    'phs001337',
    'phs001338',
    'phs001339',
    'phs001340',
    'phs001341',
    'phs001342',
    'phs001343',
    'phs001344',
    'phs001345',
    'phs001346',
    'phs001347',
    'phs001348',
    'phs001349',
    'phs001350',
    'phs001351',
    'phs001352',
    'phs001353',
    'phs001354',
    'phs001356',
    'phs001357',
    'phs001358',
    'phs001359',
    'phs001360',
    'phs001361',
    'phs001362',
    'phs001363',
    'phs001364',
    'phs001365',
    'phs001366',
    'phs001367',
    'phs001368',
    'phs001369',
    'phs001370',
    'phs001371',
    'phs001372',
    'phs001373',
    'phs001374',
    'phs001375',
    'phs001376',
    'phs001377',
    'phs001378',
    'phs001379',
    'phs001380',
    'phs001381',
    'phs001382',
    'phs001383',
    'phs001384',
    'phs001385',
    'phs001386',
    'phs001387',
    'phs001388',
    'phs001389',
    'phs001390',
    'phs001391',
    'phs001392',
    'phs001393',
    'phs001394',
    'phs001395',
    'phs001396',
    'phs001397',
    'phs001398',
    'phs001399',
    'phs001400',
    'phs001401',
    'phs001402',
    'phs001403',
    'phs001404',
    'phs001405',
    'phs001406',
    'phs001407',
    'phs001408',
    'phs001409',
    'phs001410',
    'phs001411',
    'phs001412',
    'phs001413',
    'phs001414',
    'phs001415',
    'phs001416',
    'phs001417',
    'phs001418',
    'phs001419',
    'phs001420',
    'phs001421',
    'phs001422',
    'phs001423',
    'phs001424',
    'phs001425',
    'phs001426',
    'phs001427',
    'phs001428',
    'phs001429',
    'phs001430',
    'phs001431',
    'phs001432',
    'phs001433',
    'phs001434',
    'phs001435',
    'phs001436',
    'phs001437',
    'phs001438',
    'phs001439',
    'phs001440',
    'phs001441',
    'phs001442',
    'phs001443',
    'phs001444',
    'phs001445',
    'phs001446',
    'phs001447',
    'phs001448',
    'phs001450',
    'phs001451',
    'phs001452',
    'phs001454',
    'phs001455',
    'phs001457',
    'phs001458',
    'phs001459',
    'phs001460',
    'phs001461',
    'phs001462',
    'phs001463',
    'phs001464',
    'phs001465',
    'phs001466',
    'phs001467',
    'phs001468',
    'phs001469',
    'phs001470',
    'phs001471',
    'phs001472',
    'phs001473',
    'phs001474',
    'phs001475',
    'phs001476',
    'phs001477',
    'phs001478',
    'phs001479',
    'phs001480',
    'phs001481',
    'phs001482',
    'phs001483',
    'phs001484',
    'phs001485',
    'phs001486',
    'phs001487',
    'phs001488',
    'phs001489',
    'phs001490',
    'phs001491',
    'phs001492',
    'phs001493',
    'phs001494',
    'phs001495',
    'phs001496',
    'phs001497',
    'phs001498',
    'phs001499',
    'phs001500',
    'phs001501',
    'phs001502',
    'phs001503',
    'phs001504',
    'phs001505',
    'phs001507',
    'phs001508',
    'phs001509',
    'phs001510',
    'phs001511',
    'phs001512',
    'phs001513',
    'phs001514',
    'phs001515',
    'phs001516',
    'phs001517',
    'phs001518',
    'phs001519',
    'phs001520',
    'phs001521',
    'phs001522',
    'phs001523',
    'phs001524',
    'phs001525',
    'phs001526',
    'phs001527',
    'phs001528',
    'phs001529',
    'phs001531',
    'phs001532',
    'phs001533',
    'phs001534',
    'phs001535',
    'phs001536',
    'phs001538',
    'phs001539',
    'phs001540',
    'phs001541',
    'phs001542',
    'phs001543',
    'phs001544',
    'phs001545',
    'phs001546',
    'phs001547',
    'phs001548',
    'phs001549',
    'phs001550',
    'phs001551',
    'phs001552',
    'phs001553',
    'phs001554',
    'phs001555',
    'phs001556',
    'phs001557',
    'phs001558',
    'phs001559',
    'phs001560',
    'phs001561',
    'phs001562',
    'phs001563',
    'phs001564',
    'phs001565',
    'phs001566',
    'phs001568',
    'phs001569',
    'phs001570',
    'phs001571',
    'phs001572',
    'phs001573',
    'phs001575',
    'phs001576',
    'phs001577',
    'phs001578',
    'phs001579',
    'phs001580',
    'phs001581',
    'phs001584',
    'phs001585',
    'phs001586',
    'phs001587',
    'phs001588',
    'phs001589',
    'phs001590',
    'phs001591',
    'phs001592',
    'phs001593',
    'phs001594',
    'phs001595',
    'phs001596',
    'phs001597',
    'phs001598',
    'phs001599',
    'phs001600',
    'phs001601',
    'phs001602',
    'phs001603',
    'phs001604',
    'phs001605',
    'phs001606',
    'phs001607',
    'phs001608',
    'phs001609',
    'phs001610',
    'phs001611',
    'phs001612',
    'phs001613',
    'phs001614',
    'phs001615',
    'phs001616',
    'phs001617',
    'phs001618',
    'phs001619',
    'phs001620',
    'phs001621',
    'phs001622',
    'phs001623',
    'phs001624',
    'phs001625',
    'phs001626',
    'phs001627',
    'phs001628',
    'phs001629',
    'phs001630',
    'phs001631',
    'phs001632',
    'phs001633',
    'phs001634',
    'phs001635',
    'phs001636',
    'phs001637',
    'phs001638',
    'phs001639',
    'phs001640',
    'phs001641',
    'phs001642',
    'phs001643',
    'phs001644',
    'phs001645',
    'phs001646',
    'phs001647',
    'phs001648',
    'phs001649',
    'phs001650',
    'phs001651',
    'phs001652',
    'phs001653',
    'phs001654',
    'phs001655',
    'phs001656',
    'phs001657',
    'phs001658',
    'phs001659',
    'phs001660',
    'phs001661',
    'phs001662',
    'phs001663',
    'phs001664',
    'phs001665',
    'phs001666',
    'phs001667',
    'phs001669',
    'phs001670',
    'phs001671',
    'phs001672',
    'phs001673',
    'phs001674',
    'phs001675',
    'phs001676',
    'phs001677',
    'phs001678',
    'phs001679',
    'phs001680',
    'phs001681',
    'phs001682',
    'phs001683',
    'phs001684',
    'phs001685',
    'phs001686',
    'phs001687',
    'phs001688',
    'phs001689',
    'phs001690',
    'phs001691',
    'phs001692',
    'phs001693',
    'phs001694',
    'phs001695',
    'phs001696',
    'phs001697',
    'phs001698',
    'phs001699',
    'phs001700',
    'phs001701',
    'phs001702',
    'phs001703',
    'phs001704',
    'phs001705',
    'phs001706',
    'phs001707',
    'phs001708',
    'phs001709',
    'phs001710',
    'phs001711',
    'phs001712',
    'phs001713',
    'phs001714',
    'phs001715',
    'phs001716',
    'phs001717',
    'phs001718',
    'phs001719',
    'phs001720',
    'phs001721',
    'phs001722',
    'phs001723',
    'phs001724',
    'phs001725',
    'phs001726',
    'phs001727',
    'phs001728',
    'phs001729',
    'phs001730',
    'phs001731',
    'phs001732',
    'phs001733',
    'phs001734',
    'phs001735',
    'phs001736',
    'phs001737',
    'phs001738',
    'phs001739',
    'phs001740',
    'phs001741',
    'phs001742',
    'phs001743',
    'phs001744',
    'phs001745',
    'phs001746',
    'phs001747',
    'phs001748',
    'phs001749',
    'phs001750',
    'phs001751',
    'phs001752',
    'phs001753',
    'phs001754',
    'phs001757',
    'phs001758',
    'phs001759',
    'phs001760',
    'phs001761',
    'phs001762',
    'phs001763',
    'phs001764',
    'phs001765',
    'phs001766',
    'phs001767',
    'phs001768',
    'phs001769',
    'phs001770',
    'phs001771',
    'phs001772',
    'phs001773',
    'phs001774',
    'phs001775',
    'phs001776',
    'phs001777',
    'phs001778',
    'phs001779',
    'phs001780',
    'phs001781',
    'phs001782',
    'phs001783',
    'phs001784',
    'phs001785',
    'phs001786',
    'phs001787',
    'phs001788',
    'phs001789',
    'phs001790',
    'phs001791',
    'phs001792',
    'phs001793',
    'phs001794',
    'phs001795',
    'phs001796',
    'phs001797',
    'phs001798',
    'phs001799',
    'phs001800',
    'phs001801',
    'phs001802',
    'phs001803',
    'phs001804',
    'phs001805',
    'phs001806',
    'phs001807',
    'phs001808',
    'phs001809',
    'phs001810',
    'phs001811',
    'phs001812',
    'phs001813',
    'phs001814',
    'phs001815',
    'phs001816',
    'phs001817',
    'phs001818',
    'phs001819',
    'phs001820',
    'phs001821',
    'phs001822',
    'phs001823',
    'phs001824',
    'phs001825',
    'phs001826',
    'phs001827',
    'phs001828',
    'phs001829',
    'phs001830',
    'phs001831',
    'phs001833',
    'phs001834',
    'phs001835',
    'phs001836',
    'phs001837',
    'phs001838',
    'phs001839',
    'phs001840',
    'phs001841',
    'phs001842',
    'phs001843',
    'phs001844',
    'phs001845',
    'phs001846',
    'phs001847',
    'phs001848',
    'phs001849',
    'phs001850',
    'phs001851',
    'phs001852',
    'phs001853',
    'phs001854',
    'phs001855',
    'phs001856',
    'phs001857',
    'phs001858',
    'phs001859',
    'phs001860',
    'phs001861',
    'phs001862',
    'phs001863',
    'phs001864',
    'phs001865',
    'phs001866',
    'phs001867',
    'phs001868',
    'phs001869',
    'phs001870',
    'phs001871',
    'phs001872',
    'phs001874',
    'phs001875',
    'phs001876',
    'phs001877',
    'phs001878',
    'phs001879',
    'phs001880',
    'phs001881',
    'phs001882',
    'phs001883',
    'phs001884',
    'phs001885',
    'phs001886',
    'phs001887',
    'phs001888',
    'phs001889',
    'phs001890',
    'phs001891',
    'phs001892',
    'phs001893',
    'phs001894',
    'phs001895',
    'phs001896',
    'phs001897',
    'phs001898',
    'phs001899',
    'phs001900',
    'phs001901',
    'phs001902',
    'phs001903',
    'phs001904',
    'phs001905',
    'phs001906',
    'phs001907',
    'phs001908',
    'phs001909',
    'phs001910',
    'phs001911',
    'phs001912',
    'phs001913',
    'phs001914',
    'phs001915',
    'phs001916',
    'phs001917',
    'phs001918',
    'phs001919',
    'phs001920',
    'phs001921',
    'phs001922',
    'phs001923',
    'phs001924',
    'phs001925',
    'phs001926',
    'phs001927',
    'phs001928',
    'phs001929',
    'phs001930',
    'phs001931',
    'phs001932',
    'phs001933',
    'phs001934',
    'phs001935',
    'phs001936',
    'phs001937',
    'phs001938',
    'phs001939',
    'phs001940',
    'phs001941',
    'phs001942',
    'phs001943',
    'phs001944',
    'phs001945',
    'phs001946',
    'phs001947',
    'phs001948',
    'phs001949',
    'phs001951',
    'phs001952',
    'phs001953',
    'phs001954',
    'phs001955',
    'phs001956',
    'phs001957',
    'phs001958',
    'phs001959',
    'phs001960',
    'phs001961',
    'phs001962',
    'phs001963',
    'phs001964',
    'phs001965',
    'phs001966',
    'phs001967',
    'phs001968',
    'phs001969',
    'phs001970',
    'phs001971',
    'phs001972',
    'phs001973',
    'phs001974',
    'phs001975',
    'phs001976',
    'phs001977',
    'phs001978',
    'phs001979',
    'phs001980',
    'phs001981',
    'phs001982',
    'phs001983',
    'phs001984',
    'phs001985',
    'phs001986',
    'phs001987',
    'phs001988',
    'phs001989',
    'phs001990',
    'phs001991',
    'phs001992',
    'phs001993',
    'phs001994',
    'phs001995',
    'phs001996',
    'phs001997',
    'phs001998',
    'phs001999',
    'phs002000',
    'phs002001',
    'phs002002',
    'phs002003',
    'phs002004',
    'phs002005',
    'phs002006',
    'phs002007',
    'phs002008',
    'phs002009',
    'phs002010',
    'phs002011',
    'phs002012',
    'phs002013',
    'phs002014',
    'phs002015',
    'phs002016',
    'phs002017',
    'phs002018',
    'phs002019',
    'phs002020',
    'phs002021',
    'phs002022',
    'phs002023',
    'phs002024',
    'phs002025',
    'phs002026',
    'phs002027',
    'phs002028',
    'phs002029',
    'phs002030',
    'phs002031',
    'phs002032',
    'phs002033',
    'phs002034',
    'phs002035',
    'phs002036',
    'phs002037',
    'phs002038',
    'phs002039',
    'phs002040',
    'phs002041',
    'phs002042',
    'phs002043',
    'phs002044',
    'phs002045',
    'phs002046',
    'phs002047',
    'phs002048',
    'phs002049',
    'phs002050',
    'phs002051',
    'phs002052',
    'phs002053',
    'phs002054',
    'phs002055',
    'phs002056',
    'phs002057',
    'phs002058',
    'phs002059',
    'phs002060',
    'phs002061',
    'phs002062',
    'phs002063',
    'phs002064',
    'phs002065',
    'phs002066',
    'phs002067',
    'phs002068',
    'phs002069',
    'phs002070',
    'phs002071',
    'phs002073',
    'phs002074',
    'phs002075',
    'phs002076',
    'phs002077',
    'phs002081',
    'phs002083',
    'phs002084',
    'phs002085',
    'phs002086',
    'phs002088',
    'phs002089',
    'phs002090',
    'phs002093',
    'phs002094',
    'phs002095',
    'phs002096',
    'phs002097',
    'phs002098',
    'phs002099',
    'phs002100',
    'phs002101',
    'phs002102',
    'phs002103',
    'phs002104',
    'phs002105',
    'phs002106',
    'phs002107',
    'phs002108',
    'phs002109',
    'phs002110',
    'phs002111',
    'phs002112',
    'phs002113',
    'phs002114',
    'phs002115',
    'phs002116',
    'phs002117',
    'phs002118',
    'phs002119',
    'phs002120',
    'phs002121',
    'phs002122',
    'phs002123',
    'phs002124',
    'phs002125',
    'phs002126',
    'phs002127',
    'phs002128',
    'phs002129',
    'phs002130',
    'phs002131',
    'phs002132',
    'phs002133',
    'phs002136',
    'phs002137',
    'phs002138',
    'phs002139',
    'phs002140',
    'phs002141',
    'phs002142',
    'phs002143',
    'phs002144',
    'phs002145',
    'phs002146',
    'phs002147',
    'phs002148',
    'phs002149',
    'phs002150',
    'phs002151',
    'phs002152',
    'phs002153',
    'phs002154',
    'phs002155',
    'phs002156',
    'phs002157',
    'phs002158',
    'phs002159',
    'phs002161',
    'phs002162',
    'phs002164',
    'phs002165',
    'phs002166',
    'phs002167',
    'phs002168',
    'phs002169',
    'phs002170',
    'phs002171',
    'phs002172',
    'phs002173',
    'phs002174',
    'phs002175',
    'phs002176',
    'phs002177',
    'phs002178',
    'phs002179',
    'phs002180',
    'phs002181',
    'phs002182',
    'phs002183',
    'phs002184',
    'phs002185',
    'phs002186',
    'phs002187',
    'phs002188',
    'phs002189',
    'phs002190',
    'phs002192',
    'phs002193',
    'phs002194',
    'phs002195',
    'phs002196',
    'phs002197',
    'phs002198',
    'phs002199',
    'phs002200',
    'phs002201',
    'phs002202',
    'phs002203',
    'phs002204',
    'phs002205',
    'phs002206',
    'phs002207',
    'phs002208',
    'phs002209',
    'phs002210',
    'phs002211',
    'phs002212',
    'phs002213',
    'phs002214',
    'phs002215',
    'phs002216',
    'phs002217',
    'phs002218',
    'phs002219',
    'phs002220',
    'phs002221',
    'phs002222',
    'phs002223',
    'phs002224',
    'phs002225',
    'phs002226',
    'phs002228',
    'phs002229',
    'phs002230',
    'phs002231',
    'phs002232',
    'phs002233',
    'phs002234',
    'phs002235',
    'phs002236',
    'phs002237',
    'phs002238',
    'phs002239',
    'phs002240',
    'phs002242',
    'phs002243',
    'phs002244',
    'phs002245',
    'phs002246',
    'phs002247',
    'phs002248',
    'phs002249',
    'phs002250',
    'phs002251',
    'phs002252',
    'phs002253',
    'phs002254',
    'phs002255',
    'phs002256',
    'phs002258',
    'phs002259',
    'phs002260',
    'phs002261',
    'phs002262',
    'phs002264',
    'phs002265',
    'phs002266',
    'phs002267',
    'phs002268',
    'phs002269',
    'phs002270',
    'phs002272',
    'phs002273',
    'phs002274',
    'phs002275',
    'phs002276',
    'phs002278',
    'phs002279',
    'phs002280',
    'phs002282',
    'phs002283',
    'phs002284',
    'phs002285',
    'phs002286',
    'phs002287',
    'phs002288',
    'phs002289',
    'phs002290',
    'phs002291',
    'phs002292',
    'phs002293',
    'phs002294',
    'phs002295',
    'phs002296',
    'phs002297',
    'phs002298',
    'phs002299',
    'phs002300',
    'phs002301',
    'phs002302',
    'phs002303',
    'phs002304',
    'phs002305',
    'phs002306',
    'phs002307',
    'phs002308',
    'phs002309',
    'phs002310',
    'phs002311',
    'phs002312',
    'phs002313',
    'phs002314',
    'phs002315',
    'phs002316',
    'phs002317',
    'phs002318',
    'phs002319',
    'phs002320',
    'phs002321',
    'phs002322',
    'phs002323',
    'phs002324',
    'phs002325',
    'phs002326',
    'phs002327',
    'phs002328',
    'phs002329',
    'phs002330',
    'phs002331',
    'phs002332',
    'phs002333',
    'phs002334',
    'phs002335',
    'phs002336',
    'phs002337',
    'phs002338',
    'phs002339',
    'phs002340',
    'phs002341',
    'phs002342',
    'phs002343',
    'phs002344',
    'phs002345',
    'phs002346',
    'phs002347',
    'phs002348',
    'phs002349',
    'phs002350',
    'phs002351',
    'phs002352',
    'phs002353',
    'phs002357',
    'phs002358',
    'phs002359',
    'phs002360',
    'phs002361',
    'phs002362',
    'phs002363',
    'phs002364',
    'phs002365',
    'phs002366',
    'phs002367',
    'phs002368',
    'phs002369',
    'phs002370',
    'phs002371',
    'phs002372',
    'phs002373',
    'phs002375',
    'phs002376',
    'phs002377',
    'phs002378',
    'phs002379',
    'phs002380',
    'phs002381',
    'phs002382',
    'phs002383',
    'phs002384',
    'phs002385',
    'phs002386',
    'phs002387',
    'phs002388',
    'phs002389',
    'phs002390',
    'phs002391',
    'phs002392',
    'phs002393',
    'phs002394',
    'phs002395',
    'phs002396',
    'phs002397',
    'phs002398',
    'phs002399',
    'phs002400',
    'phs002401',
    'phs002402',
    'phs002403',
    'phs002404',
    'phs002405',
    'phs002406',
    'phs002407',
    'phs002408',
    'phs002409',
    'phs002410',
    'phs002411',
    'phs002412',
    'phs002414',
    'phs002415',
    'phs002416',
    'phs002417',
    'phs002418',
    'phs002419',
    'phs002420',
    'phs002421',
    'phs002422',
    'phs002423',
    'phs002424',
    'phs002425',
    'phs002426',
    'phs002427',
    'phs002428',
    'phs002429',
    'phs002430',
    'phs002431',
    'phs002432',
    'phs002433',
    'phs002434',
    'phs002435',
    'phs002436',
    'phs002437',
    'phs002438',
    'phs002439',
    'phs002440',
    'phs002441',
    'phs002442',
    'phs002443',
    'phs002444',
    'phs002445',
    'phs002446',
    'phs002447',
    'phs002448',
    'phs002449',
    'phs002450',
    'phs002451',
    'phs002453',
    'phs002454',
    'phs002455',
    'phs002456',
    'phs002457',
    'phs002458',
    'phs002459',
    'phs002460',
    'phs002461',
    'phs002462',
    'phs002463',
    'phs002464',
    'phs002465',
    'phs002466',
    'phs002467',
    'phs002468',
    'phs002469',
    'phs002470',
    'phs002471',
    'phs002472',
    'phs002473',
    'phs002474',
    'phs002475',
    'phs002476',
    'phs002477',
    'phs002478',
    'phs002479',
    'phs002480',
    'phs002481',
    'phs002482',
    'phs002483',
    'phs002484',
    'phs002485',
    'phs002486',
    'phs002487',
    'phs002489',
    'phs002491',
    'phs002492',
    'phs002493',
    'phs002494',
    'phs002495',
    'phs002496',
    'phs002497',
    'phs002498',
    'phs002499',
    'phs002500',
    'phs002501',
    'phs002502',
    'phs002503',
    'phs002504',
    'phs002505',
    'phs002506',
    'phs002507',
    'phs002508',
    'phs002509',
    'phs002510',
    'phs002511',
    'phs002512',
    'phs002513',
    'phs002514',
    'phs002515',
    'phs002516',
    'phs002517',
    'phs002518',
    'phs002519',
    'phs002520',
    'phs002521',
    'phs002522',
    'phs002523',
    'phs002524',
    'phs002525',
    'phs002526',
    'phs002527',
    'phs002528',
    'phs002529',
    'phs002530',
    'phs002533',
    'phs002534',
    'phs002535',
    'phs002537',
    'phs002538',
    'phs002539',
    'phs002540',
    'phs002541',
    'phs002542',
    'phs002543',
    'phs002544',
    'phs002545',
    'phs002546',
    'phs002547',
    'phs002548',
    'phs002549',
    'phs002550',
    'phs002551',
    'phs002552',
    'phs002553',
    'phs002554',
    'phs002555',
    'phs002556',
    'phs002557',
    'phs002558',
    'phs002559',
    'phs002560',
    'phs002561',
    'phs002562',
    'phs002563',
    'phs002564',
    'phs002565',
    'phs002566',
    'phs002567',
    'phs002568',
    'phs002569',
    'phs002570',
    'phs002571',
    'phs002572',
    'phs002573',
    'phs002575',
    'phs002576',
    'phs002577',
    'phs002578',
    'phs002579',
    'phs002580',
    'phs002581',
    'phs002582',
    'phs002583',
    'phs002584',
    'phs002585',
    'phs002586',
    'phs002587',
    'phs002588',
    'phs002589',
    'phs002590',
    'phs002591',
    'phs002592',
    'phs002593',
    'phs002594',
    'phs002595',
    'phs002596',
    'phs002597',
    'phs002598',
    'phs002599',
    'phs002600',
    'phs002601',
    'phs002602',
    'phs002603',
    'phs002604',
    'phs002605',
    'phs002606',
    'phs002608',
    'phs002609',
    'phs002610',
    'phs002611',
    'phs002612',
    'phs002613',
    'phs002614',
    'phs002615',
    'phs002616',
    'phs002617',
    'phs002618',
    'phs002619',
    'phs002620',
    'phs002621',
    'phs002622',
    'phs002623',
    'phs002624',
    'phs002625',
    'phs002626',
    'phs002627',
    'phs002628',
    'phs002629',
    'phs002630',
    'phs002631',
    'phs002632',
    'phs002633',
    'phs002634',
    'phs002635',
    'phs002636',
    'phs002637',
    'phs002638',
    'phs002639',
    'phs002640',
    'phs002641',
    'phs002642',
    'phs002643',
    'phs002644',
    'phs002645',
    'phs002647',
    'phs002648',
    'phs002649',
    'phs002650',
    'phs002651',
    'phs002652',
    'phs002653',
    'phs002654',
    'phs002655',
    'phs002656',
    'phs002657',
    'phs002658',
    'phs002659',
    'phs002660',
    'phs002661',
    'phs002662',
    'phs002663',
    'phs002675',
    'phs002677',
    'phs002678',
    'phs002679',
    'phs002681',
    'phs002682',
    'phs002683',
    'phs002684',
    'phs002685',
    'phs002686',
    'phs002687',
    'phs002688',
    'phs002689',
    'phs002690',
    'phs002691',
    'phs002692',
    'phs002693',
    'phs002694',
    'phs002695',
    'phs002696',
    'phs002697',
    'phs002698',
    'phs002699',
    'phs002700',
    'phs002701',
    'phs002702',
    'phs002703',
    'phs002704',
    'phs002705',
    'phs002706',
    'phs002707',
    'phs002708',
    'phs002709',
    'phs002710',
    'phs002711',
    'phs002712',
    'phs002713',
    'phs002714',
    'phs002715',
    'phs002716',
    'phs002717',
    'phs002718',
    'phs002719',
    'phs002720',
    'phs002721',
    'phs002722',
    'phs002723',
    'phs002724',
    'phs002725',
    'phs002726',
    'phs002727',
    'phs002728',
    'phs002729',
    'phs002730',
    'phs002731',
    'phs002732',
    'phs002733',
    'phs002734',
    'phs002735',
    'phs002736',
    'phs002737',
    'phs002738',
    'phs002739',
    'phs002740',
    'phs002741',
    'phs002742',
    'phs002743',
    'phs002744',
    'phs002745',
    'phs002746',
    'phs002747',
    'phs002748',
    'phs002749',
    'phs002750',
    'phs002751',
    'phs002752',
    'phs002753',
    'phs002754',
    'phs002755',
    'phs002756',
    'phs002758',
    'phs002759',
    'phs002760',
    'phs002761',
    'phs002762',
    'phs002763',
    'phs002764',
    'phs002765',
    'phs002766',
    'phs002767',
    'phs002768',
    'phs002769',
    'phs002770',
    'phs002771',
    'phs002772',
    'phs002774',
    'phs002775',
    'phs002776',
    'phs002777',
    'phs002778',
    'phs002780',
    'phs002781',
    'phs002782',
    'phs002783',
    'phs002785',
    'phs002786',
    'phs002787',
    'phs002788',
    'phs002789',
    'phs002790',
    'phs002791',
    'phs002792',
    'phs002793',
    'phs002794',
    'phs002795',
    'phs002796',
    'phs002797',
    'phs002798',
    'phs002799',
    'phs002800',
    'phs002801',
    'phs002802',
    'phs002803',
    'phs002804',
    'phs002805',
    'phs002806',
    'phs002807',
    'phs002808',
    'phs002809',
    'phs002810',
    'phs002811',
    'phs002812',
    'phs002813',
    'phs002814',
    'phs002815',
    'phs002816',
    'phs002817',
    'phs002818',
    'phs002819',
    'phs002820',
    'phs002821',
    'phs002822',
    'phs002823',
    'phs002824',
    'phs002825',
    'phs002826',
    'phs002827',
    'phs002828',
    'phs002833',
    'phs002834',
    'phs002835',
    'phs002836',
    'phs002840',
    'phs002841',
    'phs002842',
    'phs002844',
    'phs002845',
    'phs002846',
    'phs002847',
    'phs002848',
    'phs002849',
    'phs002850',
    'phs002851',
    'phs002852',
    'phs002854',
    'phs002855',
    'phs002856',
    'phs002857',
    'phs002858',
    'phs002859',
    'phs002860',
    'phs002861',
    'phs002862',
    'phs002863',
    'phs002864',
    'phs002865',
    'phs002866',
    'phs002867',
    'phs002868',
    'phs002869',
    'phs002870',
    'phs002871',
    'phs002872',
    'phs002873',
    'phs002874',
    'phs002875',
    'phs002876',
    'phs002877',
    'phs002878',
    'phs002879',
    'phs002880',
    'phs002881',
    'phs002882',
    'phs002883',
    'phs002884',
    'phs002885',
    'phs002886',
    'phs002887',
    'phs002888',
    'phs002889',
    'phs002890',
    'phs002891',
    'phs002892',
    'phs002893',
    'phs002894',
    'phs002895',
    'phs002896',
    'phs002897',
    'phs002898',
    'phs002899',
    'phs002900',
    'phs002901',
    'phs002902',
    'phs002903',
    'phs002904',
    'phs002905',
    'phs002906',
    'phs002907',
    'phs002908',
    'phs002909',
    'phs002910',
    'phs002911',
    'phs002912',
    'phs002913',
    'phs002914',
    'phs002915',
    'phs002916',
    'phs002917',
    'phs002918',
    'phs002919',
    'phs002920',
    'phs002921',
    'phs002922',
    'phs002923',
    'phs002924',
    'phs002925',
    'phs002926',
    'phs002927',
    'phs002928',
    'phs002929',
    'phs002930',
    'phs002931',
    'phs002932',
    'phs002933',
    'phs002934',
    'phs002935',
    'phs002936',
    'phs002937',
    'phs002938',
    'phs002939',
    'phs002940',
    'phs002941',
    'phs002942',
    'phs002943',
    'phs002944',
    'phs002945',
    'phs002946',
    'phs002947',
    'phs002948',
    'phs002949',
    'phs002950',
    'phs002951',
    'phs002952',
    'phs002953',
    'phs002954',
    'phs002955',
    'phs002956',
    'phs002957',
    'phs002958',
    'phs002959',
    'phs002960',
    'phs002961',
    'phs002962',
    'phs002963',
    'phs002964',
    'phs002965',
    'phs002966',
    'phs002968',
    'phs002969',
    'phs002971',
    'phs002972',
    'phs002973',
    'phs002975',
    'phs002976',
    'phs002977',
    'phs002978',
    'phs002979',
    'phs002980',
    'phs002981',
    'phs002982',
    'phs002983',
    'phs002984',
    'phs002985',
    'phs002986',
    'phs002987',
    'phs002988',
    'phs002989',
    'phs002990',
    'phs002991',
    'phs002992',
    'phs002993',
    'phs002994',
    'phs002996',
    'phs002997',
    'phs002998',
    'phs002999',
    'phs003000',
    'phs003001',
    'phs003002',
    'phs003003',
    'phs003004',
    'phs003005',
    'phs003006',
    'phs003007',
    'phs003008',
    'phs003009',
    'phs003010',
    'phs003011',
    'phs003012',
    'phs003013',
    'phs003014',
    'phs003015',
    'phs003016',
    'phs003017',
    'phs003019',
    'phs003020',
    'phs003021',
    'phs003022',
    'phs003023',
    'phs003024',
    'phs003025',
    'phs003026',
    'phs003027',
    'phs003028',
    'phs003029',
    'phs003030',
    'phs003031',
    'phs003032',
    'phs003033',
    'phs003034',
    'phs003035',
    'phs003036',
    'phs003037',
    'phs003038',
    'phs003039',
    'phs003040',
    'phs003041',
    'phs003042',
    'phs003043',
    'phs003044',
    'phs003045',
    'phs003046',
    'phs003047',
    'phs003048',
    'phs003049',
    'phs003050',
    'phs003051',
    'phs003052',
    'phs003053',
    'phs003054',
    'phs003055',
    'phs003056',
    'phs003057',
    'phs003059',
    'phs003060',
    'phs003061',
    'phs003062',
    'phs003063',
    'phs003064',
    'phs003065',
    'phs003066',
    'phs003067',
    'phs003068',
    'phs003069',
    'phs003070',
    'phs003071',
    'phs003072',
    'phs003073',
    'phs003074',
    'phs003075',
    'phs003076',
    'phs003077',
    'phs003078',
    'phs003079',
    'phs003080',
    'phs003081',
    'phs003082',
    'phs003083',
    'phs003084',
    'phs003085',
    'phs003086',
    'phs003087',
    'phs003089',
    'phs003090',
    'phs003091',
    'phs003092',
    'phs003093',
    'phs003094',
    'phs003095',
    'phs003096',
    'phs003097',
    'phs003098',
    'phs003099',
    'phs003100',
    'phs003101',
    'phs003102',
    'phs003103',
    'phs003104',
    'phs003105',
    'phs003106',
    'phs003107',
    'phs003108',
    'phs003109',
    'phs003110',
    'phs003111',
    'phs003112',
    'phs003113',
    'phs003115',
    'phs003116',
    'phs003117',
    'phs003118',
    'phs003119',
    'phs003120',
    'phs003121',
    'phs003122',
    'phs003123',
    'phs003124',
    'phs003125',
    'phs003126',
    'phs003127',
    'phs003128',
    'phs003131',
    'phs003132',
    'phs003133',
    'phs003134',
    'phs003135',
    'phs003136',
    'phs003137',
    'phs003138',
    'phs003139',
    'phs003140',
    'phs003141',
    'phs003143',
    'phs003144',
    'phs003145',
    'phs003146',
    'phs003147',
    'phs003148',
    'phs003149',
    'phs003150',
    'phs003151',
    'phs003152',
    'phs003153',
    'phs003154',
    'phs003155',
    'phs003156',
    'phs003157',
    'phs003158',
    'phs003159',
    'phs003160',
    'phs003161',
    'phs003162',
    'phs003163',
    'phs003164',
    'phs003165',
    'phs003166',
    'phs003167',
    'phs003168',
    'phs003169',
    'phs003170',
    'phs003172',
    'phs003173',
    'phs003174',
    'phs003175',
    'phs003177',
    'phs003178',
    'phs003179',
    'phs003180',
    'phs003181',
    'phs003182',
    'phs003183',
    'phs003184',
    'phs003185',
    'phs003186',
    'phs003187',
    'phs003188',
    'phs003189',
    'phs003190',
    'phs003191',
    'phs003192',
    'phs003193',
    'phs003194',
    'phs003195',
    'phs003196',
    'phs003197',
    'phs003198',
    'phs003199',
    'phs003200',
    'phs003201',
    'phs003202',
    'phs003203',
    'phs003204',
    'phs003205',
    'phs003206',
    'phs003207',
    'phs003208',
    'phs003209',
    'phs003210',
    'phs003211',
    'phs003212',
    'phs003213',
    'phs003214',
    'phs003215',
    'phs003216',
    'phs003217',
    'phs003218',
    'phs003219',
    'phs003220',
    'phs003221',
    'phs003222',
    'phs003223',
    'phs003225',
    'phs003226',
    'phs003227',
    'phs003228',
    'phs003229',
    'phs003230',
    'phs003231',
    'phs003232',
    'phs003233',
    'phs003234',
    'phs003235',
    'phs003236',
    'phs003237',
    'phs003238',
    'phs003239',
    'phs003240',
    'phs003241',
    'phs003242',
    'phs003243',
    'phs003244',
    'phs003245',
    'phs003246',
    'phs003247',
    'phs003248',
    'phs003249',
    'phs003250',
    'phs003251',
    'phs003252',
    'phs003253',
    'phs003254',
    'phs003255',
    'phs003256',
    'phs003257',
    'phs003258',
    'phs003259',
    'phs003260',
    'phs003261',
    'phs003262',
    'phs003263',
    'phs003264',
    'phs003265',
    'phs003266',
    'phs003267',
    'phs003268',
    'phs003269',
    'phs003270',
    'phs003271',
    'phs003272',
    'phs003273',
    'phs003274',
    'phs003275',
    'phs003276',
    'phs003277',
    'phs003278',
    'phs003279',
    'phs003280',
    'phs003281',
    'phs003282',
    'phs003283',
    'phs003284',
    'phs003285',
    'phs003286',
    'phs003287',
    'phs003288',
    'phs003289',
    'phs003290',
    'phs003291',
    'phs003292',
    'phs003293',
    'phs003294',
    'phs003295',
    'phs003296',
    'phs003297',
    'phs003298',
    'phs003299',
    'phs003300',
    'phs003301',
    'phs003302',
    'phs003303',
    'phs003304',
    'phs003305',
    'phs003306',
    'phs003307',
    'phs003308',
    'phs003309',
    'phs003310',
    'phs003311',
    'phs003312',
    'phs003313',
    'phs003314',
    'phs003315',
    'phs003316',
    'phs003317',
    'phs003318',
    'phs003319',
    'phs003320',
    'phs003321',
    'phs003322',
    'phs003323',
    'phs003324',
    'phs003325',
    'phs003326',
    'phs003327',
    'phs003328',
    'phs003329',
    'phs003330',
    'phs003331',
    'phs003332',
    'phs003333',
    'phs003334',
    'phs003335',
    'phs003336',
    'phs003337',
    'phs003338',
    'phs003340',
    'phs003341',
    'phs003342',
    'phs003343',
    'phs003344',
    'phs003345',
    'phs003346',
    'phs003347',
    'phs003348',
    'phs003349',
    'phs003350',
    'phs003351',
    'phs003352',
    'phs003353',
    'phs003354',
    'phs003355',
    'phs003356',
    'phs003357',
    'phs003358',
    'phs003359',
    'phs003360',
    'phs003361',
    'phs003362',
    'phs003363',
    'phs003364',
    'phs003365',
    'phs003366',
    'phs003367',
    'phs003368',
    'phs003369',
    'phs003370',
    'phs003371',
    'phs003372',
    'phs003373',
    'phs003374',
    'phs003375',
    'phs003376',
    'phs003377',
    'phs003378',
    'phs003379',
    'phs003380',
    'phs003381',
    'phs003382',
    'phs003383',
    'phs003384',
    'phs003385',
    'phs003386',
    'phs003387',
    'phs003388',
    'phs003389',
    'phs003390',
    'phs003391',
    'phs003392',
    'phs003393',
    'phs003394',
    'phs003395',
    'phs003396',
    'phs003397',
    'phs003398',
    'phs003399',
    'phs003400',
    'phs003401',
    'phs003402',
    'phs003403',
    'phs003404',
    'phs003405',
    'phs003406',
    'phs003407',
    'phs003408',
    'phs003409',
    'phs003410',
    'phs003411',
    'phs003412',
    'phs003413',
    'phs003414',
    'phs003415',
    'phs003416',
    'phs003417',
    'phs003418',
    'phs003419',
    'phs003420',
    'phs003421',
    'phs003422',
    'phs003423',
    'phs003424',
    'phs003425',
    'phs003426',
    'phs003427',
    'phs003428',
    'phs003429',
    'phs003430',
    'phs003431',
    'phs003432',
    'phs003433',
    'phs003434',
    'phs003435',
    'phs003436',
    'phs003437',
    'phs003438',
    'phs003439',
    'phs003440',
    'phs003441',
    'phs003442',
    'phs003443',
    'phs003444',
    'phs003445',
    'phs003446',
    'phs003447',
    'phs003448',
    'phs003449',
    'phs003450',
    'phs003451',
    'phs003452',
    'phs003453',
    'phs003454',
    'phs003455',
    'phs003456',
    'phs003457',
    'phs003458',
    'phs003459',
    'phs003460',
    'phs003461',
    'phs003462',
    'phs003463',
    'phs003464',
    'phs003465',
    'phs003466',
    'phs003467',
    'phs003468',
    'phs003469',
    'phs003470',
    'phs003471',
    'phs003472',
    'phs003473',
    'phs003474',
    'phs003475',
    'phs003476',
    'phs003477',
    'phs003478',
    'phs003479',
    'phs003480',
    'phs003481',
    'phs003482',
    'phs003483',
    'phs003484',
    'phs003485',
    'phs003486',
    'phs003487',
    'phs003488',
    'phs003489',
    'phs003490',
    'phs003491',
    'phs003492',
    'phs003493',
    'phs003494',
    'phs003495',
    'phs003496',
    'phs003497',
    'phs003498',
    'phs003499',
    'phs003500',
    'phs003501',
    'phs003502',
    'phs003503',
    'phs003504',
    'phs003505',
    'phs003506',
    'phs003507',
    'phs003508',
    'phs003509',
    'phs003510',
    'phs003511',
    'phs003512',
    'phs003513',
    'phs003514',
    'phs003515',
    'phs003516',
    'phs003517',
    'phs003518',
    'phs003519',
    'phs003520',
    'phs003521',
    'phs003522',
    'phs003523',
    'phs003524',
    'phs003525',
    'phs003526',
    'phs003528',
    'phs003529',
    'phs003530',
    'phs003531',
    'phs003532',
    'phs003533',
    'phs003534',
    'phs003535',
    'phs003536',
    'phs003537',
    'phs003538',
    'phs003539',
    'phs003540',
    'phs003541',
    'phs003542',
    'phs003543',
    'phs003544',
    'phs003545',
    'phs003546',
    'phs003547',
    'phs003548',
    'phs003549',
    'phs003550',
    'phs003551',
    'phs003552',
    'phs003553',
    'phs003554',
    'phs003555',
    'phs003556',
    'phs003557',
    'phs003558',
    'phs003559',
    'phs003560',
    'phs003561',
    'phs003562',
    'phs003563',
    'phs003564',
    'phs003565',
    'phs003566',
    'phs003567',
    'phs003568',
    'phs003569',
    'phs003570',
    'phs003571',
    'phs003572',
    'phs003573',
    'phs003574',
    'phs003575',
    'phs003576',
    'phs003577',
    'phs003578',
    'phs003579',
    'phs003580',
    'phs003581',
    'phs003582',
    'phs003583',
    'phs003584',
    'phs003585',
    'phs003586',
    'phs003587',
    'phs003588',
    'phs003589',
    'phs003590',
    'phs003591',
    'phs003592',
    'phs003593',
    'phs003594',
    'phs003595',
    'phs003596',
    'phs003597',
    'phs003598',
    'phs003599',
    'phs003600',
    'phs003601',
    'phs003602',
    'phs003603',
    'phs003604',
    'phs003605',
    'phs003606',
    'phs003607',
    'phs003608',
    'phs003609',
    'phs003610',
    'phs003611',
    'phs003613',
    'phs003614',
    'phs003615',
    'phs003616',
    'phs003617',
    'phs003618',
    'phs003619',
    'phs003620',
    'phs003621',
    'phs003622',
    'phs003623',
    'phs003624',
    'phs003626',
    'phs003627',
    'phs003628',
    'phs003629',
    'phs003630',
    'phs003631',
    'phs003632',
    'phs003633',
    'phs003634',
    'phs003635',
    'phs003636',
    'phs003637',
    'phs003638',
    'phs003639',
    'phs003640',
    'phs003641',
    'phs003642',
    'phs003643',
    'phs003644',
    'phs003645',
    'phs003646',
    'phs003647',
    'phs003648',
    'phs003649',
    'phs003650',
    'phs003651',
    'phs003652',
    'phs003654',
    'phs003655',
    'phs003656',
    'phs003657',
    'phs003658',
    'phs003659',
    'phs003660',
    'phs003661',
    'phs003662',
    'phs003663',
    'phs003664',
    'phs003665',
    'phs003666',
    'phs003667',
    'phs003668',
    'phs003669',
    'phs003670',
    'phs003671',
    'phs003672',
    'phs003673',
    'phs003674',
    'phs003675',
    'phs003676',
    'phs003677',
    'phs003678',
    'phs003679',
    'phs003680',
    'phs003681',
    'phs003682',
    'phs003683',
    'phs003684',
    'phs003685',
    'phs003686',
    'phs003687',
    'phs003688',
    'phs003689',
    'phs003690',
    'phs003691',
    'phs003692',
    'phs003693',
    'phs003694',
    'phs003695',
    'phs003696',
    'phs003697',
    'phs003698',
    'phs003699',
    'phs003700',
    'phs003701',
    'phs003702',
    'phs003703',
    'phs003704',
    'phs003705',
    'phs003706',
    'phs003707',
    'phs003708',
    'phs003709',
    'phs003710',
    'phs003711',
    'phs003712',
    'phs003713',
    'phs003714',
    'phs003715',
    'phs003716',
    'phs003717',
    'phs003718',
    'phs003719',
    'phs003720',
    'phs003721',
    'phs003722',
    'phs003723',
    'phs003724',
    'phs003725',
    'phs003726',
    'phs003727',
    'phs003728',
    'phs003729',
    'phs003730',
    'phs003731',
    'phs003732',
    'phs003733',
    'phs003734',
    'phs003735',
    'phs003736',
    'phs003737',
    'phs003738',
    'phs003739',
    'phs003740',
    'phs003741',
    'phs003742',
    'phs003743',
    'phs003744',
    'phs003745',
    'phs003746',
    'phs003747',
    'phs003748',
    'phs003749',
    'phs003750',
    'phs003751',
    'phs003752',
    'phs003753',
    'phs003754',
    'phs003755',
    'phs003756',
    'phs003757',
    'phs003758',
    'phs003759',
    'phs003760',
    'phs003761',
    'phs003762',
    'phs003763',
    'phs003764',
    'phs003765',
    'phs003766',
    'phs003767',
    'phs003768',
    'phs003769',
    'phs003770',
    'phs003771',
    'phs003772',
    'phs003773',
    'phs003774',
    'phs003775',
    'phs003776',
    'phs003777',
    'phs003778',
    'phs003779',
    'phs003780',
    'phs003781',
    'phs003782',
    'phs003783',
    'phs003784',
    'phs003785',
    'phs003786',
    'phs003787',
    'phs003788',
    'phs003789',
    'phs003790',
    'phs003791',
    'phs003792',
    'phs003793',
    'phs003794',
    'phs003795',
    'phs003796',
    'phs003797',
    'phs003798',
    'phs003799',
    'phs003800',
    'phs003801',
    'phs003802',
    'phs003803',
    'phs003804',
    'phs003805',
    'phs003806',
    'phs003807',
    'phs003808',
    'phs003809',
    'phs003810',
    'phs003812',
    'phs003813',
    'phs003814',
    'phs003815',
    'phs003816',
    'phs003817',
    'phs003818',
    'phs003819',
    'phs003820',
    'phs003821',
    'phs003822',
    'phs003823',
    'phs003824',
    'phs003825',
    'phs003826',
    'phs003827',
    'phs003828',
    'phs003829',
    'phs003830',
    'phs003831',
    'phs003832',
    'phs003833',
    'phs003834',
    'phs003835',
    'phs003836',
    'phs003837',
    'phs003838',
    'phs003839',
    'phs003840',
    'phs003841',
    'phs003842',
    'phs003843',
    'phs003844',
    'phs003845',
    'phs003846',
    'phs003847',
    'phs003848',
    'phs003849',
    'phs003850',
    'phs003851',
    'phs003852',
    'phs003853',
    'phs003854',
    'phs003855',
    'phs003856',
    'phs003857',
    'phs003858',
    'phs003859',
    'phs003860',
    'phs003861',
    'phs003862',
    'phs003863',
    'phs003864',
    'phs003865',
    'phs003866',
    'phs003867',
    'phs003868',
    'phs003869',
    'phs003870',
    'phs003871',
    'phs003872',
    'phs003873',
    'phs003874',
    'phs003875',
    'phs003876',
    'phs003877',
    'phs003878',
    'phs003879',
    'phs003880',
    'phs003881',
    'phs003882',
    'phs003883',
    'phs003884',
    'phs003885',
    'phs003886',
    'phs003887',
    'phs003888',
    'phs003889',
    'phs003890',
    'phs003891',
    'phs003892',
    'phs003893',
    'phs003894',
    'phs003895',
    'phs003896',
    'phs003897',
    'phs003898',
    'phs003899',
    'phs003900',
    'phs003901',
    'phs003902',
    'phs003903',
    'phs003904',
    'phs003905',
    'phs003906',
    'phs003907',
    'phs003908',
    'phs003909',
    'phs003910',
    'phs003911',
    'phs003912',
    'phs003913',
    'phs003914',
    'phs003915',
    'phs003916',
    'phs003917',
    'phs003918',
    'phs003919',
    'phs003920',
    'phs003921',
    'phs003922',
    'phs003923',
    'phs003924',
    'phs003925',
    'phs003926',
    'phs003927',
    'phs003928',
    'phs003929',
    'phs003930',
    'phs003931',
    'phs003932',
    'phs003933',
    'phs003934',
    'phs003935',
    'phs003936',
    'phs003937',
    'phs003938',
    'phs003939',
    'phs003940',
    'phs003941',
    'phs003942',
    'phs003943',
    'phs003944',
    'phs003945',
    'phs003946',
    'phs003947',
    'phs003948',
    'phs003949',
    'phs003950',
    'phs003951',
    'phs003952',
    'phs003953',
    'phs003954',
    'phs003955',
    'phs003956',
    'phs003957',
    'phs003958',
    'phs003959',
    'phs003960',
    'phs003961',
    'phs003962',
    'phs003963',
    'phs003964',
    'phs003965',
    'phs003966',
    'phs003967',
    'phs003968',
    'phs003969',
    'phs003970',
    'phs003971',
    'phs003972',
    'phs003973',
    'phs003974',
    'phs003975',
    'phs003976',
    'phs003977',
    'phs003978',
    'phs003979',
    'phs003980',
    'phs003981',
    'phs003982',
    'phs003983',
    'phs003984',
    'phs003985',
    'phs003986',
    'phs003987',
    'phs003988',
    'phs003989',
    'phs003990',
    'phs003991',
    'phs003992',
    'phs003993',
    'phs003994',
    'phs003995',
    'phs003996',
    'phs003997',
    'phs003998',
    'phs003999',
    'phs004000',
    'phs004001',
    'phs004002',
    'phs004003',
    'phs004004',
    'phs004005',
    'phs004006',
    'phs004007',
    'phs004008',
    'phs004009',
    'phs004010',
    'phs004011',
    'phs004012',
    'phs004013',
    'phs004014',
    'phs004015',
    'phs004016',
    'phs004017',
    'phs004018',
    'phs004019',
    'phs004020',
    'phs004021',
    'phs004022',
    'phs004023',
    'phs004024',
    'phs004025',
    'phs004026',
    'phs004027',
    'phs004028',
    'phs004029',
    'phs004030',
    'phs004031',
    'phs004032',
    'phs004033',
    'phs004034',
    'phs004035',
    'phs004036',
    'phs004037',
    'phs004038',
    'phs004039',
    'phs004040',
    'phs004041',
    'phs004042',
    'phs004043',
    'phs004044',
    'phs004045',
    'phs004046',
    'phs004047',
    'phs004048',
    'phs004049',
    'phs004050',
    'phs004051',
    'phs004052',
    'phs004053',
    'phs004054',
    'phs004055',
    'phs004056',
    'phs004057',
    'phs004058',
    'phs004059',
    'phs004060',
    'phs004061',
    'phs004062',
    'phs004063',
    'phs004064',
    'phs004065',
    'phs004066',
    'phs004067',
    'phs004068',
    'phs004069',
    'phs004070',
    'phs004071',
    'phs004072',
    'phs004073',
    'phs004074',
    'phs004075',
    'phs004076',
    'phs004077',
    'phs004078',
    'phs004079',
    'phs004080',
    'phs004081',
    'phs004082',
    'phs004083',
    'phs004084',
    'phs004085',
    'phs004086',
    'phs004087',
]
phs_id_list = ["phs001391", "phs001415"]

# Specify a file that maps DS consent codes with modifiers removed (IRB, NPU, RD, MDS, PUB, COL, GSO) to DOID
# Form should be JSON with structure {"DS-CODE": ["http://purl.obolibrary.org/obo/DOID_12345"], ...}
ds_consent_map_file = "dataset_metadata/support/ds_consent_mapping.json"

# Specify a file of overrides to the DS consent map (for cases where a DS code has different definitions for different studies)
# Form should be JSON with structure {"PHS:DS-CODE": ["http://purl.obolibrary.org/obo/DOID_12345"], ...}
ds_consent_map_override_file = "dataset_metadata/support/ds_consent_mapping_overrides.json"

# Specify text to add to the end of the study description, if desired
desc_tag = "Cataloged for the NCPI DUO effort."

# DUOS variables (to determine if a study already exists in DUOS):
#  - duos_token - Token for use in DUOS (use: gcloud auth print-access-token | pbcopy), if empty will ignore comparison
#  - duos_env - DUOS environment to use
duos_token = ""
duos_env = "dev"

# Specify whether results should be written out to a file in the workspace bucket
write_to_bucket = False

# Specify whether results should be displayed
display_output = True


#############################################
## Execution
#############################################

# Load in contents of consent map files
storage_client = storage.Client()
bucket = storage_client.get_bucket(ws_bucket.replace("gs://", ""))
consent_map_blob = bucket.blob(ds_consent_map_file)
ds_consent_map = json.loads(consent_map_blob.download_as_string(client=None))
consent_map_override_blob = bucket.blob(ds_consent_map_override_file) 
ds_consent_map_overrides = json.loads(consent_map_override_blob.download_as_string(client=None))

# Fetch and format dbgap metadata
study_lookup = fetch_and_format_dbgap_metadata(phs_id_list, ds_consent_map, ds_consent_map_overrides, desc_tag, duos_token, duos_env, write_to_bucket, display_output)


## Pull Approved Requestors for dbGaP Studies

In [ ]:
import requests

def download_dynamic_txt_content(download_url):
    """
    Fetches the content from a URL that initiates a file download 
    (like a CGI-bin script).
    """
    try:
        # 1. Send an HTTP GET request to the download URL.
        # The 'stream=True' is often not strictly necessary for small text files, 
        # but is good practice for binary or very large files.
        response = requests.get(download_url)

        # 2. Check for HTTP errors (e.g., 404, 500, or a server-side error 
        # that redirects to an error page instead of the file).
        response.raise_for_status()

        # 3. Get the content as text.
        file_content = response.text

        return file_content

    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: Could not download the file from {download_url}.")
        print(f"Status Code: {e.response.status_code}")
        print("This might be due to a server error, access restrictions, or the file not being found.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the request: {e}")
        return None

# Your example URL
download_link = "https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/GetAuthorizedRequestDownload.cgi?study_id=phs001489.v4.p2"

content = download_dynamic_txt_content(download_link)

if content:
    print(f"Successfully downloaded {len(content)} characters of text.")
    # You can print the first 500 characters to verify
    print("--- Snippet ---")
    print(content[:500])
    
    # Example: Write the content to a local file
    # with open("downloaded_data.txt", "w", encoding="utf-8") as f:
    #     f.write(content)
    # print("\nContent saved to downloaded_data.txt")

# Script Development

## Fetch parameters from snapshot/dataset

In [ ]:
# Parameters
snapshot_id = "099d2585-1379-4333-b3b1-ffc0d26d95c5"

# Retrieve snapshot details
api_client = refresh_tdr_api_client()
datasets_api = data_repo_client.DatasetsApi(api_client=api_client)
snapshots_api = data_repo_client.SnapshotsApi(api_client=api_client)
snapshot_details = snapshots_api.retrieve_snapshot(id=snapshot_id).to_dict()
dataset_id = snapshot_details["source"][0]["dataset"]["id"]
phs_id = snapshot_details["source"][0]["dataset"]["phs_id"]

# Retrieve dataset details
dataset_details = datasets_api.retrieve_dataset(id=dataset_id, include=["PROPERTIES"]).to_dict()
if dataset_details["properties"].get("auth_domains"):
    auth_domain = dataset_details["properties"]["auth_domains"][0]
if dataset_details["properties"].get("source_workspaces"):
    source_workspace = dataset_details["properties"]["source_workspaces"][0]

# Print output
print(phs_id)
print(source_workspace)

## Pulling Workspace Attributes

In [ ]:
# Parameters
ws_project = "anvil-datastorage"
ws_name = "AnVIL_DepMap_HMB"

# Establish credentials
creds, project = google.auth.default()
auth_req = google.auth.transport.requests.Request()
creds.refresh(auth_req)

# Pull workspace attributes
ws_attributes = requests.get(
    url=f"https://api.firecloud.org/api/workspaces/{ws_project}/{ws_name}?fields=workspace.attributes,workspace.authorizationDomain,workspace.googleProject,workspace.bucketName",
    headers={"Authorization": f"Bearer {creds.token}"}
).json()

# Map to schema
terra_dict = {}
terra_dict["studyName"] = ws_attributes["workspace"]["attributes"].get("library:projectName")
terra_dict["studyType"] = ws_attributes["workspace"]["attributes"].get("library:studyDesign")
#terra_dict["studyDescription"] = ws_attributes["workspace"]["attributes"].get("description")
terra_dict["dataTypes"] = ws_attributes["workspace"]["attributes"].get("library:dataCategory")["items"]
terra_dict["phenotypeIndication"] = ws_attributes["workspace"]["attributes"].get("library:indication")
terra_dict["species"] = "Homo sapiens"
terra_dict["piName"] = ws_attributes["workspace"]["attributes"].get("library:datasetOwner")
terra_dict["dataCustodianEmail"] = ws_attributes["workspace"]["attributes"].get("library:contactEmail")
if ws_attributes["workspace"]["attributes"].get("tag:tags"):
    for tag in ws_attributes["workspace"]["attributes"].get("tag:tags")["items"]:
        if "Consortium:" in tag:
            terra_dict["consortium"] = tag.split(":")[1].strip()
        elif "dbGaP:" in tag:
            terra_dict["dbGaPPhsID"] = tag.split(":")[1].strip()
terra_dict["consentGroups.consentCode"] = ws_attributes["workspace"]["attributes"]["library:dataUseRestriction"] 
terra_dict["consentGroups.fileTypes.fileType"] = ws_attributes["workspace"]["attributes"]["library:datatype"]["items"]

# View schema
print(terra_dict)


In [ ]:
ws_attributes

In [ ]:
ws_attributes

## dbGaP XML Parse

In [ ]:
# Parameters
#phs_id = "phs003047"
phs_id = "phs003444"

# Pull and parse XML
phs_short = phs_id.replace("phs", "")
dbgap_url = "https://dbgap.ncbi.nlm.nih.gov/ss/dbgapssws.cgi?request=Study&phs=" + phs_short
response = requests.get(url=dbgap_url)
xml_data = xmltodict.parse(response.text)

# Map to schema
dac_names = ""
dbgap_xml_dict = {}
if isinstance(xml_data["dbgapss"]["Study"], list):
    study_data = xml_data["dbgapss"]["Study"][0]
else:
    study_data = xml_data["dbgapss"]["Study"] 
dbgap_xml_dict["studyName"] = study_data["StudyInfo"].get("StudyNameEntrez")
dbgap_xml_dict["studyDescription"] = study_data["StudyInfo"].get("Description")
dbgap_xml_dict["dbGaPPhsID"] = phs_id
dbgap_xml_dict["dbGaPStudyRegistrationName"] = study_data["StudyInfo"].get("StudyNameEntrez")
for ap_entry in study_data["Authority"]["Persons"]["Person"]:
    if ap_entry["Role"] == "PI":
        dbgap_xml_dict["piName"] = ap_entry["@lname"] + ", " + ap_entry["@fname"]
        dbgap_xml_dict["piEmail"] = ap_entry["@email"]
        dbgap_xml_dict["piInstitution"] = ap_entry["Organization"]
    elif ap_entry["Role"] == "PO" and ap_entry["Organization"] == "NIH":
        dbgap_xml_dict["nihProgramOfficerName"] = ap_entry["@lname"] + ", " + ap_entry["@fname"]
    elif ap_entry["Role"] == "GPA" and ap_entry["Organization"] == "NIH":
        dbgap_xml_dict["nihGenomicProgramAdministratorName"] = ap_entry["@lname"] + ", " + ap_entry["@fname"]
ic_list = []
if isinstance(study_data["Authority"]["ICs"]["IC"], list):
    for ic_entry in study_data["Authority"]["ICs"]["IC"]:
        ic_list.append(ic_entry["@name"])
else:
    ic_list.append(study_data["Authority"]["ICs"]["IC"]["@name"])
dbgap_xml_dict["nihICsSupportingStudy"] = ic_list
dbgap_xml_dict["numberOfParticipants"] = study_data.get("@num_participants")
dbgap_xml_dict["embargoReleaseDate"] = study_data["Policy"].get("@pub-embargo")
if isinstance(study_data["Policy"]["ConsentGroup"], list):
    dac_name_set = set()
    for idx, consent in enumerate(study_data["Policy"]["ConsentGroup"]):
        tmp_dac = consent["@dac_name"]
        dac_name_set.add(tmp_dac)
    dac_name_list = list(dac_name_set)
    dac_names = ", ".join(dac_name_list)
else:
    dac_names = study_data["Policy"]["ConsentGroup"]["@dac_name"]

# View schema
print(dbgap_xml_dict)


In [ ]:
dac_names

In [ ]:
type(study_data["Policy"]["ConsentGroup"])

## dbGaP Study API

In [ ]:
# Parameters
study_uid = 483191234

# Pull and parse JSON
dbgap_study_url = "https://submit.ncbi.nlm.nih.gov/dbgap/api/v1/study_config/" + str(study_uid)
response = requests.get(url=dbgap_study_url)
study_api_data = json.loads(response.text)

# Map to schema
dbgap_study_api_dict = {}
if study_api_data.get("error") == None:
    dbgap_study_api_dict["studyName"] = study_api_data["data"].get("report_name")
    dbgap_study_api_dict["studyDescription"] = study_api_data["data"].get("description")
    dbgap_study_api_dict["phenotypeIndication"] = study_api_data["data"].get("primary_disease")
    dbgap_study_api_dict["studyType"] = study_api_data["data"].get("study_design")
    for attr_entry in study_api_data["data"].get("attribution"):
        if attr_entry.get("title") == "Principal Investigator":
            dbgap_study_api_dict["piName"] = attr_entry.get("name")
            dbgap_study_api_dict["piInstitution"] = attr_entry.get("institute")
            break

# View schema
print(dbgap_study_api_dict)

In [ ]:
study_api_data

## dbGaP FHIR API

In [ ]:
# Parameters
#phs_id = "phs003047"
phs_id = "phs000693"

# Pull and parse JSON
dbgap_fhir_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/ResearchStudy?_format=json&_id=" + phs_id
response = requests.get(url=dbgap_fhir_url)
fhir_data = json.loads(response.text)

# Map to schema
dbgap_fhir_dict = {}
dbgap_fhir_dict["studyName"] = fhir_data["entry"][0]["resource"].get("title")
dbgap_fhir_dict["studyDescription"] = fhir_data["entry"][0]["resource"].get("description")
dbgap_fhir_dict["dbGaPPhsID"] = phs_id
dbgap_fhir_dict["dbGaPStudyRegistrationName"] = fhir_data["entry"][0]["resource"].get("title")
dbgap_fhir_dict["nihICsSupportingStudy"] = fhir_data["entry"][0]["resource"]["sponsor"].get("display")
# studyType
for cat_entry in fhir_data["entry"][0]["resource"].get("category"):
    for coding_entry in cat_entry.get("coding"):
        if coding_entry.get("system") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/CodeSystem/ResearchStudy-StudyDesign":
            value = coding_entry.get("display") if coding_entry.get("display") else coding_entry.get("code")
            if dbgap_fhir_dict.get("studyType") and value:
                dbgap_fhir_dict["studyType"] += f", {value}"
            elif value:
                dbgap_fhir_dict["studyType"] = value
# dataTypes
dt_list = []
for ext_entry in fhir_data["entry"][0]["resource"].get("extension"):
    if ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-MolecularDataTypes":
        for inner_ext_entry in ext_entry.get("extension"):
            if inner_ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-MolecularDataTypes-MolecularDataType":
                for coding_entry in inner_ext_entry["valueCodeableConcept"].get("coding"):
                    dt_list.append(coding_entry.get("code"))
dbgap_fhir_dict["dataTypes"] = dt_list
# phenotypeIndication
for focus_entry in fhir_data["entry"][0]["resource"].get("focus"):
    for coding_entry in focus_entry.get("coding"):
        value = coding_entry.get("display") if coding_entry.get("display") else coding_entry.get("code")
        if dbgap_fhir_dict.get("phenotypeIndication") and value:
            dbgap_fhir_dict["phenotypeIndication"] += f", {value}"
        elif value:
            dbgap_fhir_dict["phenotypeIndication"] = value
# numberOfParticipants
for ext_entry in fhir_data["entry"][0]["resource"].get("extension"):
    if ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-Content":
        for inner_ext_entry in ext_entry.get("extension"):
            if inner_ext_entry.get("url") == "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/StructureDefinition/ResearchStudy-Content-NumSubjects":
                dbgap_fhir_dict["numberOfParticipants"] = inner_ext_entry["valueCount"].get("code")

# View schema
print(dbgap_fhir_dict)

# Troubleshooting

## TSV File Parsing

In [ ]:
input_file = "gs://fc-2a9eefc3-0302-427f-9ac3-82f078741c03/dataset_metadata/input/firecloud_target_tcga_metadata_20250227.tsv" 

# Pull down specified file from the cloud
results_log = []
print(f"Downloading input file {input_file}...")
try:
    input_df = pd.read_csv(input_file, delimiter = "\t", encoding='unicode_escape')
    input_df = input_df.astype(object).where(pd.notnull(input_df),None)
    input_df.fillna("",inplace=True)
    input_dict = input_df.to_dict(orient="records")
    results_log.append(["Input File Download", "Succeeded", ""])
except Exception as e:
    msg = f"Error downloading input file ({input_file}): {str(e)}"
    results_log.append(["Input File Download", "Failed", msg])
    print(msg)

In [ ]:
file_types_dict = json.loads(input_dict[0]["consentGroups.fileTypes"])

In [ ]:
file_types_dict